# TopDrive AI — Synthetic Dataset Generation

Generates physics-based pipe threading scenarios for ML training.
**All code is embedded below — no file uploads required.**

Data is saved to Google Drive so it persists between sessions and can be
loaded by the training notebook.

## Workflow
1. Run **Setup** cells (install deps, mount Drive, write modules)
2. Configure generation parameters in the **Configuration** cell
3. Run **Generate Dataset** — takes ~1 hr for 5000 scenarios on Colab CPU
4. Data is auto-copied to Drive when done


## Step 1 — Install Dependencies

In [ ]:
# Dependencies for data generation (numpy pre-installed on Colab)
!pip install pyarrow pandas tqdm psutil -q
print("Dependencies ready.")


## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## Step 3 — Configuration

Edit these variables before running the generation cells.


In [ ]:
import os

# ── Generation parameters ─────────────────────────────────────────────────
NUM_SCENARIOS   = 5000       # Total scenarios (5000 = Phase 1 full dataset)
SEED            = 42         # Reproducibility seed
CLASS_BALANCE   = 'rebalanced'   # 'rebalanced' (50/50) or 'default' (field dist)
OUTPUT_FORMAT   = 'parquet'       # 'parquet' (faster/smaller) or 'csv'
OUTPUT_RATE_HZ  = 100.0          # Sensor data output rate (Hz)

# ── Performance ───────────────────────────────────────────────────────────
NUM_WORKERS     = 0          # 0 = auto-detect (cpu_count - 1), 1 = sequential
ENABLE_DIAGNOSTICS = True    # Print per-scenario timing and quality stats

# ── Paths ─────────────────────────────────────────────────────────────────
LOCAL_OUTPUT_DIR = '/content/synthetic_v2'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/topdrive_ai/synthetic_v2'

os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# ── System info ───────────────────────────────────────────────────────────
import multiprocessing as _mp
_cpu_count = os.cpu_count() or 2
_workers = NUM_WORKERS if NUM_WORKERS > 0 else max(1, _cpu_count - 1)

try:
    import psutil
    _ram_gb = psutil.virtual_memory().total / (1024**3)
    _ram_str = f"{_ram_gb:.1f} GB"
except ImportError:
    _ram_str = "unknown"

try:
    import torch
    _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    _gpu_vram = f"({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)" if torch.cuda.is_available() else ""
except Exception:
    _gpu_name = "N/A"
    _gpu_vram = ""

print(f"Local output : {LOCAL_OUTPUT_DIR}")
print(f"Drive output : {DRIVE_OUTPUT_DIR}")
print(f"Scenarios    : {NUM_SCENARIOS}")
print(f"Class balance: {CLASS_BALANCE}")
print(f"Format       : {OUTPUT_FORMAT}")
print(f"CPU cores    : {_cpu_count} (using {_workers} workers)")
print(f"RAM          : {_ram_str}")
print(f"GPU          : {_gpu_name} {_gpu_vram}")
print(f"Diagnostics  : {'ON' if ENABLE_DIAGNOSTICS else 'OFF'}")


## Step 4 — Write Simulation Modules

Each cell below writes one Python module to the Colab VM disk.
Run them all once per session (they reset when the runtime restarts).


### `config.py`

In [ ]:
%%writefile config.py
"""
Physical constants, pipe specifications, and simulation parameters.

Sources:
  - API RP 7G: Recommended Practice for Drill Stem Design and Operating Limits
  - API 5B / 5CT: Thread geometry and casing/tubing specifications
  - API RP 5C1: Care and Use of Casing and Tubing (Table 1 — exact torque values)
  - API RP 5A3: Thread compound friction factors
  - API 7-2: Threading and Gauging for Rotary Shouldered Connections
  - GE CPE305 PLC documentation (register layout — Section 5.1.1)
  - Farr friction model for threaded connections
  - ASTM D341 (Walther equation for oil viscosity)
  - Industrial hydraulic motor / pump specifications
  - NOV / Canrig / MHWirth top drive technical data
  - NOV ST-80/100/120 iron roughneck specifications
  - Vallourec VAM / TenarisHydril / Hunting premium connection datasheets

Design principle: Every magic number lives here. Domain randomization
perturbs these values within specified ranges to create diverse training data.
"""
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum
import numpy as np


# ═══════════════════════════════════════════════════════════════════
# Machine Type Enumeration
# ═══════════════════════════════════════════════════════════════════

class MachineType(Enum):
    """Oilfield machine categories per reference Section 2."""
    TOP_DRIVE = "top_drive"
    IRON_ROUGHNECK = "iron_roughneck"
    POWER_TONG = "power_tong"
    BUCKING_UNIT = "bucking_unit"
    CASING_RUNNING_TOOL = "casing_running_tool"


class ConnectionCategory(Enum):
    """Connection type categories per reference Section 3."""
    API_8RD_STC = "api_8rd_stc"       # API 8-round Short Thread Coupling
    API_8RD_LTC = "api_8rd_ltc"       # API 8-round Long Thread Coupling
    API_BUTTRESS = "api_buttress"      # API Buttress Thread (BTC)
    API_EXTREME_LINE = "api_xl"        # API Extreme-Line (integral flush)
    PREMIUM_SHOULDERED = "premium"     # Premium with metal-to-metal seal + shoulder
    PREMIUM_FLUSH = "premium_flush"    # Premium flush/semi-flush (integral)
    DRILL_PIPE = "drill_pipe"         # API 7-2 rotary shouldered connections


class TorqueMeasurementMethod(Enum):
    """How torque is measured — affects noise characteristics (Section 4.4)."""
    MOTOR_CURRENT = "motor_current"      # Top drive: T = I * K_motor, SNR 45-60 dB
    PRESSURE_TRANSDUCER = "pressure"     # Iron roughneck: T = P * K, SNR 50-65 dB
    LOAD_CELL = "load_cell"              # Power tong: strain gauge, SNR 55-70 dB
    CALIBRATED_LOAD_CELL = "cal_cell"    # Bucking unit: precision cell, SNR 60-75 dB


# ═══════════════════════════════════════════════════════════════════
# Pipe Steel Grades (API 5CT — Section 3.3)
# ═══════════════════════════════════════════════════════════════════

@dataclass
class PipeGrade:
    """Steel grade properties per API 5CT."""
    name: str
    min_yield_ksi: float
    max_yield_ksi: float
    min_tensile_ksi: float
    sour_service: bool
    friction_factor_scale: float = 1.0  # Relative to J55 baseline


GRADE_CATALOG: Dict[str, PipeGrade] = {
    "H-40": PipeGrade("H-40", 40, 80, 60, False, 0.90),
    "J-55": PipeGrade("J-55", 55, 80, 75, False, 1.00),
    "K-55": PipeGrade("K-55", 55, 80, 95, False, 1.00),
    "N-80": PipeGrade("N-80", 80, 110, 100, False, 1.08),
    "L-80": PipeGrade("L-80", 80, 95, 95, True, 1.05),
    "L-80-9Cr": PipeGrade("L-80-9Cr", 80, 95, 95, True, 1.10),
    "L-80-13Cr": PipeGrade("L-80-13Cr", 80, 95, 95, True, 1.15),
    "C-90": PipeGrade("C-90", 90, 105, 100, True, 1.10),
    "R-95": PipeGrade("R-95", 95, 110, 105, False, 1.12),
    "T-95": PipeGrade("T-95", 95, 110, 105, True, 1.12),
    "C-110": PipeGrade("C-110", 110, 120, 125, True, 1.18),
    "P-110": PipeGrade("P-110", 110, 140, 125, False, 1.20),
    "Q-125": PipeGrade("Q-125", 125, 150, 135, False, 1.25),
}


# ═══════════════════════════════════════════════════════════════════
# Sub-System Specifications
# ═══════════════════════════════════════════════════════════════════

@dataclass
class ACMotorSpec:
    """AC induction motor + VFD specifications for top drives (Section 2.1).

    Modern top drives use AC motors with VFDs, NOT hydraulic motors.
    Torque-speed characteristic:
      - Below base speed: constant torque (T = T_rated)
      - Above base speed: constant power (T = T_rated * RPM_base / RPM)
    """
    rated_hp: float = 800.0               # Motor nameplate HP
    rated_rpm: float = 1800.0             # Motor base speed (60Hz)
    rated_torque_nm: float = 3180.0       # T = HP * 5252 / RPM * 1.3558
    max_torque_multiplier: float = 2.5    # VFD can deliver 250% rated torque transiently
    slip_pct: float = 2.5                 # Induction motor slip at rated load
    efficiency: float = 0.94              # Motor efficiency at rated load
    power_factor: float = 0.87            # Rated power factor
    rotor_inertia_kgm2: float = 12.0     # Motor rotor inertia
    vfd_response_ms: float = 50.0         # VFD torque response time
    vfd_current_limit_pct: float = 150.0  # VFD current limit (% of rated)
    regenerative_braking: bool = True     # VFD supports regen braking


@dataclass
class TopDriveSpec:
    """Top drive mechanical specifications (Section 2.1).

    Reference machines: NOV TDS-11SA (800HP), TDS-8 (1150HP), Canrig Sigma 500T
    """
    motor: ACMotorSpec = field(default_factory=ACMotorSpec)
    gear_ratio: float = 10.5              # Helical gear reduction (8:1 to 12:1)
    max_output_rpm: float = 228.0         # Output shaft max RPM (after gearbox)
    max_continuous_torque_ftlbs: float = 37_500.0  # Continuous rating
    max_intermittent_torque_ftlbs: float = 55_000.0  # 10-second burst
    gearbox_inertia_kgm2: float = 25.0   # Gearbox reflected inertia (output side)
    gearbox_efficiency: float = 0.97      # Gear mesh efficiency
    gearbox_backlash_deg: float = 0.1     # Backlash in gear train (degrees)
    brake_torque_ftlbs: float = 80_000.0  # Hydraulic disc brake holding torque
    quill_encoder_cpr: int = 1174         # Quill shaft encoder counts per revolution
    seal_friction_nm: float = 25.0        # Coulomb friction from shaft seals
    viscous_damping_nms: float = 12.0     # Viscous damping (seals + bearings)
    torque_measurement: TorqueMeasurementMethod = TorqueMeasurementMethod.MOTOR_CURRENT
    hoist_capacity_tons: float = 500.0    # Hook load capacity


@dataclass
class IronRoughneckSpec:
    """Iron roughneck specifications (Section 2.2).

    Reference machines: NOV ST-80C, ST-100, ST-120
    Two subsystems: spinner (low torque, high RPM) + torque wrench (high torque, low angular sweep)
    """
    # Spinner subsystem
    spinner_motor_displacement_cc: float = 80.0   # Hydraulic gear motor
    spinner_max_rpm: float = 75.0                 # Spin RPM
    spinner_max_torque_ftlbs: float = 1_750.0     # Spin torque
    spinner_flow_gpm: float = 35.0                # Required flow
    spinner_pressure_psi: float = 2_500.0         # Operating pressure

    # Torque wrench subsystem
    wrench_cylinder_bore_in: float = 6.0          # Hydraulic cylinder bore
    wrench_stroke_in: float = 8.0                 # Cylinder stroke
    wrench_moment_arm_in: float = 12.0            # Torque arm length
    wrench_max_torque_ftlbs: float = 80_000.0     # Makeup torque capacity
    wrench_breakout_torque_ftlbs: float = 100_000.0  # Breakout capacity
    wrench_angular_sweep_deg: float = 45.0        # Degrees per stroke
    wrench_pressure_psi: float = 3_000.0          # Operating pressure

    # Handoff parameters
    handoff_delay_ms: float = 500.0               # Time for spinner-to-wrench transition
    handoff_rpm_threshold: float = 2.0            # RPM below which wrench engages

    # General
    pipe_od_range: Tuple[float, float] = (4.125, 8.5)
    gripper_clamp_force_lbs: float = 50_000.0
    horizontal_travel_in: float = 72.0
    weight_lbs: float = 7_800.0
    torque_measurement: TorqueMeasurementMethod = TorqueMeasurementMethod.PRESSURE_TRANSDUCER


@dataclass
class PowerTongSpec:
    """Power tong specifications (Section 2.3).

    Continuous rotation, suspended operation, requires backup tong.
    """
    motor_displacement_cc: float = 120.0
    max_rpm_high_gear: float = 12.0
    max_rpm_low_gear: float = 5.0
    max_torque_high_gear_ftlbs: float = 30_000.0
    max_torque_low_gear_ftlbs: float = 100_000.0
    operating_pressure_psi: float = 2_500.0
    flow_gpm: float = 50.0
    two_speed: bool = True
    gear_shift_torque_ftlbs: float = 25_000.0   # Auto-shift threshold
    arm_compliance_deg_per_klb: float = 0.02     # Tong arm flex
    backup_tong_friction_ftlbs: float = 500.0
    torque_measurement: TorqueMeasurementMethod = TorqueMeasurementMethod.LOAD_CELL


@dataclass
class BuckingUnitSpec:
    """Bucking unit specifications (Section 2.4).

    Stationary, horizontal, precision torque control.
    Cleanest torque-turn curves (gold standard for training data).
    """
    motor_displacement_cc: float = 150.0
    max_rpm: float = 15.0
    max_torque_ftlbs: float = 150_000.0
    operating_pressure_psi: float = 3_000.0
    flow_gpm: float = 60.0
    servo_response_ms: float = 50.0       # Faster than field machines
    position_resolution_deg: float = 0.01  # CNC-grade positioning
    torque_measurement: TorqueMeasurementMethod = TorqueMeasurementMethod.CALIBRATED_LOAD_CELL
    vibration_isolation: bool = True        # No rig vibration


@dataclass
class ValveSpec:
    """Proportional directional control valve parameters."""
    spool_time_constant_ms: float = 25.0
    rate_limit_pct_per_s: float = 500.0
    dead_zone_pct: float = 1.5
    hysteresis_pct: float = 2.0
    null_band_pct: float = 0.5
    max_flow_gpm: float = 60.0
    rated_pressure_drop_psi: float = 150.0


@dataclass
class PumpSpec:
    """Hydraulic pump (HPU) specifications."""
    pump_type: str = "axial_piston"
    displacement_cc: float = 100.0
    drive_rpm: float = 1800.0
    num_pistons: int = 9
    max_pressure_psi: float = 5000.0
    vol_efficiency_coeffs: Tuple[float, ...] = (0.97, 0.95, 0.92, 0.87)
    mech_efficiency: float = 0.92
    relief_cracking_psi: float = 4800.0
    relief_full_flow_psi: float = 5200.0
    relief_reseat_psi: float = 4600.0
    trapped_volume_in3: float = 150.0


@dataclass
class OilSpec:
    """Hydraulic fluid properties (ISO VG 46)."""
    iso_grade: int = 46
    kinematic_viscosity_40c_cst: float = 46.0
    kinematic_viscosity_100c_cst: float = 6.8
    density_kg_m3: float = 870.0
    specific_heat_j_kgk: float = 2000.0
    bulk_modulus_psi: float = 200_000.0
    air_content_pct: float = 2.0
    thermal_conductivity_w_mk: float = 0.14


@dataclass
class ThreadCompoundSpec:
    """Thread compound (dope) friction properties per API RP 5A3."""
    name: str = "API_Modified_Zinc"
    base_kf: float = 0.08
    temp_coefficient: float = -0.0003
    shear_rate_exponent: float = -0.15
    reference_temp_f: float = 77.0
    reference_shear_rate: float = 100.0
    max_service_temp_f: float = 300.0
    degradation_rate: float = 0.001


@dataclass
class PipeStringSpec:
    """Pipe string properties for torsional dynamics."""
    length_ft: float = 30.0
    num_joints: int = 1
    shear_modulus_psi: float = 11.5e6
    material_damping_ratio: float = 0.02
    joint_friction_ftlbs: float = 5.0


# ═══════════════════════════════════════════════════════════════════
# Thread Compound Catalog
# ═══════════════════════════════════════════════════════════════════

COMPOUND_CATALOG: Dict[str, ThreadCompoundSpec] = {
    "API_Modified_Zinc": ThreadCompoundSpec(
        name="API_Modified_Zinc", base_kf=0.08,
        temp_coefficient=-0.0003, shear_rate_exponent=-0.15,
    ),
    "API_Modified_Lead": ThreadCompoundSpec(
        name="API_Modified_Lead", base_kf=0.06,
        temp_coefficient=-0.0002, shear_rate_exponent=-0.12,
        max_service_temp_f=250.0,
    ),
    "Copper_Based": ThreadCompoundSpec(
        name="Copper_Based", base_kf=0.10,
        temp_coefficient=-0.0004, shear_rate_exponent=-0.10,
        max_service_temp_f=350.0,
    ),
    "Eco_Green": ThreadCompoundSpec(
        name="Eco_Green", base_kf=0.12,
        temp_coefficient=-0.0005, shear_rate_exponent=-0.18,
        max_service_temp_f=200.0,
    ),
    "Molybdenum": ThreadCompoundSpec(
        name="Molybdenum", base_kf=0.05,
        temp_coefficient=-0.0001, shear_rate_exponent=-0.08,
        max_service_temp_f=400.0,
    ),
    "Dopeless_Coating": ThreadCompoundSpec(
        name="Dopeless_Coating", base_kf=0.04,
        temp_coefficient=-0.00005, shear_rate_exponent=-0.05,
        max_service_temp_f=450.0,
        degradation_rate=0.0002,
    ),
}


# ═══════════════════════════════════════════════════════════════════
# Pipe Specifications (API 5B / 5CT / RP 7G)
# ═══════════════════════════════════════════════════════════════════

@dataclass
class PipeSpec:
    """Thread geometry and torque specs for a specific pipe size/grade/connection."""
    name: str
    od_inches: float
    weight_per_foot: float
    grade: str
    connection_type: str                # LTC, BTC, STC, PREMIUM, DRILL_PIPE
    connection_category: ConnectionCategory = ConnectionCategory.API_8RD_LTC
    optimum_torque_ftlbs: float = 0.0
    min_torque_ftlbs: float = 0.0
    max_torque_ftlbs: float = 0.0
    thread_pitch_tpi: float = 8.0
    thread_taper_ipf: float = 0.0625
    turns_to_shoulder: float = 5.0
    delta_turns: float = 0.04
    seal_diameter_inches: float = 0.0

    # Thread geometry (API 5B tables)
    thread_compound_kf: float = 0.08
    thread_length_inches: float = 0.0
    pitch_diameter_inches: float = 0.0
    root_diameter_inches: float = 0.0
    thread_height_inches: float = 0.0
    lead_inches: float = 0.0
    taper_half_angle_deg: float = 0.0
    thread_flank_angle_deg: float = 60.0
    hand_tight_turns: float = 0.0
    yield_strength_psi: float = 0.0

    # Premium connection fields (Section 3.2.3)
    shoulder_torque_min_ftlbs: float = 0.0     # Min shoulder torque
    shoulder_torque_max_ftlbs: float = 0.0     # Max shoulder torque
    yield_torque_ftlbs: float = 0.0            # Damage threshold
    seal_engagement_turn_fraction: float = 0.0  # Fraction of power-tight where seal engages
    power_tight_slope_ftlbs_per_turn: float = 0.0  # Expected slope in power-tight zone

    # Drill pipe fields
    pin_od_inches: float = 0.0
    box_od_inches: float = 0.0
    breakout_torque_ftlbs: float = 0.0

    # Compatibility with machine types
    compatible_machines: Tuple[MachineType, ...] = (
        MachineType.TOP_DRIVE, MachineType.IRON_ROUGHNECK,
        MachineType.POWER_TONG, MachineType.BUCKING_UNIT,
    )

    def __post_init__(self):
        if self.lead_inches == 0 and self.thread_pitch_tpi > 0:
            self.lead_inches = 1.0 / self.thread_pitch_tpi
        if self.taper_half_angle_deg == 0 and self.thread_taper_ipf > 0:
            self.taper_half_angle_deg = np.degrees(
                np.arctan(self.thread_taper_ipf / 24.0)
            )
        if self.thread_height_inches == 0 and self.thread_pitch_tpi > 0:
            if self.connection_category in (
                ConnectionCategory.API_BUTTRESS,
            ):
                self.thread_height_inches = 0.500 / self.thread_pitch_tpi
            else:
                self.thread_height_inches = 0.626 / self.thread_pitch_tpi
        if self.yield_strength_psi == 0:
            grade_obj = GRADE_CATALOG.get(self.grade)
            if grade_obj:
                self.yield_strength_psi = grade_obj.min_yield_ksi * 1000
            else:
                self.yield_strength_psi = 80_000
        if self.seal_diameter_inches == 0:
            self.seal_diameter_inches = self.od_inches * 0.88

    @property
    def total_turns(self) -> float:
        return self.turns_to_shoulder + self.delta_turns

    @property
    def torque_gradient(self) -> float:
        if self.delta_turns > 0:
            return self.optimum_torque_ftlbs / self.delta_turns
        return self.optimum_torque_ftlbs / 0.04

    @property
    def id_inches(self) -> float:
        id_sq = self.od_inches**2 - self.weight_per_foot / 10.68
        return np.sqrt(max(id_sq, 0.1))

    @property
    def wall_thickness_inches(self) -> float:
        return (self.od_inches - self.id_inches) / 2.0

    @property
    def cross_section_area_in2(self) -> float:
        return np.pi / 4.0 * (self.od_inches**2 - self.id_inches**2)

    @property
    def polar_moment_in4(self) -> float:
        return np.pi / 32.0 * (self.od_inches**4 - self.id_inches**4)

    @property
    def is_premium(self) -> bool:
        return self.connection_category in (
            ConnectionCategory.PREMIUM_SHOULDERED,
            ConnectionCategory.PREMIUM_FLUSH,
        )

    @property
    def is_drill_pipe(self) -> bool:
        return self.connection_category == ConnectionCategory.DRILL_PIPE

    @property
    def is_buttress(self) -> bool:
        return self.connection_category == ConnectionCategory.API_BUTTRESS


# ═══════════════════════════════════════════════════════════════════
# Pipe Catalog — API 8-Round Casing (API RP 5C1 Table 1, EXACT values)
# ═══════════════════════════════════════════════════════════════════

def _ltc(name, od, wt, grade, min_t, opt_t, max_t, turns, delta,
         seal_d=0.0, tl=0.0, pd=0.0, ht=0.0):
    return PipeSpec(
        name=name, od_inches=od, weight_per_foot=wt, grade=grade,
        connection_type="LTC", connection_category=ConnectionCategory.API_8RD_LTC,
        optimum_torque_ftlbs=opt_t, min_torque_ftlbs=min_t, max_torque_ftlbs=max_t,
        thread_pitch_tpi=8, thread_taper_ipf=0.0625,
        turns_to_shoulder=turns, delta_turns=delta,
        seal_diameter_inches=seal_d if seal_d else od * 0.88,
        thread_length_inches=tl, pitch_diameter_inches=pd,
        hand_tight_turns=turns * 0.8,
    )

def _stc(name, od, wt, grade, min_t, opt_t, max_t, turns, delta, seal_d=0.0):
    return PipeSpec(
        name=name, od_inches=od, weight_per_foot=wt, grade=grade,
        connection_type="STC", connection_category=ConnectionCategory.API_8RD_STC,
        optimum_torque_ftlbs=opt_t, min_torque_ftlbs=min_t, max_torque_ftlbs=max_t,
        thread_pitch_tpi=8, thread_taper_ipf=0.0625,
        turns_to_shoulder=turns, delta_turns=delta,
        seal_diameter_inches=seal_d if seal_d else od * 0.88,
        hand_tight_turns=turns * 0.8,
    )

def _btc(name, od, wt, grade, min_t, opt_t, max_t, turns, delta, seal_d=0.0):
    return PipeSpec(
        name=name, od_inches=od, weight_per_foot=wt, grade=grade,
        connection_type="BTC", connection_category=ConnectionCategory.API_BUTTRESS,
        optimum_torque_ftlbs=opt_t, min_torque_ftlbs=min_t, max_torque_ftlbs=max_t,
        thread_pitch_tpi=5, thread_taper_ipf=0.0625,
        thread_flank_angle_deg=13.0,  # Buttress: 3° + 10° asymmetric
        turns_to_shoulder=turns, delta_turns=delta,
        seal_diameter_inches=seal_d if seal_d else od * 0.88,
        hand_tight_turns=turns * 0.8,
    )


PIPE_CATALOG: Dict[str, PipeSpec] = {
    # ═══════════════════════════════════════════════════════════════
    # API 8-Round LTC Casing — API RP 5C1 Table 1 (EXACT VALUES)
    # ═══════════════════════════════════════════════════════════════

    # 4-1/2" casing
    "4.5in_11.6lb_J55_LTC": _ltc("4.5in_11.6lb_J55_LTC",
        4.5, 11.60, "J-55", 2_140, 2_680, 3_350, 4.5, 0.25),
    "4.5in_11.6lb_N80_LTC": _ltc("4.5in_11.6lb_N80_LTC",
        4.5, 11.60, "N-80", 2_680, 3_350, 4_190, 4.5, 0.25),
    "4.5in_11.6lb_P110_LTC": _ltc("4.5in_11.6lb_P110_LTC",
        4.5, 11.60, "P-110", 3_620, 4_530, 5_660, 4.5, 0.25),

    # 5-1/2" casing
    "5.5in_17lb_J55_LTC": _ltc("5.5in_17lb_J55_LTC",
        5.5, 17.00, "J-55", 2_870, 3_590, 4_490, 5.5, 0.30),
    "5.5in_17lb_N80_LTC": _ltc("5.5in_17lb_N80_LTC",
        5.5, 17.00, "N-80", 3_480, 4_350, 5_440, 5.5, 0.30),
    "5.5in_17lb_P110_LTC": _ltc("5.5in_17lb_P110_LTC",
        5.5, 17.00, "P-110", 4_580, 5_730, 7_160, 5.5, 0.30),
    "5.5in_23lb_P110_LTC": _ltc("5.5in_23lb_P110_LTC",
        5.5, 23.00, "P-110", 6_620, 8_280, 10_350, 5.5, 0.30),

    # 7" casing
    "7in_23lb_J55_LTC": _ltc("7in_23lb_J55_LTC",
        7.0, 23.00, "J-55", 2_780, 3_470, 4_340, 6.0, 0.35),
    "7in_23lb_N80_LTC": _ltc("7in_23lb_N80_LTC",
        7.0, 23.00, "N-80", 4_010, 5_010, 6_260, 6.0, 0.35),
    "7in_23lb_P110_LTC": _ltc("7in_23lb_P110_LTC",
        7.0, 23.00, "P-110", 5_020, 6_280, 7_850, 6.0, 0.35),
    "7in_29lb_N80_LTC": _ltc("7in_29lb_N80_LTC",
        7.0, 29.00, "N-80", 5_440, 6_800, 8_500, 6.0, 0.35),
    "7in_29lb_P110_LTC": _ltc("7in_29lb_P110_LTC",
        7.0, 29.00, "P-110", 6_960, 8_700, 10_870, 6.0, 0.35),
    "7in_35lb_P110_LTC": _ltc("7in_35lb_P110_LTC",
        7.0, 35.00, "P-110", 9_070, 11_340, 14_170, 6.0, 0.35),

    # 9-5/8" casing
    "9.625in_36lb_J55_LTC": _ltc("9.625in_36lb_J55_LTC",
        9.625, 36.00, "J-55", 5_560, 6_950, 8_690, 8.0, 0.50),
    "9.625in_36lb_N80_LTC": _ltc("9.625in_36lb_N80_LTC",
        9.625, 36.00, "N-80", 6_600, 8_250, 10_310, 8.0, 0.50),
    "9.625in_40lb_N80_LTC": _ltc("9.625in_40lb_N80_LTC",
        9.625, 40.00, "N-80", 7_660, 9_580, 11_970, 8.0, 0.50),
    "9.625in_40lb_P110_LTC": _ltc("9.625in_40lb_P110_LTC",
        9.625, 40.00, "P-110", 9_880, 12_350, 15_440, 8.0, 0.50),
    "9.625in_47lb_P110_LTC": _ltc("9.625in_47lb_P110_LTC",
        9.625, 47.00, "P-110", 12_310, 15_390, 19_240, 8.0, 0.50),
    "9.625in_53.5lb_P110_LTC": _ltc("9.625in_53.5lb_P110_LTC",
        9.625, 53.50, "P-110", 14_760, 18_450, 23_060, 8.0, 0.50),

    # 10-3/4" casing
    "10.75in_40.5lb_J55_LTC": _ltc("10.75in_40.5lb_J55_LTC",
        10.75, 40.50, "J-55", 5_730, 7_160, 8_950, 9.0, 0.55),
    "10.75in_40.5lb_N80_LTC": _ltc("10.75in_40.5lb_N80_LTC",
        10.75, 40.50, "N-80", 6_780, 8_480, 10_600, 9.0, 0.55),
    "10.75in_51lb_P110_LTC": _ltc("10.75in_51lb_P110_LTC",
        10.75, 51.00, "P-110", 12_420, 15_530, 19_410, 9.0, 0.55),

    # 11-3/4" casing
    "11.75in_42lb_J55_LTC": _ltc("11.75in_42lb_J55_LTC",
        11.75, 42.00, "J-55", 5_680, 7_100, 8_880, 9.5, 0.60),
    "11.75in_47lb_N80_LTC": _ltc("11.75in_47lb_N80_LTC",
        11.75, 47.00, "N-80", 8_020, 10_030, 12_540, 9.5, 0.60),

    # 13-3/8" casing
    "13.375in_48lb_J55_LTC": _ltc("13.375in_48lb_J55_LTC",
        13.375, 48.00, "J-55", 6_010, 7_510, 9_390, 10.5, 0.70),
    "13.375in_54.5lb_N80_LTC": _ltc("13.375in_54.5lb_N80_LTC",
        13.375, 54.50, "N-80", 8_650, 10_810, 13_510, 10.5, 0.70),
    "13.375in_61lb_N80_LTC": _ltc("13.375in_61lb_N80_LTC",
        13.375, 61.00, "N-80", 10_030, 12_540, 15_680, 10.5, 0.70),
    "13.375in_68lb_P110_LTC": _ltc("13.375in_68lb_P110_LTC",
        13.375, 68.00, "P-110", 14_910, 18_640, 23_300, 10.5, 0.70),
    "13.375in_72lb_P110_LTC": _ltc("13.375in_72lb_P110_LTC",
        13.375, 72.00, "P-110", 16_480, 20_600, 25_750, 10.5, 0.70),

    # ═══════════════════════════════════════════════════════════════
    # API Buttress Thread (BTC) — position-controlled, steeper rise
    # ═══════════════════════════════════════════════════════════════

    "5.5in_23lb_P110_BTC": _btc("5.5in_23lb_P110_BTC",
        5.5, 23.00, "P-110", 5_400, 7_200, 9_000, 4.5, 0.04),
    "7in_26lb_N80_BTC": _btc("7in_26lb_N80_BTC",
        7.0, 26.00, "N-80", 6_380, 8_500, 10_630, 5.0, 0.045),
    "7.625in_33.7lb_N80_BTC": _btc("7.625in_33.7lb_N80_BTC",
        7.625, 33.70, "N-80", 7_650, 10_200, 12_750, 5.2, 0.05),
    "9.625in_36lb_N80_BTC": _btc("9.625in_36lb_N80_BTC",
        9.625, 36.00, "N-80", 9_150, 12_200, 15_250, 5.5, 0.05),
    "9.625in_47lb_P110_BTC": _btc("9.625in_47lb_P110_BTC",
        9.625, 47.00, "P-110", 12_380, 16_500, 20_630, 5.5, 0.05),
    "10.75in_45.5lb_N80_BTC": _btc("10.75in_45.5lb_N80_BTC",
        10.75, 45.50, "N-80", 10_880, 14_500, 18_130, 5.8, 0.055),
    "13.375in_54.5lb_K55_BTC": _btc("13.375in_54.5lb_K55_BTC",
        13.375, 54.50, "K-55", 12_600, 16_800, 21_000, 6.0, 0.06),
    "13.375in_68lb_N80_BTC": _btc("13.375in_68lb_N80_BTC",
        13.375, 68.00, "N-80", 15_750, 21_000, 26_250, 6.0, 0.06),

    # ═══════════════════════════════════════════════════════════════
    # Surface / Conductor Casing (STC)
    # ═══════════════════════════════════════════════════════════════

    "16in_75lb_K55_STC": _stc("16in_75lb_K55_STC",
        16.0, 75.00, "K-55", 13_500, 18_000, 22_500, 6.5, 0.07),
    "20in_94lb_K55_STC": _stc("20in_94lb_K55_STC",
        20.0, 94.00, "K-55", 16_500, 22_000, 27_500, 7.0, 0.08),
}


# ═══════════════════════════════════════════════════════════════════
# Premium Connection Catalog (Section 3.2)
# ═══════════════════════════════════════════════════════════════════

PREMIUM_CATALOG: Dict[str, PipeSpec] = {
    # VAM 21 — Vallourec (Section 3.2.2)
    "7in_29lb_P110_VAM21": PipeSpec(
        name="7in_29lb_P110_VAM21",
        od_inches=7.0, weight_per_foot=29.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=18_500, min_torque_ftlbs=14_800,
        max_torque_ftlbs=22_200, thread_pitch_tpi=5, thread_taper_ipf=0.0625,
        turns_to_shoulder=4.0, delta_turns=0.8,
        seal_diameter_inches=6.1,
        shoulder_torque_min_ftlbs=3_700, shoulder_torque_max_ftlbs=7_400,
        yield_torque_ftlbs=28_000,
        seal_engagement_turn_fraction=0.7,
        power_tight_slope_ftlbs_per_turn=18_500,
    ),
    "9.625in_47lb_P110_VAM21": PipeSpec(
        name="9.625in_47lb_P110_VAM21",
        od_inches=9.625, weight_per_foot=47.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=28_000, min_torque_ftlbs=22_400,
        max_torque_ftlbs=33_600, thread_pitch_tpi=5, thread_taper_ipf=0.0625,
        turns_to_shoulder=5.0, delta_turns=1.0,
        seal_diameter_inches=8.7,
        shoulder_torque_min_ftlbs=5_600, shoulder_torque_max_ftlbs=11_200,
        yield_torque_ftlbs=42_000,
        seal_engagement_turn_fraction=0.65,
        power_tight_slope_ftlbs_per_turn=22_400,
    ),
    "13.375in_68lb_P110_VAM21": PipeSpec(
        name="13.375in_68lb_P110_VAM21",
        od_inches=13.375, weight_per_foot=68.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=45_000, min_torque_ftlbs=36_000,
        max_torque_ftlbs=54_000, thread_pitch_tpi=5, thread_taper_ipf=0.0625,
        turns_to_shoulder=6.0, delta_turns=1.2,
        seal_diameter_inches=12.2,
        shoulder_torque_min_ftlbs=9_000, shoulder_torque_max_ftlbs=18_000,
        yield_torque_ftlbs=67_500,
        seal_engagement_turn_fraction=0.6,
        power_tight_slope_ftlbs_per_turn=30_000,
    ),

    # TenarisHydril Wedge 563 (Section 3.2.2)
    "7in_29lb_P110_W563": PipeSpec(
        name="7in_29lb_P110_W563",
        od_inches=7.0, weight_per_foot=29.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=20_000, min_torque_ftlbs=16_000,
        max_torque_ftlbs=24_000, thread_pitch_tpi=4, thread_taper_ipf=0.0625,
        turns_to_shoulder=3.5, delta_turns=0.6,
        seal_diameter_inches=6.0,
        shoulder_torque_min_ftlbs=4_000, shoulder_torque_max_ftlbs=8_000,
        yield_torque_ftlbs=30_000,
        seal_engagement_turn_fraction=0.75,
        power_tight_slope_ftlbs_per_turn=26_600,
    ),
    "9.625in_47lb_P110_W563": PipeSpec(
        name="9.625in_47lb_P110_W563",
        od_inches=9.625, weight_per_foot=47.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=30_000, min_torque_ftlbs=24_000,
        max_torque_ftlbs=36_000, thread_pitch_tpi=4, thread_taper_ipf=0.0625,
        turns_to_shoulder=4.5, delta_turns=0.8,
        seal_diameter_inches=8.6,
        shoulder_torque_min_ftlbs=6_000, shoulder_torque_max_ftlbs=12_000,
        yield_torque_ftlbs=45_000,
        seal_engagement_turn_fraction=0.7,
        power_tight_slope_ftlbs_per_turn=30_000,
    ),

    # Hunting SEAL-LOK Apex
    "5.5in_23lb_P110_SealLok": PipeSpec(
        name="5.5in_23lb_P110_SealLok",
        od_inches=5.5, weight_per_foot=23.0, grade="P-110",
        connection_type="PREMIUM", connection_category=ConnectionCategory.PREMIUM_SHOULDERED,
        optimum_torque_ftlbs=15_000, min_torque_ftlbs=12_000,
        max_torque_ftlbs=18_000, thread_pitch_tpi=5, thread_taper_ipf=0.0625,
        turns_to_shoulder=4.0, delta_turns=0.7,
        seal_diameter_inches=4.6,
        shoulder_torque_min_ftlbs=3_000, shoulder_torque_max_ftlbs=6_000,
        yield_torque_ftlbs=22_500,
        seal_engagement_turn_fraction=0.65,
        power_tight_slope_ftlbs_per_turn=17_100,
    ),
}


# ═══════════════════════════════════════════════════════════════════
# Drill Pipe Connection Catalog (API 7-2 — Section 3.4)
# ═══════════════════════════════════════════════════════════════════

DRILL_PIPE_CATALOG: Dict[str, PipeSpec] = {
    "NC26_2.375DP": PipeSpec(
        name="NC26_2.375DP", od_inches=2.375, weight_per_foot=6.65,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=7_000, min_torque_ftlbs=6_000,
        max_torque_ftlbs=8_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=3.0, delta_turns=0.5,
        seal_diameter_inches=2.0, pin_od_inches=3.375, box_od_inches=3.875,
        breakout_torque_ftlbs=9_000,
        shoulder_torque_min_ftlbs=2_100, shoulder_torque_max_ftlbs=3_500,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC31_2.875DP": PipeSpec(
        name="NC31_2.875DP", od_inches=2.875, weight_per_foot=10.40,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=10_500, min_torque_ftlbs=9_000,
        max_torque_ftlbs=12_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=3.2, delta_turns=0.5,
        seal_diameter_inches=2.5, pin_od_inches=3.875, box_od_inches=4.625,
        breakout_torque_ftlbs=13_500,
        shoulder_torque_min_ftlbs=3_150, shoulder_torque_max_ftlbs=5_250,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC38_3.5DP": PipeSpec(
        name="NC38_3.5DP", od_inches=3.5, weight_per_foot=13.30,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=17_500, min_torque_ftlbs=15_000,
        max_torque_ftlbs=20_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=3.5, delta_turns=0.6,
        seal_diameter_inches=3.0, pin_od_inches=4.625, box_od_inches=5.250,
        breakout_torque_ftlbs=22_500,
        shoulder_torque_min_ftlbs=5_250, shoulder_torque_max_ftlbs=8_750,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC40_4DP": PipeSpec(
        name="NC40_4DP", od_inches=4.0, weight_per_foot=14.00,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=21_000, min_torque_ftlbs=18_000,
        max_torque_ftlbs=24_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=3.5, delta_turns=0.6,
        seal_diameter_inches=3.4, pin_od_inches=5.000, box_od_inches=5.500,
        breakout_torque_ftlbs=27_000,
        shoulder_torque_min_ftlbs=6_300, shoulder_torque_max_ftlbs=10_500,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC46_4DP_heavy": PipeSpec(
        name="NC46_4DP_heavy", od_inches=4.0, weight_per_foot=15.70,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=25_000, min_torque_ftlbs=22_000,
        max_torque_ftlbs=28_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=3.8, delta_turns=0.6,
        seal_diameter_inches=3.8, pin_od_inches=5.500, box_od_inches=6.250,
        breakout_torque_ftlbs=31_500,
        shoulder_torque_min_ftlbs=7_500, shoulder_torque_max_ftlbs=12_500,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC50_4.5DP": PipeSpec(
        name="NC50_4.5DP", od_inches=4.5, weight_per_foot=16.60,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=35_000, min_torque_ftlbs=30_000,
        max_torque_ftlbs=40_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=4.0, delta_turns=0.7,
        seal_diameter_inches=4.2, pin_od_inches=6.625, box_od_inches=7.000,
        breakout_torque_ftlbs=45_000,
        shoulder_torque_min_ftlbs=10_500, shoulder_torque_max_ftlbs=17_500,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "NC56_5DP": PipeSpec(
        name="NC56_5DP", od_inches=5.0, weight_per_foot=19.50,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=47_500, min_torque_ftlbs=40_000,
        max_torque_ftlbs=55_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=4.2, delta_turns=0.7,
        seal_diameter_inches=4.8, pin_od_inches=7.250, box_od_inches=7.750,
        breakout_torque_ftlbs=62_500,
        shoulder_torque_min_ftlbs=14_250, shoulder_torque_max_ftlbs=23_750,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "6.625REG_5.5DP": PipeSpec(
        name="6.625REG_5.5DP", od_inches=5.5, weight_per_foot=21.90,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=57_500, min_torque_ftlbs=50_000,
        max_torque_ftlbs=65_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=4.5, delta_turns=0.8,
        seal_diameter_inches=5.3, pin_od_inches=7.500, box_od_inches=8.500,
        breakout_torque_ftlbs=75_000,
        shoulder_torque_min_ftlbs=17_250, shoulder_torque_max_ftlbs=28_750,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
    "7.625REG_5.875DP": PipeSpec(
        name="7.625REG_5.875DP", od_inches=5.875, weight_per_foot=23.40,
        grade="S-135", connection_type="DRILL_PIPE",
        connection_category=ConnectionCategory.DRILL_PIPE,
        optimum_torque_ftlbs=70_000, min_torque_ftlbs=60_000,
        max_torque_ftlbs=80_000, thread_pitch_tpi=4,
        thread_taper_ipf=0.0833, turns_to_shoulder=4.8, delta_turns=0.8,
        seal_diameter_inches=5.7, pin_od_inches=8.750, box_od_inches=9.500,
        breakout_torque_ftlbs=90_000,
        shoulder_torque_min_ftlbs=21_000, shoulder_torque_max_ftlbs=35_000,
        compatible_machines=(MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE),
    ),
}


# Merge all catalogs into a single lookup
ALL_CONNECTIONS: Dict[str, PipeSpec] = {}
ALL_CONNECTIONS.update(PIPE_CATALOG)
ALL_CONNECTIONS.update(PREMIUM_CATALOG)
ALL_CONNECTIONS.update(DRILL_PIPE_CATALOG)


# ═══════════════════════════════════════════════════════════════════
# Sensor Noise Profiles by Machine Type (Section 4.4)
# ═══════════════════════════════════════════════════════════════════

@dataclass
class SensorNoiseProfile:
    """Machine-type-specific noise characteristics."""
    torque_snr_db: float
    torque_noise_floor_pct_fs: float    # % of full scale
    rpm_resolution: float               # RPM
    rpm_snr_db: float
    turns_resolution: float             # turns
    pressure_snr_db: float
    temperature_snr_db: float
    hookload_snr_db: float
    emi_60hz_pct_fs: float              # 60Hz EMI as % full scale
    dominant_noise: str                  # Primary noise source description


MACHINE_NOISE_PROFILES: Dict[MachineType, SensorNoiseProfile] = {
    MachineType.TOP_DRIVE: SensorNoiseProfile(
        torque_snr_db=52.0,             # Motor current calc — worst SNR
        torque_noise_floor_pct_fs=0.5,
        rpm_resolution=0.05, rpm_snr_db=72.0,
        turns_resolution=0.002, pressure_snr_db=60.0,
        temperature_snr_db=50.0, hookload_snr_db=57.0,
        emi_60hz_pct_fs=0.5,
        dominant_noise="VFD harmonics, gear mesh, bearing vibration",
    ),
    MachineType.IRON_ROUGHNECK: SensorNoiseProfile(
        torque_snr_db=57.0,             # Pressure transducer
        torque_noise_floor_pct_fs=0.3,
        rpm_resolution=0.1, rpm_snr_db=68.0,
        turns_resolution=0.005, pressure_snr_db=62.0,
        temperature_snr_db=50.0, hookload_snr_db=0.0,  # No hookload
        emi_60hz_pct_fs=0.3,
        dominant_noise="Pump ripple, valve chatter, hydraulic 1/f",
    ),
    MachineType.POWER_TONG: SensorNoiseProfile(
        torque_snr_db=62.0,             # Load cell on tong arm
        torque_noise_floor_pct_fs=0.2,
        rpm_resolution=0.1, rpm_snr_db=65.0,
        turns_resolution=0.003, pressure_snr_db=60.0,
        temperature_snr_db=48.0, hookload_snr_db=0.0,
        emi_60hz_pct_fs=0.4,
        dominant_noise="Vibration, arm compliance, temp drift",
    ),
    MachineType.BUCKING_UNIT: SensorNoiseProfile(
        torque_snr_db=67.0,             # Calibrated load cell — best SNR
        torque_noise_floor_pct_fs=0.1,
        rpm_resolution=0.01, rpm_snr_db=78.0,
        turns_resolution=0.001, pressure_snr_db=65.0,
        temperature_snr_db=52.0, hookload_snr_db=0.0,
        emi_60hz_pct_fs=0.1,
        dominant_noise="Quantization, minimal vibration",
    ),
}


# ═══════════════════════════════════════════════════════════════════
# Simulation Configuration
# ═══════════════════════════════════════════════════════════════════

@dataclass
class SimConfig:
    """Master configuration for the simulation."""

    # --- Machine Type ---
    machine_type: MachineType = MachineType.TOP_DRIVE

    # --- Timing (Section 5.3) ---
    physics_dt: float = 0.01                # Physics timestep (100 Hz)
    sensor_sample_dt: float = 0.01          # Sensor sampling rate (100 Hz)
    modbus_update_dt: float = 0.2           # PLC register update rate (5 Hz)
    dataset_output_rate_hz: float = 100.0   # CSV output rate (100 Hz per reference)
    total_time: float = 120.0               # Max simulation duration (seconds)

    # --- Sub-System Specifications ---
    top_drive: TopDriveSpec = field(default_factory=TopDriveSpec)
    iron_roughneck: IronRoughneckSpec = field(default_factory=IronRoughneckSpec)
    power_tong: PowerTongSpec = field(default_factory=PowerTongSpec)
    bucking_unit: BuckingUnitSpec = field(default_factory=BuckingUnitSpec)
    valve: ValveSpec = field(default_factory=ValveSpec)
    pump: PumpSpec = field(default_factory=PumpSpec)
    oil: OilSpec = field(default_factory=OilSpec)
    compound: ThreadCompoundSpec = field(default_factory=ThreadCompoundSpec)
    pipe_string: PipeStringSpec = field(default_factory=PipeStringSpec)

    # --- Hydraulic (for non-top-drive machines) ---
    motor_displacement_cc: float = 250.0
    motor_efficiency_mech: float = 0.90
    motor_efficiency_vol: float = 0.93
    max_pressure_psi: float = 5000.0
    operating_pressure_psi: float = 2500.0
    hydraulic_tau_ms: float = 120.0
    pressure_overshoot: float = 0.10
    oil_bulk_modulus_psi: float = 200_000

    # --- PID Controller ---
    pid_kp: float = 1.2
    pid_ki: float = 0.5
    pid_kd: float = 0.05
    pid_output_min: float = 0.0
    pid_output_max: float = 100.0
    pid_integral_clamp: float = 50.0
    pid_scan_rate_ms: float = 50.0
    pid_derivative_filter_tau: float = 0.1
    pid_deadband_pct: float = 1.0
    pid_valve_rate_limit_pct_s: float = 500.0
    pid_feedforward_gain: float = 0.3

    # --- Encoder ---
    encoder_cpr: int = 1174
    encoder_index_pulse: bool = True

    # --- Thermal Model (Section 4.3) ---
    ambient_temp_f: float = 75.0
    thermal_capacity_btu_f: float = 50.0
    heat_dissipation_btu_hr: float = 5000
    oil_heat_rate_btu_per_hp: float = 2545 / 3600
    temp_warning_f: float = 140.0
    temp_shutdown_f: float = 180.0
    manifold_thermal_capacity: float = 2.0
    motor_case_thermal_capacity: float = 15.0

    # --- Torque Calculation ---
    torque_cell_capacity_ftlbs: float = 50_000
    torque_cell_accuracy_pct: float = 0.25

    # --- Sensor Noise (defaults; overridden by machine-type profiles) ---
    pressure_snr_db: float = 60.0
    pressure_drift_pct_per_c: float = 0.02
    encoder_jitter_counts: int = 1
    temp_snr_db: float = 50.0
    temp_self_heat_f: float = 0.5
    torque_snr_db: float = 55.0
    torque_creep_pct: float = 0.02
    emi_60hz_amplitude: float = 0.005
    vfd_noise_amplitude: float = 0.003
    pink_noise_alpha: float = 1.0
    hookload_snr_db: float = 57.0

    # --- ADC / PLC ---
    adc_bits: int = 16
    adc_vref: float = 10.0

    # ─── GE CPE305 Register Map (Section 5.1.1 — CANONICAL) ─────
    # These addresses match the reference document exactly.
    # "Confirmed" = verified on Steve's PLC; "Estimated" = educated guess.
    reg_torque: int = 6000            # FLOAT32 — Torque (ft-lb)         [Confirmed]
    reg_rpm: int = 6002               # FLOAT32 — RPM                    [Confirmed]
    reg_pressure: int = 6004          # FLOAT32 — System pressure (PSI)  [Confirmed]
    reg_temperature: int = 6006       # FLOAT32 — Oil temperature (°F)   [Confirmed]
    reg_encoder_counts: int = 6008    # FLOAT32 — Encoder counts         [Confirmed]
    reg_pid_setpoint: int = 6010      # FLOAT32 — PID setpoint           [Estimated]
    reg_pid_error: int = 6012         # FLOAT32 — PID error              [Estimated]
    reg_pid_output: int = 6014        # INT16   — PID output (% * 100)   [Estimated]
    reg_mode: int = 6015              # INT16   — Operating mode (enum)  [Confirmed]
    reg_target_torque: int = 6016     # FLOAT32 — Target torque (ft-lb)  [Estimated]
    reg_turns_count: int = 6018       # FLOAT32 — Accumulated turns      [Estimated]
    reg_fault_code: int = 6020        # INT16   — Fault code (bitmask)   [Estimated]
    reg_state: int = 6021             # INT16   — Connection state       [Confirmed]
    reg_peak_torque: int = 6022       # FLOAT32 — Peak torque (ft-lb)    [Estimated]
    reg_hookload: int = 6024          # FLOAT32 — Hookload (klbs)        [Estimated]
    reg_shoulder_torque: int = 6026   # FLOAT32 — Shoulder torque        [Estimated]
    reg_slope: int = 6028             # FLOAT32 — Slope dT/dN            [Estimated]
    reg_connection_count: int = 6030  # INT16   — Connection count       [Estimated]

    # --- Domain Randomization Ranges (Section 6.1) ---
    rand_friction: Tuple[float, float] = (0.80, 1.35)       # Log-normal
    rand_shoulder_position: Tuple[float, float] = (0.95, 1.05)  # Normal
    rand_power_tight_slope: Tuple[float, float] = (0.85, 1.20)  # Normal
    rand_hydraulic_tau: Tuple[float, float] = (0.6, 1.5)    # Uniform (80-200ms)
    rand_motor_efficiency: Tuple[float, float] = (0.82, 0.95)  # Normal
    rand_pid_gains: Tuple[float, float] = (0.85, 1.15)      # Normal
    rand_ambient_temp: Tuple[float, float] = (-20.0, 120.0) # Uniform
    rand_noise_amp: Tuple[float, float] = (0.70, 1.50)      # Log-normal
    rand_emi_amplitude: Tuple[float, float] = (0.1, 2.0)    # Log-normal (% FS)
    rand_encoder_cpr: Tuple[int, int] = (1000, 2000)        # Discrete
    rand_pipe_tolerance: Tuple[float, float] = (0.97, 1.03)
    rand_rpm_setpoint: Tuple[float, float] = (8.0, 25.0)
    rand_valve_dead_zone: Tuple[float, float] = (1.0, 5.0)  # Wider range per ref
    rand_valve_hysteresis: Tuple[float, float] = (0.5, 4.0)
    rand_oil_viscosity: Tuple[float, float] = (0.6, 1.8)    # Wider per ref (compound viscosity)
    rand_string_length: Tuple[float, float] = (30.0, 120.0)
    rand_compound_kf: Tuple[float, float] = (0.80, 1.35)    # Match friction range
    rand_pump_efficiency: Tuple[float, float] = (0.88, 0.97)
    rand_backlash_deg: Tuple[float, float] = (0.05, 0.3)    # Gearbox backlash
    rand_adc_bits: Tuple[int, int] = (12, 16)               # ADC quantization
    rand_pipe_straightness: Tuple[float, float] = (0.0, 0.05)  # deg/ft

    def apply_machine_noise_profile(self):
        """Override sensor noise defaults with machine-type-specific values."""
        profile = MACHINE_NOISE_PROFILES.get(self.machine_type)
        if profile:
            self.torque_snr_db = profile.torque_snr_db
            self.pressure_snr_db = profile.pressure_snr_db
            self.temp_snr_db = profile.temperature_snr_db
            self.emi_60hz_amplitude = profile.emi_60hz_pct_fs / 100.0
            self.hookload_snr_db = profile.hookload_snr_db

    def randomize(self, rng: Optional[np.random.Generator] = None) -> 'SimConfig':
        """Create a domain-randomized copy of this config (Section 6.1)."""
        rng = rng or np.random.default_rng()
        import copy
        cfg = copy.deepcopy(self)

        # --- Friction (log-normal distribution) ---
        friction_scale = np.exp(rng.normal(0, 0.12))  # ~log-normal around 1.0
        friction_scale = np.clip(friction_scale, *self.rand_friction)
        cfg.compound.base_kf *= friction_scale

        # --- Hydraulic ---
        cfg.motor_efficiency_mech = rng.uniform(*self.rand_motor_efficiency)
        cfg.hydraulic_tau_ms *= rng.uniform(*self.rand_hydraulic_tau)
        cfg.top_drive.viscous_damping_nms *= rng.uniform(0.8, 1.2)
        cfg.top_drive.gearbox_backlash_deg = rng.uniform(*self.rand_backlash_deg)

        # --- AC Motor (for top drive) ---
        cfg.top_drive.motor.efficiency = rng.uniform(0.90, 0.96)
        cfg.top_drive.motor.vfd_response_ms *= rng.uniform(0.7, 1.5)

        # --- Valve ---
        cfg.valve.dead_zone_pct = rng.uniform(*self.rand_valve_dead_zone)
        cfg.valve.hysteresis_pct = rng.uniform(*self.rand_valve_hysteresis)
        cfg.valve.spool_time_constant_ms *= rng.uniform(0.7, 1.4)

        # --- Pump ---
        pump_eff_scale = rng.uniform(*self.rand_pump_efficiency)
        cfg.pump.vol_efficiency_coeffs = tuple(
            min(e * pump_eff_scale / 0.92, 0.99)
            for e in self.pump.vol_efficiency_coeffs
        )

        # --- Oil (wider range per reference — compound viscosity 0.6-1.8x) ---
        visc_scale = rng.uniform(*self.rand_oil_viscosity)
        cfg.oil.kinematic_viscosity_40c_cst *= visc_scale
        cfg.oil.kinematic_viscosity_100c_cst *= visc_scale
        cfg.oil.air_content_pct = rng.uniform(0.5, 5.0)

        # --- Pipe string ---
        cfg.pipe_string.length_ft = rng.uniform(*self.rand_string_length)
        cfg.pipe_string.num_joints = int(rng.integers(1, 4))

        # --- PID ---
        cfg.pid_kp *= rng.uniform(*self.rand_pid_gains)
        cfg.pid_ki *= rng.uniform(*self.rand_pid_gains)
        cfg.pid_kd *= rng.uniform(*self.rand_pid_gains)
        cfg.pid_scan_rate_ms = rng.choice([10.0, 20.0, 50.0, 100.0])

        # --- Sensor noise (log-normal distribution) ---
        noise_scale = np.exp(rng.normal(0, 0.15))
        noise_scale = np.clip(noise_scale, *self.rand_noise_amp)
        cfg.pressure_snr_db += rng.uniform(-5, 5)
        cfg.torque_snr_db += rng.uniform(-5, 5)
        cfg.temp_snr_db += rng.uniform(-5, 5)
        cfg.ambient_temp_f = rng.uniform(*self.rand_ambient_temp)
        cfg.emi_60hz_amplitude *= noise_scale
        cfg.vfd_noise_amplitude *= noise_scale

        # --- Encoder (discrete randomization) ---
        cfg.encoder_cpr = int(rng.integers(*self.rand_encoder_cpr))

        # --- ADC (discrete randomization) ---
        cfg.adc_bits = int(rng.integers(*self.rand_adc_bits))

        # --- Apply machine-specific noise profile ---
        cfg.apply_machine_noise_profile()

        return cfg


### `physics_engine.py`

In [ ]:
%%writefile physics_engine.py
"""
Physics Engine: The Truth Model
=================================
Runs at 100Hz (10ms timestep) to capture hydraulic/electrical transients.

Architecture — seven coupled subsystems:

  1. Oil Property Model (algebraic)
     Walther equation for viscosity-temperature relationship.

  2. Torque-Turn Models (algebraic + state)
     - API 8-Round: Farr equation with 4-phase curve
     - Buttress: Steeper shoulder, position-controlled
     - Premium Shouldered: Distinct shoulder + steep slope + optional seal inflection
     - Drill Pipe: Shouldered with breakout tracking
     Temperature and shear-rate dependent friction (Stribeck curve).
     Thread damage accumulation (Archard wear model).

  3. Drive System Models
     - AC Motor + VFD (top drive): constant-torque/constant-power regions,
       VFD current limiting, regenerative braking
     - Hydraulic Motor (iron roughneck spinner, power tong, bucking unit)
     - Hydraulic Cylinder (iron roughneck torque wrench)

  4. Advanced PID Controller (discrete, scan-rate)
     Runs at PLC scan rate (50ms), not physics rate.
     Bumpless RPM↔torque transfer, derivative filter, dead band,
     output rate limiting, feed-forward, tracking anti-windup.

  5. Multi-Zone Thermal Model (ODEs)
     3-node network: oil_reservoir ↔ manifold ↔ motor_casing.

  6. Pipe String Dynamics (ODE)
     Torsional spring-damper between motor and connection.

  7. Shoulder Detection & Slope Calculator
     Real-time detection of shoulder contact point and rolling dT/dN.

State machine sequences:
  IDLE → APPROACH → SPIN_IN → SHOULDER → POWER_TIGHT → HOLD → COMPLETE
  IDLE → BREAKOUT → BACKOFF → COMPLETE
  (Iron Roughneck): SPIN_IN uses spinner, SHOULDER triggers handoff, POWER_TIGHT uses wrench
  Any → FAULT / STALL / E_STOP
"""
import numpy as np
from dataclasses import dataclass, field
from enum import IntEnum
from typing import Optional, Tuple

from config import (
    SimConfig, PipeSpec, OilSpec, ThreadCompoundSpec,
    MachineType, ConnectionCategory, TorqueMeasurementMethod,
    GRADE_CATALOG,
)


# ═══════════════════════════════════════════════════════════════════
# Enumerations
# ═══════════════════════════════════════════════════════════════════

class ConnectionState(IntEnum):
    IDLE = 0
    APPROACH = 1
    SPIN_IN = 2
    SHOULDER = 3
    POWER_TIGHT = 4
    HOLD = 5
    BREAKOUT = 6
    BACKOFF = 7
    FAULT = 8
    COMPLETE = 9
    STALL = 10
    E_STOP = 11
    FAULT_RECOVERY = 12
    HANDOFF = 13          # Iron roughneck spinner→wrench transition


class OperatingMode(IntEnum):
    IDLE = 0
    THREADING = 1
    BACKOFF = 2
    OSCILLATION = 3
    E_STOP = 4


class FaultCode(IntEnum):
    """Fault code bitmask for %R6020."""
    NONE = 0x0000
    OVER_TORQUE = 0x0001
    UNDER_TORQUE = 0x0002
    CROSS_THREAD = 0x0004
    GALLING = 0x0008
    STALL = 0x0010
    OVER_TEMP = 0x0020
    STICK_SLIP = 0x0040
    STRIPPED_THREAD = 0x0080
    MISALIGNED_STAB = 0x0100
    WRONG_COMPOUND = 0x0200
    WASHOUT = 0x0400
    CONNECTION_JUMP = 0x0800


# ═══════════════════════════════════════════════════════════════════
# Physics State Vector
# ═══════════════════════════════════════════════════════════════════

@dataclass
class PhysicsState:
    """Complete physics state vector at a given instant."""
    # --- Core fields ---
    time: float = 0.0
    turns: float = 0.0
    rpm: float = 0.0
    encoder_counts: int = 0
    torque_ftlbs: float = 0.0
    target_torque_ftlbs: float = 0.0
    pressure_psi: float = 0.0
    valve_command_pct: float = 0.0
    flow_gpm: float = 0.0
    oil_temp_f: float = 75.0
    pid_setpoint: float = 0.0
    pid_error: float = 0.0
    pid_output: float = 0.0
    pid_integral: float = 0.0
    connection_state: ConnectionState = ConnectionState.IDLE
    operating_mode: OperatingMode = OperatingMode.IDLE

    # --- Fault flags ---
    is_cross_threaded: bool = False
    is_galled: bool = False
    is_over_torque: bool = False
    is_over_temp: bool = False
    is_stalled: bool = False
    is_stick_slip: bool = False
    is_stripped: bool = False
    is_misaligned: bool = False
    is_wrong_compound: bool = False

    # --- Extended state ---
    valve_spool_pct: float = 0.0
    pump_pressure_psi: float = 0.0
    motor_pressure_drop_psi: float = 0.0
    leakage_flow_gpm: float = 0.0
    oil_viscosity_cst: float = 46.0
    motor_mech_efficiency: float = 0.90
    string_twist_deg: float = 0.0
    connection_rpm: float = 0.0
    motor_torque_ftlbs: float = 0.0
    thread_damage: float = 0.0
    compound_friction_kf: float = 0.08
    pump_ripple_psi: float = 0.0
    manifold_temp_f: float = 75.0
    motor_case_temp_f: float = 75.0
    stick_slip_frequency_hz: float = 0.0
    pid_mode: str = "rpm"

    # --- New fields (Section 5.1.1 register map) ---
    fault_code: int = 0               # Bitmask of active faults
    peak_torque_ftlbs: float = 0.0    # Peak torque in this connection
    hookload_klbs: float = 0.0        # Hook load (top drive only)
    shoulder_torque_ftlbs: float = 0.0  # Detected shoulder torque
    slope_dT_dN: float = 0.0          # Current torque-turn slope (ft-lb/turn)
    connection_count: int = 0          # Total connections completed

    # --- Machine-specific ---
    machine_phase: str = "idle"        # Iron roughneck: "spinner" or "wrench"
    vfd_frequency_hz: float = 0.0      # Top drive VFD output frequency
    vfd_current_pct: float = 0.0       # VFD current as % of rated
    motor_speed_rpm: float = 0.0       # Motor shaft RPM (before gearbox)


# ═══════════════════════════════════════════════════════════════════
# Oil Property Model
# ═══════════════════════════════════════════════════════════════════

class OilPropertyModel:
    """Walther equation viscosity-temperature model (ASTM D341)."""

    def __init__(self, oil: OilSpec):
        self.oil = oil
        self.A, self.B = self._compute_walther_coefficients()

    def _compute_walther_coefficients(self) -> Tuple[float, float]:
        T1_K = 273.15 + 40.0
        T2_K = 273.15 + 100.0
        v1 = self.oil.kinematic_viscosity_40c_cst
        v2 = self.oil.kinematic_viscosity_100c_cst
        W1 = np.log10(np.log10(v1 + 0.7))
        W2 = np.log10(np.log10(v2 + 0.7))
        logT1 = np.log10(T1_K)
        logT2 = np.log10(T2_K)
        B = (W1 - W2) / (logT2 - logT1)
        A = W1 + B * logT1
        return A, B

    def viscosity_at_temp(self, temp_f: float) -> float:
        temp_c = (temp_f - 32.0) * 5.0 / 9.0
        temp_k = max(temp_c + 273.15, 250.0)
        W = self.A - self.B * np.log10(temp_k)
        W = np.clip(W, -0.7, 1.5)
        viscosity = 10.0 ** (10.0 ** W) - 0.7
        return np.clip(viscosity, 1.0, 10000.0)

    def bulk_modulus_effective(self, pressure_psi: float) -> float:
        p_atm = 14.7
        p = max(pressure_psi, p_atm)
        air_factor = (self.oil.air_content_pct / 100.0) * (p_atm / p) ** (1.0 / 1.4)
        return self.oil.bulk_modulus_psi / (1.0 + air_factor)

    def density_at_temp(self, temp_f: float) -> float:
        temp_c = (temp_f - 32.0) * 5.0 / 9.0
        return self.oil.density_kg_m3 - 0.7 * (temp_c - 15.0)


# ═══════════════════════════════════════════════════════════════════
# Torque-Turn Models
# ═══════════════════════════════════════════════════════════════════

class FarrTorqueTurnModel:
    """API 8-Round thread torque-turn model (Farr equation).

    Farr equation:
      T = (σ_y × A / 12) × [ p/(2π) + r_t × Kf/cos(θ) + r_s × Kf ]

    Four phases: Spin-in → Shoulder → Power-tight → Yield
    """

    def __init__(self, pipe: PipeSpec, compound: ThreadCompoundSpec,
                 rng: Optional[np.random.Generator] = None):
        self.pipe = pipe
        self.compound = compound
        self.rng = rng or np.random.default_rng()

        self._tolerance_scale = self.rng.uniform(0.98, 1.02)
        self._kf_scale = self.rng.uniform(0.90, 1.25)
        self._damage_accumulated: float = 0.0
        self._fault_multiplier: float = 1.0

        # Stribeck friction
        self._kf_static = compound.base_kf * self._kf_scale
        self._kf_kinetic = self._kf_static * 0.70
        self._v_stribeck = 0.01
        self._stribeck_exponent = 1.5
        self._viscous_coeff = 0.001

        self._breakout_ratio = self.rng.uniform(1.20, 1.50)

        self._compute_geometry()
        self._compute_phase_boundaries()
        self._compute_farr_constants()

    def _compute_geometry(self):
        p = self.pipe
        self.lead = p.lead_inches if p.lead_inches > 0 else (1.0 / p.thread_pitch_tpi)
        self.theta_rad = np.radians(p.thread_flank_angle_deg / 2.0)
        if p.pitch_diameter_inches > 0:
            self.r_thread_mean = p.pitch_diameter_inches / 2.0
        else:
            self.r_thread_mean = (p.od_inches - p.thread_height_inches) / 2.0
        self.r_seal = p.seal_diameter_inches / 2.0
        thread_circ = np.pi * (p.pitch_diameter_inches if p.pitch_diameter_inches > 0 else p.od_inches)
        thread_depth = p.thread_height_inches if p.thread_height_inches > 0 else (0.626 / max(p.thread_pitch_tpi, 1))
        self.engagement_area = thread_circ * thread_depth * 0.5
        self.sigma_y = p.yield_strength_psi

    def _compute_phase_boundaries(self):
        p = self.pipe
        if p.thread_length_inches > 0:
            total_thread_turns = p.thread_length_inches / self.lead
            self.t_spin_end = total_thread_turns * 0.85 * self._tolerance_scale
            self.t_shoulder = total_thread_turns * self._tolerance_scale
        else:
            self.t_spin_end = p.turns_to_shoulder * 0.85 * self._tolerance_scale
            self.t_shoulder = p.turns_to_shoulder * self._tolerance_scale
        self.t_power_end = self.t_shoulder + p.delta_turns * self._tolerance_scale
        self.scaled_turns = self.t_shoulder
        self.scaled_delta = p.delta_turns * self._tolerance_scale

    def _compute_farr_constants(self):
        p = self.pipe
        self.C_lead = (self.sigma_y * self.engagement_area / 12.0) * (self.lead / (2.0 * np.pi))
        self.C_thread = (self.sigma_y * self.engagement_area / 12.0) * (self.r_thread_mean / np.cos(self.theta_rad))
        self.C_seal = (self.sigma_y * self.engagement_area / 12.0) * self.r_seal
        self.spin_in_base = p.optimum_torque_ftlbs * self.rng.uniform(0.01, 0.05)
        self.shoulder_base = p.optimum_torque_ftlbs * self.rng.uniform(0.20, 0.40)
        self.scaled_optimum = p.optimum_torque_ftlbs * self._kf_scale
        self.scaled_max = p.max_torque_ftlbs * self._kf_scale
        self.scaled_min = p.min_torque_ftlbs * self._kf_scale
        if self.scaled_delta > 0:
            self.power_gradient = (self.scaled_optimum - self.shoulder_base) / self.scaled_delta
        else:
            self.power_gradient = self.scaled_optimum / 0.04

    def compute_kf(self, temp_f: float, omega_rpm: float) -> float:
        c = self.compound
        delta_t = temp_f - c.reference_temp_f
        kf_temp = c.base_kf * self._kf_scale * (1.0 + c.temp_coefficient * delta_t)
        sliding_velocity = abs(omega_rpm) * 2.0 * np.pi / 60.0 * self.r_thread_mean
        shear_rate = max(sliding_velocity / 0.001, 1.0)
        shear_factor = (shear_rate / c.reference_shear_rate) ** c.shear_rate_exponent
        damage_factor = 1.0 + self._damage_accumulated * c.degradation_rate * 100.0
        kf = kf_temp * shear_factor * damage_factor
        return np.clip(kf, 0.02, 0.30)

    def torque_at_turns(self, turns: float, temp_f: float = 77.0,
                        omega_rpm: float = 15.0) -> float:
        if turns <= 0:
            return 0.0
        kf = self.compute_kf(temp_f, omega_rpm)

        # Phase 1: Spin-in
        if turns < self.t_spin_end:
            progress = turns / self.t_spin_end
            base_torque = self.spin_in_base * (0.3 + 0.7 * progress)
            farr_contribution = self.C_lead * kf * progress * 0.1
            return (base_torque + farr_contribution) * self._fault_multiplier

        # Phase 2: Shoulder transition
        if turns < self.t_shoulder:
            progress = (turns - self.t_spin_end) / (self.t_shoulder - self.t_spin_end)
            base_torque = self.spin_in_base + (self.shoulder_base - self.spin_in_base) * progress
            farr_thread = self.C_thread * kf * progress * 0.3
            return (base_torque + farr_thread) * self._fault_multiplier

        # Phase 3: Power-tight
        if turns < self.t_power_end:
            delta = turns - self.t_shoulder
            normalized_delta = delta / self.scaled_delta if self.scaled_delta > 0 else 0
            linear_component = self.shoulder_base + self.power_gradient * delta
            nonlinear_factor = 1.0 + 2.0 * normalized_delta ** 2
            torque = linear_component * nonlinear_factor
            farr_scaled = (self.C_lead + self.C_thread * kf + self.C_seal * kf) * normalized_delta * 0.1
            torque = max(torque, self.shoulder_base) + farr_scaled
            return min(torque, self.scaled_max * 1.2) * self._fault_multiplier

        # Phase 4: Yield zone
        overshoot_turns = turns - self.t_power_end
        overshoot_gradient = self.power_gradient * 1.5
        torque = self.scaled_optimum + overshoot_gradient * overshoot_turns
        return min(torque, self.scaled_max * 1.5) * self._fault_multiplier

    def breakout_torque_profile(self, reverse_turns: float) -> float:
        t_static = self.scaled_optimum * self._breakout_ratio
        t_kinetic = self.scaled_optimum * 0.30
        torque = t_kinetic + (t_static - t_kinetic) * np.exp(-10.0 * abs(reverse_turns))
        return torque * self._fault_multiplier

    def accumulate_damage(self, contact_pressure: float, sliding_distance: float, dt: float):
        damage_increment = (contact_pressure / 1e6) ** 1.5 * sliding_distance * dt * 1e-6
        self._damage_accumulated = min(self._damage_accumulated + damage_increment, 1.0)

    def set_fault_multiplier(self, multiplier: float):
        self._fault_multiplier = max(multiplier, 0.0)

    def reset_fault_multiplier(self):
        self._fault_multiplier = 1.0

    @property
    def spin_in_torque(self) -> float:
        return self.spin_in_base


class ButtressTorqueTurnModel(FarrTorqueTurnModel):
    """Buttress thread (BTC) model — steeper shoulder, position-controlled.

    Key difference from 8-round: flat flank engagement creates more abrupt
    shoulder and steeper power-tight rise (Section 3.1.3).
    """

    def _compute_phase_boundaries(self):
        super()._compute_phase_boundaries()
        # Buttress has steeper transition — shoulder zone is narrower
        self.t_spin_end = self.t_shoulder * 0.90 * self._tolerance_scale

    def _compute_farr_constants(self):
        super()._compute_farr_constants()
        # Steeper power-tight gradient for buttress
        self.power_gradient *= 1.4
        # More abrupt shoulder
        self.shoulder_base = self.pipe.optimum_torque_ftlbs * self.rng.uniform(0.25, 0.50)


class PremiumConnectionModel(FarrTorqueTurnModel):
    """Premium shouldered connection model (Section 3.2.3).

    Four distinct phases:
      Phase 1 (Spin-in): Same as API but typically fewer turns, lower friction
      Phase 2 (Shoulder contact): STEP CHANGE — rapid rise over 0.02-0.05 turns
      Phase 3 (Shoulder-to-final): Nearly linear, slope is critical quality metric
      Phase 4 (Metal seal engagement): Optional inflection at 60-80% of final torque
    """

    def _compute_phase_boundaries(self):
        p = self.pipe
        # Premium connections have fewer spin-in turns (coarser pitch, fewer turns)
        self.t_spin_end = p.turns_to_shoulder * 0.95 * self._tolerance_scale
        self.t_shoulder = p.turns_to_shoulder * self._tolerance_scale
        self.t_power_end = self.t_shoulder + p.delta_turns * self._tolerance_scale
        self.scaled_turns = self.t_shoulder
        self.scaled_delta = p.delta_turns * self._tolerance_scale

        # Shoulder contact occurs over a very narrow window (0.02-0.05 turns)
        self.shoulder_contact_width = self.rng.uniform(0.02, 0.05)

        # Seal engagement point (fraction of power-tight zone)
        self.seal_engagement_frac = p.seal_engagement_turn_fraction
        if self.seal_engagement_frac == 0:
            self.seal_engagement_frac = self.rng.uniform(0.60, 0.80)

    def _compute_farr_constants(self):
        p = self.pipe
        # Spin-in torque is lower for premium (Dopeless coatings)
        self.spin_in_base = p.optimum_torque_ftlbs * self.rng.uniform(0.005, 0.03)

        # Shoulder torque from spec or randomized
        if p.shoulder_torque_min_ftlbs > 0:
            self.shoulder_base = self.rng.uniform(
                p.shoulder_torque_min_ftlbs, p.shoulder_torque_max_ftlbs
            ) * self._kf_scale
        else:
            self.shoulder_base = p.optimum_torque_ftlbs * self.rng.uniform(0.15, 0.35) * self._kf_scale

        self.scaled_optimum = p.optimum_torque_ftlbs * self._kf_scale
        self.scaled_max = p.max_torque_ftlbs * self._kf_scale
        self.scaled_min = p.min_torque_ftlbs * self._kf_scale

        # Power-tight slope from spec or calculated
        if p.power_tight_slope_ftlbs_per_turn > 0:
            self.power_gradient = p.power_tight_slope_ftlbs_per_turn * self._kf_scale * self.rng.uniform(0.85, 1.15)
        elif self.scaled_delta > 0:
            self.power_gradient = (self.scaled_optimum - self.shoulder_base) / self.scaled_delta
        else:
            self.power_gradient = self.scaled_optimum * 2.0

        # Seal engagement produces a kink (secondary inflection)
        self.seal_kink_magnitude = self.scaled_optimum * self.rng.uniform(0.03, 0.08)

        # Farr constants
        self.C_lead = 0
        self.C_thread = 0
        self.C_seal = 0
        self.engagement_area = 1.0
        self.sigma_y = p.yield_strength_psi
        self.r_thread_mean = p.od_inches / 2.0
        self.r_seal = p.seal_diameter_inches / 2.0
        self.lead = 1.0 / max(p.thread_pitch_tpi, 1)
        self.theta_rad = np.radians(p.thread_flank_angle_deg / 2.0)

    def torque_at_turns(self, turns: float, temp_f: float = 77.0,
                        omega_rpm: float = 15.0) -> float:
        if turns <= 0:
            return 0.0

        kf = self.compute_kf(temp_f, omega_rpm)

        # Phase 1: Spin-in (low friction, free running)
        if turns < self.t_spin_end:
            progress = turns / max(self.t_spin_end, 0.01)
            return self.spin_in_base * (0.2 + 0.8 * progress) * self._fault_multiplier

        # Phase 2: Shoulder contact (STEP CHANGE — rapid rise over 0.02-0.05 turns)
        if turns < self.t_shoulder:
            into_shoulder = turns - self.t_spin_end
            shoulder_zone = self.t_shoulder - self.t_spin_end
            if shoulder_zone > 0:
                frac = into_shoulder / shoulder_zone
                # Sigmoid-shaped rise for natural shoulder contact
                sigmoid = 1.0 / (1.0 + np.exp(-12.0 * (frac - 0.5)))
                torque = self.spin_in_base + (self.shoulder_base - self.spin_in_base) * sigmoid
                return torque * self._fault_multiplier
            return self.shoulder_base * self._fault_multiplier

        # Phase 3: Shoulder-to-final (nearly linear with optional seal kink)
        if turns < self.t_power_end:
            delta = turns - self.t_shoulder
            frac = delta / max(self.scaled_delta, 0.001)

            # Linear rise from shoulder to final
            torque = self.shoulder_base + self.power_gradient * delta

            # Phase 4: Seal engagement kink (optional inflection)
            if frac > self.seal_engagement_frac:
                seal_frac = (frac - self.seal_engagement_frac) / (1.0 - self.seal_engagement_frac)
                torque += self.seal_kink_magnitude * seal_frac

            return min(torque, self.scaled_max * 1.1) * self._fault_multiplier

        # Beyond target — yield zone
        overshoot = turns - self.t_power_end
        torque = self.scaled_optimum + self.power_gradient * 0.5 * overshoot
        yield_torque = self.pipe.yield_torque_ftlbs if self.pipe.yield_torque_ftlbs > 0 else self.scaled_max * 1.3
        return min(torque, yield_torque) * self._fault_multiplier


class DrillPipeConnectionModel(PremiumConnectionModel):
    """Drill pipe connection model (API 7-2 — NC/IF/REG).

    Similar to premium shouldered but:
    - Higher breakout ratio (1.3-1.5x)
    - More pronounced shoulder due to tool joint shoulder face
    - Less seal-related kink (shoulder is primary seal mechanism)
    """

    def _compute_farr_constants(self):
        super()._compute_farr_constants()
        # Drill pipe has stronger shoulder and less seal contribution
        self.shoulder_base *= 1.2
        self.seal_kink_magnitude *= 0.3
        self._breakout_ratio = self.rng.uniform(1.25, 1.50)


def create_torque_model(pipe: PipeSpec, compound: ThreadCompoundSpec,
                        rng: Optional[np.random.Generator] = None):
    """Factory: select appropriate torque-turn model based on connection type."""
    if pipe.is_premium:
        return PremiumConnectionModel(pipe, compound, rng)
    elif pipe.is_drill_pipe:
        return DrillPipeConnectionModel(pipe, compound, rng)
    elif pipe.is_buttress:
        return ButtressTorqueTurnModel(pipe, compound, rng)
    else:
        return FarrTorqueTurnModel(pipe, compound, rng)


# ═══════════════════════════════════════════════════════════════════
# AC Motor + VFD Model (Top Drive — Section 2.1 / 4.2.1)
# ═══════════════════════════════════════════════════════════════════

class ACMotorVFDModel:
    """AC induction motor with Variable Frequency Drive.

    Torque-speed characteristic:
      - Below base speed: T = T_rated (constant torque region)
      - Above base speed: T = T_rated * (RPM_base / RPM) (constant power)
      - VFD current limit caps torque at vfd_current_limit_pct * T_rated

    Motor torque equation:
      T_motor = (P_rated * 5252) / RPM_rated for rated conditions
    """

    def __init__(self, cfg: SimConfig):
        self.cfg = cfg
        td = cfg.top_drive
        motor = td.motor

        self.rated_hp = motor.rated_hp
        self.base_rpm = motor.rated_rpm
        self.rated_torque_ftlbs = motor.rated_hp * 5252.0 / motor.rated_rpm
        self.max_torque_ftlbs = self.rated_torque_ftlbs * motor.max_torque_multiplier
        self.vfd_limit_torque = self.rated_torque_ftlbs * motor.vfd_current_limit_pct / 100.0
        self.gear_ratio = td.gear_ratio
        self.gear_efficiency = td.gearbox_efficiency
        self.backlash_rad = np.radians(td.gearbox_backlash_deg)

        # Rotational dynamics (reflected to output shaft)
        self.J_motor_reflected = motor.rotor_inertia_kgm2 * td.gear_ratio ** 2
        self.J_gearbox = td.gearbox_inertia_kgm2
        self.J_total = self.J_motor_reflected + self.J_gearbox
        self.B_viscous = td.viscous_damping_nms
        self.T_seal = td.seal_friction_nm

        # VFD dynamics
        self.vfd_tau = motor.vfd_response_ms / 1000.0
        self._vfd_torque_cmd = 0.0  # Internal VFD torque command
        self._motor_rpm = 0.0
        self._output_omega = 0.0    # Output shaft rad/s

    def step(self, torque_setpoint_pct: float, load_torque_ftlbs: float,
             dt: float) -> Tuple[float, float, float, float, float]:
        """Step the AC motor model.

        Args:
            torque_setpoint_pct: 0-100% of rated capacity
            load_torque_ftlbs: Load torque at output shaft
            dt: Timestep

        Returns: (output_torque_ftlbs, output_rpm, motor_rpm, vfd_current_pct, vfd_freq_hz)
        """
        # VFD torque command with first-order lag
        target_torque_motor = (torque_setpoint_pct / 100.0) * self.rated_torque_ftlbs
        alpha = dt / (self.vfd_tau + dt)
        self._vfd_torque_cmd += alpha * (target_torque_motor - self._vfd_torque_cmd)

        # Apply VFD current limit
        motor_torque = np.clip(self._vfd_torque_cmd, 0.0, self.vfd_limit_torque)

        # Torque-speed characteristic
        motor_rpm = abs(self._motor_rpm)
        if motor_rpm > self.base_rpm and motor_rpm > 0:
            # Constant power region: reduce available torque
            derating = self.base_rpm / motor_rpm
            motor_torque = min(motor_torque, self.rated_torque_ftlbs * derating)

        # Through gearbox to output shaft
        output_torque_ftlbs = motor_torque * self.gear_ratio * self.gear_efficiency

        # Rotational dynamics on output shaft (all in Nm for calculation)
        output_torque_nm = output_torque_ftlbs * 1.3558
        load_nm = load_torque_ftlbs * 1.3558

        net_torque = output_torque_nm - load_nm - self.B_viscous * self._output_omega
        if abs(self._output_omega) > 0.01:
            net_torque -= self.T_seal * np.sign(self._output_omega)
        elif abs(output_torque_nm - load_nm) > self.T_seal:
            net_torque -= self.T_seal * np.sign(output_torque_nm - load_nm)
        else:
            net_torque = 0.0

        angular_accel = net_torque / max(self.J_total, 0.1)
        self._output_omega += angular_accel * dt
        self._output_omega = np.clip(
            self._output_omega, 0.0,
            self.cfg.top_drive.max_output_rpm * 2.0 * np.pi / 60.0
        )

        output_rpm = self._output_omega * 60.0 / (2.0 * np.pi)
        self._motor_rpm = output_rpm * self.gear_ratio

        # Effective torque delivered to connection
        actual_torque = min(output_torque_ftlbs, load_torque_ftlbs) if load_torque_ftlbs > 0 else 0.0

        # VFD outputs
        vfd_current_pct = (motor_torque / max(self.rated_torque_ftlbs, 1)) * 100.0
        vfd_freq_hz = (self._motor_rpm / self.base_rpm) * 60.0  # Proportional to speed

        return actual_torque, output_rpm, self._motor_rpm, vfd_current_pct, vfd_freq_hz

    def reset(self):
        self._vfd_torque_cmd = 0.0
        self._motor_rpm = 0.0
        self._output_omega = 0.0


# ═══════════════════════════════════════════════════════════════════
# Hydraulic System Models (Iron Roughneck, Power Tong, Bucking Unit)
# ═══════════════════════════════════════════════════════════════════

class ProportionalValve:
    """Spool valve dynamics: command → actual spool position."""

    def __init__(self, spec):
        self.spec = spec
        self.spool_position: float = 0.0
        self._prev_command: float = 0.0
        self._rising: bool = True

    def step(self, command_pct: float, dt: float) -> float:
        if abs(command_pct) < self.spec.dead_zone_pct:
            target = 0.0
        else:
            sign = np.sign(command_pct)
            target = sign * (abs(command_pct) - self.spec.dead_zone_pct) / (
                100.0 - self.spec.dead_zone_pct) * 100.0

        delta_cmd = command_pct - self._prev_command
        half_hyst = self.spec.hysteresis_pct / 2.0
        if delta_cmd > 0:
            if not self._rising:
                target -= half_hyst
            self._rising = True
        elif delta_cmd < 0:
            if self._rising:
                target += half_hyst
            self._rising = False
        self._prev_command = command_pct

        max_change = self.spec.rate_limit_pct_per_s * dt
        delta_spool = np.clip(target - self.spool_position, -max_change, max_change)
        tau = self.spec.spool_time_constant_ms / 1000.0
        alpha = dt / (tau + dt) if tau > 0 else 1.0
        self.spool_position += alpha * delta_spool
        return np.clip(self.spool_position, 0.0, 100.0)

    def reset(self):
        self.spool_position = 0.0
        self._prev_command = 0.0
        self._rising = True


class HydraulicPump:
    """Pump flow/pressure model with volumetric efficiency and ripple."""

    def __init__(self, spec):
        self.spec = spec
        self._shaft_angle: float = 0.0
        d_in3 = spec.displacement_cc * 0.0610237
        self.ideal_flow_gpm = d_in3 * spec.drive_rpm / 231.0

    def flow_at_pressure(self, pressure_psi: float, viscosity_cst: float) -> float:
        p_ratio = np.clip(pressure_psi / self.spec.max_pressure_psi, 0.0, 1.0)
        eta_v = np.interp(p_ratio, [0.25, 0.50, 0.75, 1.00],
                          list(self.spec.vol_efficiency_coeffs))
        visc_factor = np.clip(viscosity_cst / 15.0, 0.6, 1.0)
        return self.ideal_flow_gpm * eta_v * visc_factor

    def ripple(self, dt: float, operating_pressure: float) -> float:
        self._shaft_angle += 2.0 * np.pi * self.spec.drive_rpm / 60.0 * dt
        if self._shaft_angle > 2.0e6:
            self._shaft_angle -= 2.0e6
        freq = self.spec.num_pistons * self.spec.drive_rpm / 60.0
        amplitude = 0.03 * operating_pressure
        t_eff = self._shaft_angle / (self.spec.drive_rpm / 60.0 * 2 * np.pi)
        return amplitude * np.sin(2.0 * np.pi * freq * t_eff)

    def relief_flow(self, pressure_psi: float) -> float:
        if pressure_psi < self.spec.relief_cracking_psi:
            return 0.0
        elif pressure_psi >= self.spec.relief_full_flow_psi:
            return 30.0
        else:
            fraction = (pressure_psi - self.spec.relief_cracking_psi) / (
                self.spec.relief_full_flow_psi - self.spec.relief_cracking_psi)
            return fraction * 30.0

    def reset(self):
        self._shaft_angle = 0.0


class HydraulicDriveModel:
    """Hydraulic motor + valve + pump system (for non-top-drive machines).

    Used by iron roughneck (spinner phase), power tong, and bucking unit.
    """

    def __init__(self, cfg: SimConfig, oil_model: OilPropertyModel):
        self.cfg = cfg
        self.oil_model = oil_model
        self.valve = ProportionalValve(cfg.valve)
        self.pump = HydraulicPump(cfg.pump)

        self.D_motor_in3 = cfg.motor_displacement_cc * 0.0610237
        self.gear_ratio = cfg.top_drive.gear_ratio if cfg.machine_type == MachineType.TOP_DRIVE else 1.0
        self.J_total = 5.0  # Default inertia for hydraulic machines
        self.B_viscous = 8.0
        self.T_coulomb_nm = 15.0
        self.K_leak = 1e-5
        self.V_trapped = cfg.pump.trapped_volume_in3
        self.kp_flow = 1.0 / 70.0
        self.max_output_rpm = 100.0
        self.pressure: float = 0.0
        self.omega: float = 0.0

    def step(self, valve_command_pct: float, load_torque_ftlbs: float,
             viscosity_cst: float, dt: float) -> Tuple[
                 float, float, float, float, float, float, float, float]:
        spool_pos = self.valve.step(valve_command_pct, dt)
        flow_fraction = np.clip(spool_pos * self.kp_flow, 0.0, 1.0)

        pump_flow_gpm = self.pump.flow_at_pressure(self.pressure, viscosity_cst) * flow_fraction
        Q_in = pump_flow_gpm * 231.0 / 60.0

        motor_vol_eff = np.clip(self.cfg.motor_efficiency_vol, 0.8, 0.98)
        Q_motor = self.D_motor_in3 * abs(self.omega) / (2.0 * np.pi) / motor_vol_eff

        visc_ratio = 46.0 / max(viscosity_cst, 1.0)
        Q_leak = self.K_leak * self.pressure * visc_ratio
        Q_leak_gpm = Q_leak * 60.0 / 231.0

        Q_relief = self.pump.relief_flow(self.pressure) * 231.0 / 60.0

        beta_eff = self.oil_model.bulk_modulus_effective(self.pressure)
        dQ = Q_in - Q_motor - Q_leak - Q_relief
        dp_dt = (beta_eff / max(self.V_trapped, 1.0)) * dQ
        self.pressure += dp_dt * dt
        self.pressure = np.clip(self.pressure, 0.0, self.cfg.pump.max_pressure_psi * 1.1)

        ripple = self.pump.ripple(dt, self.pressure)
        display_pressure = self.pressure + ripple

        motor_mech_eff = self._motor_efficiency(self.omega, self.pressure)
        motor_torque_inlbs = self.D_motor_in3 * self.pressure / (2.0 * np.pi) * motor_mech_eff
        motor_torque_ftlbs = motor_torque_inlbs / 12.0
        output_torque_ftlbs = motor_torque_ftlbs * self.gear_ratio

        load_nm = load_torque_ftlbs * 1.3558
        output_nm = output_torque_ftlbs * 1.3558
        net_torque = output_nm - load_nm - self.B_viscous * self.omega
        if abs(self.omega) > 0.01:
            net_torque -= self.T_coulomb_nm * np.sign(self.omega)
        elif abs(output_nm - load_nm) > self.T_coulomb_nm:
            net_torque -= self.T_coulomb_nm * np.sign(output_nm - load_nm)
        else:
            net_torque = 0.0

        self.omega += (net_torque / max(self.J_total, 0.1)) * dt
        self.omega = np.clip(self.omega, 0.0, self.max_output_rpm * 2.0 * np.pi / 60.0)

        output_rpm = self.omega * 60.0 / (2.0 * np.pi)
        actual_torque = min(output_torque_ftlbs, load_torque_ftlbs) if load_torque_ftlbs > 0 else 0.0

        return (display_pressure, actual_torque, output_rpm, pump_flow_gpm,
                spool_pos, Q_leak_gpm, ripple, motor_mech_eff)

    def _motor_efficiency(self, omega: float, pressure: float) -> float:
        rpm = abs(omega * 60.0 / (2.0 * np.pi))
        coulomb_loss = min(0.05 / max(rpm, 0.1), 0.40)
        viscous_loss = 0.001 * rpm / max(self.max_output_rpm, 1.0)
        pressure_loss = 0.02 * pressure / max(self.cfg.pump.max_pressure_psi, 1.0)
        return np.clip(1.0 - coulomb_loss - viscous_loss - pressure_loss, 0.40, 0.95)

    def reset(self):
        self.pressure = 0.0
        self.omega = 0.0
        self.valve.reset()
        self.pump.reset()


class IronRoughneckModel:
    """Dual-phase model: spinner (hydraulic motor) + torque wrench (hydraulic cylinder).

    Section 2.2: The iron roughneck has a DISCONTINUITY at spin-to-torque handoff.
    Spinner stops, torque wrench engages, makeup continues at much lower RPM
    with much higher torque capacity.
    """

    def __init__(self, cfg: SimConfig, oil_model: OilPropertyModel):
        self.cfg = cfg
        self.oil_model = oil_model
        ir = cfg.iron_roughneck

        # Spinner subsystem (hydraulic motor)
        self.spinner_valve = ProportionalValve(cfg.valve)
        self.spinner_pump = HydraulicPump(cfg.pump)
        self.spinner_D_in3 = ir.spinner_motor_displacement_cc * 0.0610237
        self.spinner_max_rpm = ir.spinner_max_rpm
        self.spinner_max_torque = ir.spinner_max_torque_ftlbs

        # Torque wrench subsystem (hydraulic cylinder)
        cyl_area = np.pi / 4.0 * ir.wrench_cylinder_bore_in ** 2
        self.wrench_torque_per_psi = cyl_area * ir.wrench_moment_arm_in / 12.0  # ft-lb/PSI
        self.wrench_max_torque = ir.wrench_max_torque_ftlbs
        self.wrench_sweep_deg = ir.wrench_angular_sweep_deg
        self.wrench_max_pressure = ir.wrench_pressure_psi

        # Handoff state
        self.handoff_delay = ir.handoff_delay_ms / 1000.0
        self.handoff_rpm_threshold = ir.handoff_rpm_threshold
        self._phase = "spinner"  # "spinner", "handoff", "wrench"
        self._handoff_timer = 0.0

        # Shared state
        self.pressure: float = 0.0
        self.omega: float = 0.0
        self._K_leak = 1e-5
        self._V_trapped = cfg.pump.trapped_volume_in3

    def step(self, valve_command_pct: float, load_torque_ftlbs: float,
             viscosity_cst: float, dt: float) -> Tuple[
                 float, float, float, float, float, float, float, float]:
        """Returns same signature as HydraulicDriveModel for compatibility."""

        if self._phase == "spinner":
            return self._step_spinner(valve_command_pct, load_torque_ftlbs, viscosity_cst, dt)
        elif self._phase == "handoff":
            return self._step_handoff(dt)
        else:  # wrench
            return self._step_wrench(valve_command_pct, load_torque_ftlbs, viscosity_cst, dt)

    def _step_spinner(self, cmd, load, visc, dt):
        """Spinner phase: high RPM, low torque."""
        spool = self.spinner_valve.step(cmd, dt)
        flow_frac = np.clip(spool / 100.0, 0.0, 1.0)
        pump_flow = self.spinner_pump.flow_at_pressure(self.pressure, visc) * flow_frac
        Q_in = pump_flow * 231.0 / 60.0

        Q_motor = self.spinner_D_in3 * abs(self.omega) / (2.0 * np.pi) / 0.93
        Q_leak = self._K_leak * self.pressure * 46.0 / max(visc, 1.0)
        Q_relief = self.spinner_pump.relief_flow(self.pressure) * 231.0 / 60.0

        beta = self.oil_model.bulk_modulus_effective(self.pressure)
        self.pressure += (beta / max(self._V_trapped, 1.0)) * (Q_in - Q_motor - Q_leak - Q_relief) * dt
        self.pressure = np.clip(self.pressure, 0.0, self.cfg.iron_roughneck.spinner_pressure_psi * 1.2)

        ripple = self.spinner_pump.ripple(dt, self.pressure)
        motor_torque = self.spinner_D_in3 * self.pressure / (2.0 * np.pi) * 0.88 / 12.0

        load_nm = load * 1.3558
        motor_nm = motor_torque * 1.3558
        net = motor_nm - load_nm - 5.0 * self.omega
        self.omega += (net / 3.0) * dt
        self.omega = np.clip(self.omega, 0.0, self.spinner_max_rpm * 2.0 * np.pi / 60.0)

        rpm = self.omega * 60.0 / (2.0 * np.pi)
        actual_torque = min(motor_torque, load) if load > 0 else 0.0

        return (self.pressure + ripple, actual_torque, rpm, pump_flow,
                spool, Q_leak * 60.0 / 231.0, ripple, 0.88)

    def _step_handoff(self, dt):
        """Transition phase: spinner stopped, wrench engaging."""
        self._handoff_timer += dt
        # Decelerate spinner
        self.omega *= max(0, 1.0 - dt * 10.0)
        rpm = self.omega * 60.0 / (2.0 * np.pi)

        if self._handoff_timer >= self.handoff_delay:
            self._phase = "wrench"
            self.omega = 0.0
            self.pressure = 0.0

        return (self.pressure, 0.0, rpm, 0.0, 0.0, 0.0, 0.0, 0.0)

    def _step_wrench(self, cmd, load, visc, dt):
        """Torque wrench phase: low RPM, high torque via hydraulic cylinder."""
        # Pressure builds proportionally to valve command
        target_pressure = (cmd / 100.0) * self.wrench_max_pressure
        tau = 0.15  # Slower cylinder response
        alpha = dt / (tau + dt)
        self.pressure += alpha * (target_pressure - self.pressure)
        self.pressure = np.clip(self.pressure, 0.0, self.wrench_max_pressure * 1.1)

        # Torque from cylinder pressure × moment arm
        wrench_torque = self.wrench_torque_per_psi * self.pressure
        wrench_torque = min(wrench_torque, self.wrench_max_torque)

        # Very slow rotation (limited angular sweep per stroke)
        # Effective RPM: sweep_deg / 360 * strokes_per_second * 60
        effective_rpm = (self.wrench_sweep_deg / 360.0) * (cmd / 100.0) * 2.0  # ~2 RPM max
        self.omega = effective_rpm * 2.0 * np.pi / 60.0

        actual_torque = min(wrench_torque, load) if load > 0 else 0.0

        return (self.pressure, actual_torque, effective_rpm, 0.0,
                cmd, 0.0, 0.0, 0.88)

    def trigger_handoff(self):
        """Called by state machine when shoulder is detected."""
        if self._phase == "spinner":
            self._phase = "handoff"
            self._handoff_timer = 0.0

    @property
    def current_phase(self) -> str:
        return self._phase

    def reset(self):
        self.pressure = 0.0
        self.omega = 0.0
        self._phase = "spinner"
        self._handoff_timer = 0.0
        self.spinner_valve.reset()
        self.spinner_pump.reset()


# ═══════════════════════════════════════════════════════════════════
# Shoulder Detection & Slope Calculator
# ═══════════════════════════════════════════════════════════════════

class ShoulderDetector:
    """Real-time shoulder contact detection.

    Monitors torque-turn derivative (dT/dN). Shoulder is detected when
    dT/dN exceeds a threshold relative to the spin-in baseline.
    """

    def __init__(self, threshold_multiplier: float = 5.0, window_size: int = 10):
        self.threshold_mult = threshold_multiplier
        self.window_size = window_size
        self._torque_history: list = []
        self._turns_history: list = []
        self._baseline_slope: float = 0.0
        self._shoulder_detected: bool = False
        self._shoulder_torque: float = 0.0
        self._shoulder_turn: float = 0.0

    def update(self, torque: float, turns: float) -> bool:
        """Feed new data point, returns True if shoulder just detected."""
        self._torque_history.append(torque)
        self._turns_history.append(turns)

        if len(self._torque_history) < self.window_size + 1:
            return False

        if self._shoulder_detected:
            return False

        # Compute current slope
        recent_t = self._torque_history[-self.window_size:]
        recent_n = self._turns_history[-self.window_size:]
        dt_dn = (recent_t[-1] - recent_t[0]) / max(recent_n[-1] - recent_n[0], 0.001)

        # Compute baseline slope (from early spin-in data)
        if len(self._torque_history) > 2 * self.window_size and self._baseline_slope == 0:
            early_t = self._torque_history[:self.window_size]
            early_n = self._turns_history[:self.window_size]
            self._baseline_slope = max(
                abs(early_t[-1] - early_t[0]) / max(early_n[-1] - early_n[0], 0.001),
                1.0
            )

        if self._baseline_slope > 0 and dt_dn > self._baseline_slope * self.threshold_mult:
            self._shoulder_detected = True
            self._shoulder_torque = torque
            self._shoulder_turn = turns
            return True

        return False

    @property
    def shoulder_torque(self) -> float:
        return self._shoulder_torque

    @property
    def shoulder_turn(self) -> float:
        return self._shoulder_turn

    @property
    def detected(self) -> bool:
        return self._shoulder_detected

    def reset(self):
        self._torque_history.clear()
        self._turns_history.clear()
        self._baseline_slope = 0.0
        self._shoulder_detected = False
        self._shoulder_torque = 0.0
        self._shoulder_turn = 0.0


class SlopeCalculator:
    """Rolling dT/dN (torque-turn slope) calculator.

    Critical quality metric for premium connections per Section 3.2.3.
    """

    def __init__(self, window_turns: float = 0.05):
        self.window_turns = window_turns
        self._torque_buf: list = []
        self._turns_buf: list = []

    def update(self, torque: float, turns: float) -> float:
        self._torque_buf.append(torque)
        self._turns_buf.append(turns)

        # Trim buffer to window
        while (len(self._turns_buf) > 2 and
               self._turns_buf[-1] - self._turns_buf[0] > self.window_turns * 2):
            self._torque_buf.pop(0)
            self._turns_buf.pop(0)

        if len(self._turns_buf) < 2:
            return 0.0

        dn = self._turns_buf[-1] - self._turns_buf[0]
        if dn < 0.001:
            return 0.0

        return (self._torque_buf[-1] - self._torque_buf[0]) / dn

    def reset(self):
        self._torque_buf.clear()
        self._turns_buf.clear()


# ═══════════════════════════════════════════════════════════════════
# Advanced PID Controller
# ═══════════════════════════════════════════════════════════════════

class AdvancedPIDController:
    """PID controller matching GE CPE305 PLC behavior."""

    def __init__(self, cfg: SimConfig):
        self.cfg = cfg
        self.scan_dt = cfg.pid_scan_rate_ms / 1000.0
        self._accumulator: float = 0.0
        self.kp = cfg.pid_kp
        self.ki = cfg.pid_ki
        self.kd = cfg.pid_kd
        self.integral: float = 0.0
        self.prev_error: float = 0.0
        self.prev_output: float = 0.0
        self.prev_derivative: float = 0.0
        self.mode: str = "rpm"

    def update(self, setpoint: float, measured: float, dt: float,
               feedforward: float = 0.0) -> Tuple[float, float, float]:
        self._accumulator += dt
        if self._accumulator < self.scan_dt:
            return self.prev_output, self.prev_error, self.integral

        effective_dt = self._accumulator
        self._accumulator = 0.0
        error = setpoint - measured

        if setpoint > 0 and abs(error) < self.cfg.pid_deadband_pct / 100.0 * abs(setpoint):
            error = 0.0

        p_term = self.kp * error

        self.integral += error * effective_dt
        self.integral = np.clip(self.integral, -self.cfg.pid_integral_clamp, self.cfg.pid_integral_clamp)
        i_term = self.ki * self.integral

        tau_d = self.cfg.pid_derivative_filter_tau
        if effective_dt > 0:
            d_raw = self.kd * (error - self.prev_error) / effective_dt
            if tau_d > 0:
                alpha_d = effective_dt / (tau_d + effective_dt)
                d_term = alpha_d * d_raw + (1.0 - alpha_d) * self.prev_derivative
            else:
                d_term = d_raw
            self.prev_derivative = d_term
        else:
            d_term = 0.0

        raw_output = p_term + i_term + d_term + self.cfg.pid_feedforward_gain * feedforward

        max_change = self.cfg.pid_valve_rate_limit_pct_s * effective_dt
        delta = np.clip(raw_output - self.prev_output, -max_change, max_change)
        output = self.prev_output + delta
        output = np.clip(output, self.cfg.pid_output_min, self.cfg.pid_output_max)

        if output != raw_output and self.ki > 0:
            self.integral -= (raw_output - output) / self.ki * 0.1

        self.prev_error = error
        self.prev_output = output
        return output, error, self.integral

    def switch_mode(self, new_mode: str, current_output: float):
        if new_mode != self.mode:
            self.mode = new_mode
            if self.ki > 0:
                self.integral = (current_output - self.kp * self.prev_error) / self.ki
            self.prev_derivative = 0.0

    def reset(self):
        self.integral = 0.0
        self.prev_error = 0.0
        self.prev_output = 0.0
        self.prev_derivative = 0.0
        self._accumulator = 0.0


# ═══════════════════════════════════════════════════════════════════
# Multi-Zone Thermal Model
# ═══════════════════════════════════════════════════════════════════

class MultiZoneThermalModel:
    """3-node thermal network: oil reservoir, manifold, motor casing."""

    def __init__(self, cfg: SimConfig, oil_model: OilPropertyModel):
        self.cfg = cfg
        self.oil_model = oil_model
        ambient = cfg.ambient_temp_f
        self.T_reservoir: float = ambient
        self.T_manifold: float = ambient
        self.T_motor_case: float = ambient
        self.C_reservoir = cfg.thermal_capacity_btu_f
        self.C_manifold = cfg.manifold_thermal_capacity
        self.C_motor_case = cfg.motor_case_thermal_capacity
        self.G_res_ambient = cfg.heat_dissipation_btu_hr / 3600.0 / 65.0
        self.G_manifold_res = 0.5
        self.G_motor_res = 0.2
        self.G_motor_ambient = 0.05

    def step(self, Q_pump_hp: float, Q_valve_hp: float, Q_motor_hp: float,
             dt: float) -> Tuple[float, float, float]:
        btu_per_hp_s = 2545.0 / 3600.0
        ambient = self.cfg.ambient_temp_f
        q_pump = Q_pump_hp * btu_per_hp_s
        q_valve = Q_valve_hp * btu_per_hp_s
        q_motor = Q_motor_hp * btu_per_hp_s

        q_res_to_amb = self.G_res_ambient * (self.T_reservoir - ambient)
        q_man_to_res = self.G_manifold_res * (self.T_manifold - self.T_reservoir)
        q_motor_to_res = self.G_motor_res * (self.T_motor_case - self.T_reservoir)
        q_motor_to_amb = self.G_motor_ambient * (self.T_motor_case - ambient)

        self.T_reservoir += (q_pump + q_man_to_res + q_motor_to_res - q_res_to_amb) / max(self.C_reservoir, 0.1) * dt
        self.T_manifold += (q_valve - q_man_to_res) / max(self.C_manifold, 0.1) * dt
        self.T_motor_case += (q_motor - q_motor_to_res - q_motor_to_amb) / max(self.C_motor_case, 0.1) * dt

        return self.T_reservoir, self.T_manifold, self.T_motor_case

    def get_oil_temp(self) -> float:
        return self.T_reservoir

    def reset(self):
        ambient = self.cfg.ambient_temp_f
        self.T_reservoir = ambient
        self.T_manifold = ambient
        self.T_motor_case = ambient


# ═══════════════════════════════════════════════════════════════════
# Pipe String Torsional Dynamics
# ═══════════════════════════════════════════════════════════════════

class PipeStringModel:
    """Torsional spring-damper between motor and connection."""

    def __init__(self, pipe: PipeSpec, string_spec, top_drive):
        self.pipe = pipe
        self.string = string_spec

        J_polar = pipe.polar_moment_in4
        L_inches = max(string_spec.length_ft * string_spec.num_joints * 12.0, 12.0)
        G = string_spec.shear_modulus_psi

        self.K_torsional = G * J_polar / L_inches
        self.K_torsional_ftlbs = self.K_torsional / 12.0

        r_mean_m = (pipe.od_inches + pipe.id_inches) / 4.0 * 0.0254
        mass_kg = pipe.weight_per_foot / 2.205 * string_spec.length_ft * string_spec.num_joints
        self.J_string = mass_kg * r_mean_m ** 2

        motor_inertia = getattr(top_drive, 'gearbox_inertia_kgm2', 25.0)
        rotor_inertia = getattr(getattr(top_drive, 'motor', None), 'rotor_inertia_kgm2', 12.0) if hasattr(top_drive, 'motor') else 12.0
        self.J_total = rotor_inertia + motor_inertia + self.J_string

        c_critical = 2.0 * np.sqrt(self.K_torsional_ftlbs * 1.3558 * self.J_total)
        self.C_damping = (string_spec.material_damping_ratio * c_critical +
                          string_spec.joint_friction_ftlbs * string_spec.num_joints * 1.3558)

        self.twist: float = 0.0
        self.twist_rate: float = 0.0

    def step(self, motor_omega: float, connection_omega: float,
             dt: float) -> Tuple[float, float]:
        self.twist += (motor_omega - connection_omega) * dt
        self.twist_rate = motor_omega - connection_omega
        spring_torque_nm = self.K_torsional_ftlbs * 1.3558 * self.twist + self.C_damping * self.twist_rate
        return spring_torque_nm / 1.3558, self.twist

    def natural_frequency_hz(self) -> float:
        K_nm = self.K_torsional_ftlbs * 1.3558
        return np.sqrt(K_nm / max(self.J_total, 0.01)) / (2.0 * np.pi)

    def reset(self):
        self.twist = 0.0
        self.twist_rate = 0.0


# ═══════════════════════════════════════════════════════════════════
# Physics Engine — Main Orchestrator
# ═══════════════════════════════════════════════════════════════════

class PhysicsEngine:
    """Orchestrates all physics subsystems per timestep.

    Supports multiple machine types (top drive, iron roughneck, etc.)
    and multiple connection types (API 8-round, buttress, premium, drill pipe).
    """

    def __init__(self, cfg: SimConfig, pipe: PipeSpec,
                 rng: Optional[np.random.Generator] = None):
        self.cfg = cfg
        self.pipe = pipe
        self.rng = rng or np.random.default_rng()

        self.oil_model = OilPropertyModel(cfg.oil)
        self.torque_model = create_torque_model(pipe, cfg.compound, self.rng)
        self.pid = AdvancedPIDController(cfg)
        self.thermal = MultiZoneThermalModel(cfg, self.oil_model)
        self.string_model = PipeStringModel(pipe, cfg.pipe_string, cfg.top_drive)
        self.shoulder_detector = ShoulderDetector()
        self.slope_calculator = SlopeCalculator()

        # Create appropriate drive model based on machine type
        if cfg.machine_type == MachineType.TOP_DRIVE:
            self.drive = ACMotorVFDModel(cfg)
            self._use_ac_motor = True
        elif cfg.machine_type == MachineType.IRON_ROUGHNECK:
            self.drive = IronRoughneckModel(cfg, self.oil_model)
            self._use_ac_motor = False
        else:
            self.drive = HydraulicDriveModel(cfg, self.oil_model)
            self._use_ac_motor = False

        self.state = PhysicsState(oil_temp_f=cfg.ambient_temp_f)
        self._rpm_setpoint: float = 15.0
        self._torque_setpoint: float = pipe.optimum_torque_ftlbs
        self._breakout_torque: float = 0.0
        self._breakout_rpm: float = 10.0
        self._time: float = 0.0
        self._peak_torque: float = 0.0
        self._stall_timer: float = 0.0
        self._e_stop_timer: float = 0.0
        self._connection_omega: float = 0.0
        self._hookload_klbs: float = 0.0

        # Hookload simulation (top drive only)
        if cfg.machine_type == MachineType.TOP_DRIVE:
            self._hookload_klbs = pipe.weight_per_foot * cfg.pipe_string.length_ft * cfg.pipe_string.num_joints / 1000.0

    def configure_connection(self, rpm: float = 15.0,
                              target_torque: Optional[float] = None,
                              breakout_rpm: float = 10.0):
        self._rpm_setpoint = rpm
        self._torque_setpoint = target_torque or self.pipe.optimum_torque_ftlbs
        self._breakout_rpm = breakout_rpm
        self.state.target_torque_ftlbs = self._torque_setpoint

    def start_threading(self):
        self.state.connection_state = ConnectionState.APPROACH
        self.state.operating_mode = OperatingMode.THREADING
        self.state.turns = 0.0
        self.state.encoder_counts = 0
        self._peak_torque = 0.0
        self.pid.reset()
        self.drive.reset()
        self.string_model.reset()
        self.shoulder_detector.reset()
        self.slope_calculator.reset()
        self.torque_model.reset_fault_multiplier()

    def start_breakout(self):
        self.state.connection_state = ConnectionState.BREAKOUT
        self.state.operating_mode = OperatingMode.BACKOFF
        self._breakout_torque = self.torque_model.breakout_torque_profile(0.0)
        self._peak_torque = 0.0
        self.pid.reset()

    def step(self, dt: Optional[float] = None) -> PhysicsState:
        dt = dt or self.cfg.physics_dt
        self._time += dt
        s = self.state
        s.time = self._time

        # 1. State machine
        self._update_state_machine(dt)

        # 2. Oil viscosity
        viscosity = self.oil_model.viscosity_at_temp(s.oil_temp_f)
        s.oil_viscosity_cst = viscosity

        # 3. Torque demand
        load_torque = self._compute_load_torque()

        # 4. Compound friction
        s.compound_friction_kf = self.torque_model.compute_kf(s.oil_temp_f, s.rpm)

        # 5. PID
        feedforward = self._estimate_feedforward(load_torque)
        pid_out, pid_err, pid_int = self._run_pid(feedforward, dt)
        s.pid_output = pid_out
        s.pid_error = pid_err
        s.pid_integral = pid_int
        s.valve_command_pct = pid_out

        # 6. Drive system (machine-type specific)
        if self._use_ac_motor:
            torque, rpm, motor_rpm, vfd_current, vfd_freq = self.drive.step(
                pid_out, load_torque, dt)
            s.pressure_psi = 0.0  # No hydraulic pressure for AC motor
            s.motor_torque_ftlbs = torque
            s.rpm = rpm
            s.flow_gpm = 0.0
            s.valve_spool_pct = pid_out
            s.leakage_flow_gpm = 0.0
            s.pump_ripple_psi = 0.0
            s.motor_mech_efficiency = self.cfg.top_drive.motor.efficiency
            s.vfd_current_pct = vfd_current
            s.vfd_frequency_hz = vfd_freq
            s.motor_speed_rpm = motor_rpm
        else:
            (pressure, torque, rpm, flow, spool_pos, leak_flow,
             ripple, motor_eff) = self.drive.step(pid_out, load_torque, viscosity, dt)
            s.pressure_psi = pressure
            s.motor_torque_ftlbs = torque
            s.rpm = rpm
            s.flow_gpm = flow
            s.valve_spool_pct = spool_pos
            s.leakage_flow_gpm = leak_flow
            s.pump_ripple_psi = ripple
            s.motor_mech_efficiency = motor_eff

        # 7. String dynamics
        motor_omega = rpm * 2.0 * np.pi / 60.0
        spring_torque, twist = self.string_model.step(motor_omega, self._connection_omega, dt)
        s.string_twist_deg = np.degrees(twist)

        conn_accel = spring_torque * 1.3558 / max(self.string_model.J_string, 0.01)
        self._connection_omega += conn_accel * dt
        self._connection_omega = np.clip(
            self._connection_omega, 0.0,
            motor_omega * 1.1 if motor_omega > 0 else 0.0
        )
        s.connection_rpm = self._connection_omega * 60.0 / (2.0 * np.pi)

        s.torque_ftlbs = min(torque, load_torque) if load_torque > 0 else 0.0

        # 8. Encoder & turns
        effective_rpm = s.connection_rpm if s.connection_rpm > 0 else s.rpm
        if s.operating_mode == OperatingMode.THREADING:
            s.turns += effective_rpm / 60.0 * dt
        elif s.operating_mode == OperatingMode.BACKOFF:
            s.turns -= abs(effective_rpm) / 60.0 * dt
        s.encoder_counts = int(s.turns * self.cfg.encoder_cpr)

        # 9. Shoulder detection & slope calculation
        if s.operating_mode == OperatingMode.THREADING:
            just_detected = self.shoulder_detector.update(s.torque_ftlbs, s.turns)
            if just_detected:
                s.shoulder_torque_ftlbs = self.shoulder_detector.shoulder_torque
            s.slope_dT_dN = self.slope_calculator.update(s.torque_ftlbs, s.turns)

        # 10. Thermal
        Q_pump_hp = self._compute_pump_heat(s.pressure_psi, s.flow_gpm)
        Q_valve_hp = self._compute_valve_heat(s.pressure_psi, s.valve_spool_pct, s.flow_gpm)
        Q_motor_hp = self._compute_motor_heat(s.pressure_psi, s.flow_gpm, s.motor_mech_efficiency)
        T_res, T_man, T_mc = self.thermal.step(Q_pump_hp, Q_valve_hp, Q_motor_hp, dt)
        s.oil_temp_f = T_res
        s.manifold_temp_f = T_man
        s.motor_case_temp_f = T_mc

        # 11. Thread damage
        if s.operating_mode == OperatingMode.THREADING and s.turns > self.torque_model.t_shoulder:
            contact_p = s.torque_ftlbs / max(self.pipe.seal_diameter_inches, 1.0)
            sliding_dist = abs(s.rpm) / 60.0 * self.pipe.seal_diameter_inches * np.pi
            self.torque_model.accumulate_damage(contact_p, sliding_dist, dt)
            s.thread_damage = self.torque_model._damage_accumulated

        # 12. Peak torque tracking
        self._peak_torque = max(self._peak_torque, s.torque_ftlbs)
        s.peak_torque_ftlbs = self._peak_torque

        # 13. Hookload (top drive only — varies with pipe weight and motion)
        if self.cfg.machine_type == MachineType.TOP_DRIVE:
            s.hookload_klbs = self._hookload_klbs + self.rng.normal(0, 0.1)

        # 14. Machine phase (iron roughneck)
        if isinstance(self.drive, IronRoughneckModel):
            s.machine_phase = self.drive.current_phase

        # 15. Fault code assembly
        self._update_fault_code()

        # 16. Stick-slip detection
        self._detect_stick_slip()

        return s

    def _compute_load_torque(self) -> float:
        s = self.state
        if s.operating_mode == OperatingMode.THREADING:
            return self.torque_model.torque_at_turns(s.turns, s.oil_temp_f, s.rpm)
        elif s.operating_mode == OperatingMode.BACKOFF:
            if s.connection_state == ConnectionState.BREAKOUT:
                return self.torque_model.breakout_torque_profile(0.01)
            else:
                turns_rem = max(0, self.torque_model.scaled_turns - abs(s.turns))
                frac = turns_rem / self.torque_model.scaled_turns if self.torque_model.scaled_turns > 0 else 0
                return self.torque_model.spin_in_torque * frac
        return 0.0

    def _estimate_feedforward(self, load_torque: float) -> float:
        if load_torque <= 0:
            return 0.0
        if self._use_ac_motor:
            est_pct = (load_torque / max(self.cfg.top_drive.max_continuous_torque_ftlbs, 1)) * 100.0
            return np.clip(est_pct, 0.0, 100.0)
        else:
            if self.cfg.operating_pressure_psi <= 0:
                return 0.0
            est_pressure = load_torque * 12.0 / max(self.cfg.motor_displacement_cc * 0.0610237 / (2.0 * np.pi) * 0.9, 0.01)
            est_valve = est_pressure / (self.cfg.operating_pressure_psi / 70.0)
            return np.clip(est_valve, 0.0, 100.0)

    def _run_pid(self, feedforward: float, dt: float) -> Tuple[float, float, float]:
        s = self.state
        if s.connection_state in (ConnectionState.SPIN_IN, ConnectionState.SHOULDER, ConnectionState.HANDOFF):
            s.pid_setpoint = self._rpm_setpoint
            s.pid_mode = "rpm"
            return self.pid.update(self._rpm_setpoint, s.rpm, dt, feedforward)
        elif s.connection_state == ConnectionState.POWER_TIGHT:
            s.pid_setpoint = self._torque_setpoint
            s.pid_mode = "torque"
            return self.pid.update(self._torque_setpoint, s.torque_ftlbs, dt, feedforward)
        elif s.connection_state in (ConnectionState.BREAKOUT, ConnectionState.BACKOFF):
            s.pid_setpoint = self._breakout_rpm
            s.pid_mode = "rpm"
            return self.pid.update(self._breakout_rpm, abs(s.rpm), dt, feedforward)
        elif s.connection_state == ConnectionState.APPROACH:
            s.pid_setpoint = 5.0
            s.pid_mode = "rpm"
            return self.pid.update(5.0, s.rpm, dt, 0.0)
        else:
            s.pid_setpoint = 0.0
            s.pid_mode = "rpm"
            return 0.0, 0.0, 0.0

    def _compute_pump_heat(self, pressure: float, flow: float) -> float:
        if self._use_ac_motor:
            # AC motor heat is from I²R losses
            return self.cfg.top_drive.motor.rated_hp * (1.0 - self.cfg.top_drive.motor.efficiency) * (self.state.vfd_current_pct / 100.0) ** 2
        pump_eff = self.cfg.pump.mech_efficiency
        hp = pressure * flow / 1714.0
        return hp * (1.0 - pump_eff) / max(pump_eff, 0.1)

    def _compute_valve_heat(self, pressure: float, spool: float, flow: float) -> float:
        if self._use_ac_motor:
            return 0.0  # No valve heat loss for AC motor
        drop = pressure * (1.0 - spool / 100.0) * 0.3
        return drop * flow / 1714.0

    def _compute_motor_heat(self, pressure: float, flow: float, eff: float) -> float:
        if self._use_ac_motor:
            return 0.0  # Already accounted in pump_heat for AC motor
        return pressure * flow / 1714.0 * (1.0 - eff)

    def _update_fault_code(self):
        s = self.state
        code = FaultCode.NONE
        if s.is_over_torque:
            code |= FaultCode.OVER_TORQUE
        if s.is_cross_threaded:
            code |= FaultCode.CROSS_THREAD
        if s.is_galled:
            code |= FaultCode.GALLING
        if s.is_stalled:
            code |= FaultCode.STALL
        if s.is_over_temp:
            code |= FaultCode.OVER_TEMP
        if s.is_stick_slip:
            code |= FaultCode.STICK_SLIP
        if s.is_stripped:
            code |= FaultCode.STRIPPED_THREAD
        if s.is_misaligned:
            code |= FaultCode.MISALIGNED_STAB
        if s.is_wrong_compound:
            code |= FaultCode.WRONG_COMPOUND
        s.fault_code = int(code)

        # Over-torque check
        if s.torque_ftlbs > self.pipe.max_torque_ftlbs * self.torque_model._kf_scale:
            s.is_over_torque = True
        if s.oil_temp_f > self.cfg.temp_shutdown_f:
            s.is_over_temp = True

    def _detect_stick_slip(self):
        s = self.state
        if s.connection_state not in (ConnectionState.SHOULDER, ConnectionState.POWER_TIGHT):
            s.is_stick_slip = False
            return
        if s.rpm < 5.0 and abs(self.string_model.twist_rate) > 0.5:
            s.is_stick_slip = True
            s.stick_slip_frequency_hz = self.string_model.natural_frequency_hz()
        else:
            s.is_stick_slip = False

    def _update_state_machine(self, dt: float):
        s = self.state
        tm = self.torque_model

        # Stall detection
        if s.operating_mode == OperatingMode.THREADING:
            if s.rpm < 0.5 and s.valve_command_pct > 50.0:
                self._stall_timer += dt
                if self._stall_timer > 3.0:
                    s.is_stalled = True
                    s.connection_state = ConnectionState.STALL
                    return
            else:
                self._stall_timer = 0.0

        if s.connection_state == ConnectionState.E_STOP:
            self._e_stop_timer += dt
            if self._e_stop_timer > 2.0:
                s.connection_state = ConnectionState.FAULT
                s.operating_mode = OperatingMode.IDLE
            return

        if s.connection_state == ConnectionState.IDLE:
            return
        elif s.connection_state == ConnectionState.APPROACH:
            if self._time > 0.5 or s.turns > 0.1:
                s.connection_state = ConnectionState.SPIN_IN
        elif s.connection_state == ConnectionState.SPIN_IN:
            if s.turns >= tm.t_spin_end:
                s.connection_state = ConnectionState.SHOULDER
                # Trigger iron roughneck handoff at shoulder
                if isinstance(self.drive, IronRoughneckModel):
                    self.drive.trigger_handoff()
                    s.connection_state = ConnectionState.HANDOFF
        elif s.connection_state == ConnectionState.HANDOFF:
            # Wait for iron roughneck handoff to complete
            if isinstance(self.drive, IronRoughneckModel):
                if self.drive.current_phase == "wrench":
                    s.connection_state = ConnectionState.SHOULDER
            else:
                s.connection_state = ConnectionState.SHOULDER
        elif s.connection_state == ConnectionState.SHOULDER:
            if s.turns >= tm.t_shoulder:
                s.connection_state = ConnectionState.POWER_TIGHT
                self.pid.switch_mode("torque", s.valve_command_pct)
        elif s.connection_state == ConnectionState.POWER_TIGHT:
            if s.torque_ftlbs >= self._torque_setpoint * 0.98:
                s.connection_state = ConnectionState.HOLD
                s.operating_mode = OperatingMode.IDLE
        elif s.connection_state == ConnectionState.HOLD:
            s.connection_state = ConnectionState.COMPLETE
        elif s.connection_state == ConnectionState.BREAKOUT:
            if self._peak_torque > 0 and s.torque_ftlbs < self._peak_torque * 0.5:
                s.connection_state = ConnectionState.BACKOFF
        elif s.connection_state == ConnectionState.BACKOFF:
            if abs(s.turns) <= 0.1 or s.torque_ftlbs < tm.spin_in_torque * 0.1:
                s.connection_state = ConnectionState.COMPLETE
                s.operating_mode = OperatingMode.IDLE
        elif s.connection_state == ConnectionState.STALL:
            if s.rpm > 2.0:
                s.connection_state = ConnectionState.FAULT_RECOVERY
                s.is_stalled = False
        elif s.connection_state == ConnectionState.FAULT_RECOVERY:
            s.connection_state = ConnectionState.SHOULDER
            s.operating_mode = OperatingMode.THREADING

    def reset(self):
        self.state = PhysicsState(oil_temp_f=self.cfg.ambient_temp_f)
        self.drive.reset()
        self.pid.reset()
        self.thermal.reset()
        self.string_model.reset()
        self.shoulder_detector.reset()
        self.slope_calculator.reset()
        self.torque_model.reset_fault_multiplier()
        self._time = 0.0
        self._peak_torque = 0.0
        self._stall_timer = 0.0
        self._e_stop_timer = 0.0
        self._connection_omega = 0.0


### `sensor_models.py`

In [ ]:
%%writefile sensor_models.py
"""
Sensor Models: Reality Layer
==============================
Transforms ground-truth physics into what the PLC actually measures.

Machine-type-specific noise profiles (Section 4.4):
  - Top Drive:       Torque from motor current calc (SNR 45-60 dB), VFD harmonics
  - Iron Roughneck:  Torque from pressure transducer (SNR 50-65 dB), pump ripple
  - Power Tong:      Torque from load cell (SNR 55-70 dB), arm compliance
  - Bucking Unit:    Calibrated load cell (SNR 60-75 dB), minimal noise

Noise sources per channel:
  1. White noise (thermal/electronic) — Gaussian
  2. Pink noise (1/f) — Voss-McCartney algorithm
  3. EMI (60 Hz) — power line pickup
  4. VFD switching noise — broadband spikes (top drive only)
  5. Pump ripple — at pump frequency (hydraulic machines only)
  6. Drift — temperature-dependent offset
  7. Quantization — ADC discretization (12-16 bit)
"""
import numpy as np
from dataclasses import dataclass
from typing import Optional

from config import SimConfig, MachineType, MACHINE_NOISE_PROFILES


class PinkNoiseGenerator:
    """Voss-McCartney algorithm for 1/f pink noise.
    O(1) per sample vs O(N) for FFT methods.
    """

    def __init__(self, num_sources: int = 16, rng: Optional[np.random.Generator] = None):
        self.rng = rng or np.random.default_rng()
        self.num_sources = num_sources
        self.sources = self.rng.standard_normal(num_sources)
        self.counter = 0

    def sample(self) -> float:
        self.counter += 1
        for i in range(self.num_sources):
            if self.counter % (1 << i) == 0:
                self.sources[i] = self.rng.standard_normal()
        return np.sum(self.sources) / np.sqrt(self.num_sources)

    def samples(self, n: int) -> np.ndarray:
        return np.array([self.sample() for _ in range(n)])


@dataclass
class SensorReading:
    """A single instant of all sensor readings after noise corruption.
    These are the values written into PLC registers.
    """
    encoder_counts: int = 0
    rpm: float = 0.0
    torque_ftlbs: float = 0.0
    pressure_psi: float = 0.0
    oil_temp_f: float = 0.0
    pid_setpoint: float = 0.0
    pid_error: float = 0.0
    pid_output: float = 0.0
    operating_mode: int = 0
    connection_state: int = 0
    pid_kp: float = 0.0
    pid_ki: float = 0.0
    target_torque: float = 0.0
    turns: float = 0.0

    # New fields matching expanded register map
    fault_code: int = 0
    peak_torque: float = 0.0
    hookload_klbs: float = 0.0
    shoulder_torque: float = 0.0
    slope_dT_dN: float = 0.0
    connection_count: int = 0


class SensorModel:
    """Applies realistic, machine-type-specific noise to ground-truth physics."""

    def __init__(self, cfg: SimConfig, rng: Optional[np.random.Generator] = None):
        self.cfg = cfg
        self.rng = rng or np.random.default_rng()
        self.machine_type = cfg.machine_type

        # Get machine-specific noise profile
        profile = MACHINE_NOISE_PROFILES.get(cfg.machine_type)
        if profile:
            torque_snr = profile.torque_snr_db
            pressure_snr = profile.pressure_snr_db
            temp_snr = profile.temperature_snr_db
            self._emi_amplitude = profile.emi_60hz_pct_fs / 100.0
        else:
            torque_snr = cfg.torque_snr_db
            pressure_snr = cfg.pressure_snr_db
            temp_snr = cfg.temp_snr_db
            self._emi_amplitude = cfg.emi_60hz_amplitude

        # Noise standard deviations from SNR
        self.pressure_noise_std = cfg.max_pressure_psi * 10 ** (-pressure_snr / 20)
        self.torque_noise_std = cfg.torque_cell_capacity_ftlbs * 10 ** (-torque_snr / 20)
        self.temp_noise_std = 200.0 * 10 ** (-temp_snr / 20)
        self.hookload_noise_std = 500.0 * 10 ** (-cfg.hookload_snr_db / 20) if cfg.hookload_snr_db > 0 else 0.0

        # RPM noise varies by machine type
        if profile:
            self.rpm_noise_std = profile.rpm_resolution * 1.5
        else:
            self.rpm_noise_std = 0.1

        # Pink noise generators (one per channel)
        self.pressure_pink = PinkNoiseGenerator(rng=self.rng)
        self.torque_pink = PinkNoiseGenerator(rng=self.rng)
        self.temp_pink = PinkNoiseGenerator(rng=self.rng)
        self.hookload_pink = PinkNoiseGenerator(rng=self.rng)
        self.rpm_pink = PinkNoiseGenerator(rng=self.rng)

        # EMI phase (shared across channels)
        self._emi_phase = self.rng.uniform(0, 2 * np.pi)

        # Drift accumulators
        self._pressure_drift = 0.0
        self._torque_drift = 0.0
        self._temp_drift = 0.0
        self._hookload_drift = 0.0

        # Pump ripple parameters (hydraulic machines only)
        self._pump_freq_hz = cfg.pump.num_pistons * cfg.pump.drive_rpm / 60.0
        self._pump_ripple_enabled = cfg.machine_type in (
            MachineType.IRON_ROUGHNECK, MachineType.POWER_TONG, MachineType.BUCKING_UNIT
        )

        # VFD noise (top drive only)
        self._vfd_noise_enabled = cfg.machine_type == MachineType.TOP_DRIVE
        self._vfd_noise_amplitude = cfg.vfd_noise_amplitude

        self._t = 0.0

    def corrupt(self, truth, dt: float) -> SensorReading:
        """Apply noise to ground-truth PhysicsState, return SensorReading."""
        self._t += dt
        reading = SensorReading()

        # ─── Encoder (digital, nearly ideal) ──────────────────
        jitter = self.rng.integers(-self.cfg.encoder_jitter_counts,
                                    self.cfg.encoder_jitter_counts + 1)
        reading.encoder_counts = truth.encoder_counts + jitter

        # ─── RPM (derived from encoder + pink noise) ─────────
        rpm_noise = self.rng.normal(0, self.rpm_noise_std)
        rpm_pink = self.rpm_pink.sample() * self.rpm_noise_std * 0.3
        reading.rpm = max(0, truth.rpm + rpm_noise + rpm_pink)

        # ─── Pressure (strain gauge transducer) ──────────────
        pressure_base = truth.pressure_psi
        # Add pump ripple for hydraulic machines
        if self._pump_ripple_enabled and pressure_base > 0:
            pump_ripple = 0.02 * pressure_base * np.sin(
                2 * np.pi * self._pump_freq_hz * self._t)
            pressure_base += pump_ripple
        reading.pressure_psi = self._apply_analog_noise(
            pressure_base,
            white_std=self.pressure_noise_std,
            pink_gen=self.pressure_pink,
            drift_state='pressure',
            full_scale=self.cfg.max_pressure_psi,
            dt=dt
        )
        reading.pressure_psi = max(0, reading.pressure_psi)

        # ─── Torque (method depends on machine type) ─────────
        torque_base = truth.torque_ftlbs
        # Creep under sustained load
        creep = truth.torque_ftlbs * self.cfg.torque_creep_pct * min(self._t / 60.0, 1.0)
        torque_base += creep

        # Top drive: torque from motor current — additional noise from VFD harmonics
        if self._vfd_noise_enabled:
            # VFD harmonics at switching frequency (~4-8 kHz, aliased)
            vfd_harm = self._vfd_noise_amplitude * self.cfg.torque_cell_capacity_ftlbs
            if self.rng.random() < 0.15:  # 15% chance per sample
                torque_base += self.rng.normal(0, vfd_harm)
            # Gear mesh noise (proportional to RPM)
            gear_mesh_freq = truth.rpm * self.cfg.top_drive.gear_ratio / 60.0
            if gear_mesh_freq > 0:
                gear_noise = 0.001 * self.cfg.torque_cell_capacity_ftlbs * np.sin(
                    2 * np.pi * gear_mesh_freq * self._t)
                torque_base += gear_noise

        reading.torque_ftlbs = self._apply_analog_noise(
            torque_base,
            white_std=self.torque_noise_std,
            pink_gen=self.torque_pink,
            drift_state='torque',
            full_scale=self.cfg.torque_cell_capacity_ftlbs,
            dt=dt
        )
        reading.torque_ftlbs = max(0, reading.torque_ftlbs)

        # ─── Temperature (RTD) ───────────────────────────────
        reading.oil_temp_f = self._apply_analog_noise(
            truth.oil_temp_f + self.cfg.temp_self_heat_f,
            white_std=self.temp_noise_std,
            pink_gen=self.temp_pink,
            drift_state='temp',
            full_scale=200.0,
            dt=dt
        )

        # ─── Hookload (top drive only) ───────────────────────
        if self.cfg.machine_type == MachineType.TOP_DRIVE and hasattr(truth, 'hookload_klbs'):
            reading.hookload_klbs = self._apply_analog_noise(
                truth.hookload_klbs,
                white_std=self.hookload_noise_std,
                pink_gen=self.hookload_pink,
                drift_state='hookload',
                full_scale=500.0,
                dt=dt
            )
        else:
            reading.hookload_klbs = 0.0

        # ─── Pass-through digital values (no noise) ──────────
        reading.pid_setpoint = truth.pid_setpoint
        reading.pid_error = truth.pid_error
        reading.pid_output = truth.pid_output
        reading.operating_mode = int(truth.operating_mode)
        reading.connection_state = int(truth.connection_state)
        reading.pid_kp = self.cfg.pid_kp
        reading.pid_ki = self.cfg.pid_ki
        reading.target_torque = truth.target_torque_ftlbs
        reading.turns = truth.turns

        # New digital fields
        reading.fault_code = truth.fault_code
        reading.peak_torque = truth.peak_torque_ftlbs
        reading.shoulder_torque = truth.shoulder_torque_ftlbs
        reading.slope_dT_dN = truth.slope_dT_dN
        reading.connection_count = truth.connection_count

        # ─── ADC Quantization ────────────────────────────────
        reading = self._quantize(reading)

        return reading

    def _apply_analog_noise(self, true_value: float, white_std: float,
                             pink_gen: PinkNoiseGenerator, drift_state: str,
                             full_scale: float, dt: float) -> float:
        value = true_value

        # 1. Drift (bounded random walk)
        drift_attr = f'_{drift_state}_drift'
        drift = getattr(self, drift_attr, 0.0)
        drift_step = self.rng.normal(0, full_scale * 0.0001 * np.sqrt(dt))
        drift = np.clip(drift + drift_step, -full_scale * 0.005, full_scale * 0.005)
        setattr(self, drift_attr, drift)
        value += drift

        # 2. White noise
        value += self.rng.normal(0, white_std)

        # 3. Pink noise (1/f, ~30% of white level)
        value += pink_gen.sample() * white_std * 0.3

        # 4. 60 Hz EMI (coherent across channels)
        emi = self._emi_amplitude * full_scale * np.sin(
            2 * np.pi * 60.0 * self._t + self._emi_phase
        )
        value += emi

        return value

    def _quantize(self, reading: SensorReading) -> SensorReading:
        bits = self.cfg.adc_bits
        levels = 2 ** bits

        if self.cfg.max_pressure_psi > 0:
            lsb_p = self.cfg.max_pressure_psi / levels
            reading.pressure_psi = round(reading.pressure_psi / lsb_p) * lsb_p

        if self.cfg.torque_cell_capacity_ftlbs > 0:
            lsb_t = self.cfg.torque_cell_capacity_ftlbs / levels
            reading.torque_ftlbs = round(reading.torque_ftlbs / lsb_t) * lsb_t

        lsb_temp = 300.0 / levels
        reading.oil_temp_f = round(reading.oil_temp_f / lsb_temp) * lsb_temp

        if reading.hookload_klbs > 0:
            lsb_h = 500.0 / levels
            reading.hookload_klbs = round(reading.hookload_klbs / lsb_h) * lsb_h

        return reading

    def reset(self):
        self._pressure_drift = 0.0
        self._torque_drift = 0.0
        self._temp_drift = 0.0
        self._hookload_drift = 0.0
        self._t = 0.0
        self.pressure_pink = PinkNoiseGenerator(rng=self.rng)
        self.torque_pink = PinkNoiseGenerator(rng=self.rng)
        self.temp_pink = PinkNoiseGenerator(rng=self.rng)
        self.hookload_pink = PinkNoiseGenerator(rng=self.rng)
        self.rpm_pink = PinkNoiseGenerator(rng=self.rng)


### `scenario.py`

In [ ]:
%%writefile scenario.py
"""
Scenario Generator: Domain Randomization & Failure Modes
=========================================================
Generates diverse pipe threading scenarios for training data.

Scenario distribution targets from reference Section 6.2:
  Normal casing makeup (STC/LTC)   25%    Top drive, power tong
  Normal casing makeup (BTC)       15%    Top drive, power tong
  Normal casing makeup (premium)   10%    Top drive, power tong
  Normal drill pipe makeup         15%    Iron roughneck, top drive
  Normal tubing makeup              5%    Power tong
  Full cycle (make + break)         8%    All
  Cross-thread fault                5%    All
  Galling fault                     4%    All (more common w/ premium)
  Over-torque fault                 3%    All
  Under-torque fault                3%    All
  Stall (motor limit)               2%    Top drive, roughneck
  Wrong compound                    2%    All
  Misaligned stabbing               2%    All
  Multi-connection batch            1%    All

Failure mode signatures from Section 4.1.3:
  Cross-thread:      Spike torque at low turns (<2 turns), erratic
  Galling:           Progressive rise above normal curve, rough/jerky
  Stripped thread:   Torque plateau or drop before target
  Over-torque:       Exceeds max envelope, continues rising
  Under-torque:      Target not reached, RPM at limit
  Wrong compound:    Abnormal shoulder position or slope
  Misaligned stab:   High torque at spin-in, oscillating first 2 turns
"""
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from enum import Enum

from config import (
    SimConfig, PipeSpec, MachineType, ConnectionCategory,
    PIPE_CATALOG, PREMIUM_CATALOG, DRILL_PIPE_CATALOG,
    ALL_CONNECTIONS, COMPOUND_CATALOG,
)


class ScenarioType(Enum):
    NORMAL_CASING_LTC = "normal_casing_ltc"
    NORMAL_CASING_BTC = "normal_casing_btc"
    NORMAL_CASING_PREMIUM = "normal_casing_premium"
    NORMAL_DRILL_PIPE = "normal_drill_pipe"
    NORMAL_TUBING = "normal_tubing"
    NORMAL_BREAKOUT = "normal_breakout"
    FULL_CYCLE = "full_cycle"
    CROSS_THREAD = "cross_thread"
    GALLING = "galling"
    OVER_TORQUE = "over_torque"
    UNDER_TORQUE = "under_torque"
    STALL = "stall"
    WRONG_COMPOUND = "wrong_compound"
    MISALIGNED_STABBING = "misaligned_stabbing"
    STRIPPED_THREAD = "stripped_thread"
    MULTI_CONNECTION = "multi_connection"
    STICK_SLIP = "stick_slip"
    CONNECTION_JUMP = "connection_jump"
    WASHOUT = "washout"
    COLD_START = "cold_start"
    HOT_ENVIRONMENT = "hot_environment"
    STAGED_FAULT = "staged_fault"


@dataclass
class ConnectionScenario:
    """A fully-specified scenario ready for simulation."""
    scenario_type: ScenarioType
    pipe: PipeSpec
    config: SimConfig
    machine_type: MachineType = MachineType.TOP_DRIVE
    rpm_setpoint: float = 15.0
    target_torque: float = 0.0
    breakout_rpm: float = 10.0
    num_connections: int = 1

    # Fault injection parameters
    cross_thread_turns: float = 0.0
    galling_onset_turns: float = 0.0
    galling_rate: float = 0.0
    over_torque_factor: float = 1.0
    stall_pressure_limit: float = 0.0

    # Advanced fault parameters
    stick_slip_enabled: bool = False
    stick_slip_critical_rpm: float = 5.0
    connection_jump_turn: float = 0.0
    connection_jump_severity: float = 0.0
    washout_enabled: bool = False
    washout_leak_rate: float = 0.0
    ambient_temp_override: float = 0.0
    oil_start_temp_override: float = 0.0
    staged_faults: list = field(default_factory=list)
    compound_name: str = "API_Modified_Zinc"
    string_length_ft: float = 30.0
    string_num_joints: int = 1

    # New fault parameters (Section 4.1.3)
    misaligned_severity: float = 0.0        # 0-1 severity of misalignment
    wrong_compound_kf_shift: float = 0.0    # Friction factor shift
    stripped_thread_turn: float = 0.0       # Turn at which stripping occurs
    stripped_thread_severity: float = 0.0   # How much torque drops

    # Metadata
    label: str = ""
    seed: int = 0


class FaultInjector:
    """Encapsulates all fault injection logic per Section 4.1.3."""

    def __init__(self, scenario: ConnectionScenario, rng: np.random.Generator):
        self.scenario = scenario
        self.rng = rng
        self._fault_state = {}

    def apply(self, engine, dt: float):
        state = engine.state

        if self.scenario.cross_thread_turns > 0:
            self._cross_thread(engine, state)

        if self.scenario.galling_onset_turns > 0:
            self._galling(engine, state, dt)

        if self.scenario.connection_jump_turn > 0:
            self._connection_jump(engine, state)

        if self.scenario.washout_enabled:
            self._washout(engine, state, dt)

        if self.scenario.misaligned_severity > 0:
            self._misaligned_stabbing(engine, state)

        if self.scenario.wrong_compound_kf_shift != 0:
            self._wrong_compound(engine, state)

        if self.scenario.stripped_thread_turn > 0:
            self._stripped_thread(engine, state)

        for stage_turn, fault_type, params in self.scenario.staged_faults:
            if state.turns >= stage_turn:
                self._apply_staged_fault(engine, state, fault_type, params, dt)

        if self.scenario.stall_pressure_limit > 0:
            engine.cfg.max_pressure_psi = min(
                engine.cfg.max_pressure_psi,
                self.scenario.stall_pressure_limit
            )

    def _cross_thread(self, engine, state):
        """Section 4.1.3: Spike torque at low turns (<2 turns), erratic.
        Detection: High torque before shoulder.
        """
        if state.turns < self.scenario.cross_thread_turns:
            return
        if not state.is_cross_threaded:
            state.is_cross_threaded = True

        excess = state.turns - self.scenario.cross_thread_turns
        baseline = 1.0 + 3.0 * (1.0 - np.exp(-excess * 2.0))
        ratchet = 0.3 * np.sin(excess * 2.0 * np.pi * 8.0)
        engine.torque_model.set_fault_multiplier(baseline + ratchet)

    def _galling(self, engine, state, dt):
        """Section 4.1.3: Progressive rise above normal curve, rough/jerky.
        Detection: Slope deviation > 2 sigma.
        """
        if state.turns < self.scenario.galling_onset_turns:
            return
        if not state.is_galled:
            state.is_galled = True
        if 'galling_severity' not in self._fault_state:
            self._fault_state['galling_severity'] = 0.0

        contact_pressure = state.torque_ftlbs / max(engine.pipe.seal_diameter_inches, 1.0)
        sliding_velocity = abs(state.rpm) / 60.0 * engine.pipe.seal_diameter_inches * np.pi
        damage_rate = contact_pressure * sliding_velocity * 1e-8
        self._fault_state['galling_severity'] += damage_rate * dt
        severity = self._fault_state['galling_severity']

        # Rough/jerky signature — random torque fluctuations
        jitter = self.rng.normal(0, 0.05 * severity)
        multiplier = 1.0 + severity * self.scenario.galling_rate + jitter

        if severity > 1.0:
            multiplier *= 3.0
            state.is_stalled = True

        engine.torque_model.set_fault_multiplier(multiplier)

    def _misaligned_stabbing(self, engine, state):
        """Section 4.1.3: High torque at spin-in, oscillating first 2 turns.
        Detection: Erratic first 2 turns.
        """
        if state.turns > 2.0:
            # Misalignment effects diminish after threads catch
            severity_decay = max(0, 1.0 - (state.turns - 2.0) * 2.0)
            if severity_decay <= 0:
                engine.torque_model.reset_fault_multiplier()
                return

        if not state.is_misaligned:
            state.is_misaligned = True

        severity = self.scenario.misaligned_severity
        # Oscillating torque (pipe wobbling in the box)
        oscillation = severity * 2.0 * np.sin(state.turns * 2.0 * np.pi * 4.0)
        # Elevated baseline (friction from misaligned threads)
        baseline = 1.0 + severity * 3.0 * max(0, 1.0 - state.turns / 2.0)
        engine.torque_model.set_fault_multiplier(baseline + oscillation)

    def _wrong_compound(self, engine, state):
        """Section 4.1.3: Abnormal shoulder position or slope.
        Detection: Shoulder shift > 0.5 turns.
        """
        if not state.is_wrong_compound:
            state.is_wrong_compound = True
            # Shift the shoulder position by modifying the torque model
            shift = self.scenario.wrong_compound_kf_shift
            engine.torque_model._kf_scale *= (1.0 + shift)
            # Re-compute to reflect shifted friction
            engine.torque_model._compute_farr_constants()

    def _stripped_thread(self, engine, state):
        """Section 4.1.3: Torque plateau or drop before target.
        Detection: Torque stall < 80% target.
        """
        if state.turns < self.scenario.stripped_thread_turn:
            return

        if not state.is_stripped:
            state.is_stripped = True

        excess = state.turns - self.scenario.stripped_thread_turn
        severity = self.scenario.stripped_thread_severity

        # Torque drops as threads strip — exponential decay
        decay = 1.0 - severity * (1.0 - np.exp(-excess * 5.0))
        engine.torque_model.set_fault_multiplier(max(decay, 0.2))

    def _connection_jump(self, engine, state):
        if state.turns < self.scenario.connection_jump_turn:
            return
        if 'jump_applied' not in self._fault_state:
            self._fault_state['jump_applied'] = False
        if not self._fault_state['jump_applied']:
            turn_distance = state.turns - self.scenario.connection_jump_turn
            if turn_distance < 0.1:
                spike = 1.0 + 5.0 * self.scenario.connection_jump_severity * np.exp(-turn_distance * 50.0)
                engine.torque_model.set_fault_multiplier(spike)
            else:
                self._fault_state['jump_applied'] = True
                engine.torque_model.set_fault_multiplier(0.8)

    def _washout(self, engine, state, dt):
        from physics_engine import ConnectionState
        if state.connection_state not in (ConnectionState.HOLD, ConnectionState.COMPLETE):
            return
        if 'washout_start_torque' not in self._fault_state:
            self._fault_state['washout_start_torque'] = state.torque_ftlbs
            self._fault_state['washout_time'] = 0.0
        self._fault_state['washout_time'] += dt
        decay = np.exp(-self.scenario.washout_leak_rate * self._fault_state['washout_time'])
        engine.torque_model.set_fault_multiplier(decay)

    def _apply_staged_fault(self, engine, state, fault_type, params, dt):
        if fault_type == 'mild_galling':
            rate = params.get('rate', 1.0)
            key = f'staged_galling_{fault_type}'
            if key not in self._fault_state:
                self._fault_state[key] = 0.0
            self._fault_state[key] += rate * dt * 0.1
            engine.torque_model.set_fault_multiplier(1.0 + self._fault_state[key])
        elif fault_type == 'severe_galling':
            rate = params.get('rate', 3.0)
            key = f'staged_galling_{fault_type}'
            if key not in self._fault_state:
                self._fault_state[key] = self._fault_state.get('staged_galling_mild_galling', 0.0)
            self._fault_state[key] += rate * dt * 0.1
            engine.torque_model.set_fault_multiplier(1.0 + self._fault_state[key])
        elif fault_type == 'seizure':
            multiplier = params.get('multiplier', 4.0)
            engine.torque_model.set_fault_multiplier(multiplier)
            state.is_stalled = True

    def reset(self):
        self._fault_state = {}


# ═══════════════════════════════════════════════════════════════════
# Pipe Selection Helpers
# ═══════════════════════════════════════════════════════════════════

def _get_pipes_by_category(category: ConnectionCategory) -> List[str]:
    """Get pipe names matching a connection category."""
    return [name for name, pipe in ALL_CONNECTIONS.items()
            if pipe.connection_category == category]

def _get_ltc_stc_pipes() -> List[str]:
    return [name for name, pipe in PIPE_CATALOG.items()
            if pipe.connection_category in (ConnectionCategory.API_8RD_LTC, ConnectionCategory.API_8RD_STC)]

def _get_btc_pipes() -> List[str]:
    return [name for name, pipe in PIPE_CATALOG.items()
            if pipe.connection_category == ConnectionCategory.API_BUTTRESS]

def _get_premium_pipes() -> List[str]:
    return list(PREMIUM_CATALOG.keys())

def _get_drill_pipe_pipes() -> List[str]:
    return list(DRILL_PIPE_CATALOG.keys())

def _get_small_casing_pipes() -> List[str]:
    """Tubing-sized pipes (OD < 5.5")."""
    return [name for name, pipe in PIPE_CATALOG.items() if pipe.od_inches <= 5.5]


class ScenarioGenerator:
    """Generates randomized scenarios matching reference Section 6.2 distribution."""

    def __init__(self, seed: int = 42,
                 base_config: Optional[SimConfig] = None):
        self.rng = np.random.default_rng(seed)
        self.base_config = base_config or SimConfig()

        # Section 6.2 target distribution
        self.weights = {
            ScenarioType.NORMAL_CASING_LTC: 0.25,
            ScenarioType.NORMAL_CASING_BTC: 0.15,
            ScenarioType.NORMAL_CASING_PREMIUM: 0.10,
            ScenarioType.NORMAL_DRILL_PIPE: 0.15,
            ScenarioType.NORMAL_TUBING: 0.02,
            ScenarioType.FULL_CYCLE: 0.08,
            ScenarioType.CROSS_THREAD: 0.05,
            ScenarioType.GALLING: 0.04,
            ScenarioType.OVER_TORQUE: 0.03,
            ScenarioType.UNDER_TORQUE: 0.03,
            ScenarioType.STALL: 0.02,
            ScenarioType.WRONG_COMPOUND: 0.02,
            ScenarioType.MISALIGNED_STABBING: 0.02,
            ScenarioType.STRIPPED_THREAD: 0.01,
            ScenarioType.MULTI_CONNECTION: 0.01,
            ScenarioType.STICK_SLIP: 0.01,
            ScenarioType.WASHOUT: 0.005,
            ScenarioType.CONNECTION_JUMP: 0.005,
        }

    def generate_batch(self, n: int,
                       pipe_names: Optional[List[str]] = None,
                       machine_types: Optional[List[MachineType]] = None
                       ) -> List[ConnectionScenario]:
        types = list(self.weights.keys())
        probs = np.array([self.weights[t] for t in types])
        probs /= probs.sum()

        scenarios = []
        for i in range(n):
            stype = types[self.rng.choice(len(types), p=probs)]
            seed = int(self.rng.integers(0, 2**31))

            # Select pipe and machine based on scenario type
            pipe_name, machine = self._select_pipe_and_machine(stype, pipe_names, machine_types)
            pipe = ALL_CONNECTIONS[pipe_name]

            cfg = self.base_config.randomize(np.random.default_rng(seed))
            cfg.machine_type = machine
            cfg.apply_machine_noise_profile()

            scenario = self._build_scenario(stype, pipe, cfg, seed, machine)
            scenarios.append(scenario)

        return scenarios

    def generate_one(self, scenario_type: ScenarioType,
                     pipe_name: str = "7in_23lb_N80_LTC",
                     machine_type: Optional[MachineType] = None,
                     seed: Optional[int] = None) -> ConnectionScenario:
        pipe = ALL_CONNECTIONS[pipe_name]
        seed = seed or int(self.rng.integers(0, 2**31))
        machine = machine_type or MachineType.TOP_DRIVE
        cfg = self.base_config.randomize(np.random.default_rng(seed))
        cfg.machine_type = machine
        cfg.apply_machine_noise_profile()
        return self._build_scenario(scenario_type, pipe, cfg, seed, machine)

    def _select_pipe_and_machine(self, stype: ScenarioType,
                                  pipe_filter: Optional[List[str]],
                                  machine_filter: Optional[List[MachineType]]
                                  ) -> Tuple[str, MachineType]:
        """Select appropriate pipe and machine based on scenario type."""

        if stype == ScenarioType.NORMAL_CASING_LTC:
            pipes = _get_ltc_stc_pipes()
            machines = [MachineType.TOP_DRIVE, MachineType.POWER_TONG]
        elif stype == ScenarioType.NORMAL_CASING_BTC:
            pipes = _get_btc_pipes()
            machines = [MachineType.TOP_DRIVE, MachineType.POWER_TONG]
        elif stype == ScenarioType.NORMAL_CASING_PREMIUM:
            pipes = _get_premium_pipes()
            machines = [MachineType.TOP_DRIVE, MachineType.POWER_TONG]
        elif stype == ScenarioType.NORMAL_DRILL_PIPE:
            pipes = _get_drill_pipe_pipes()
            machines = [MachineType.IRON_ROUGHNECK, MachineType.TOP_DRIVE]
        elif stype == ScenarioType.NORMAL_TUBING:
            pipes = _get_small_casing_pipes()
            machines = [MachineType.POWER_TONG]
        else:
            # Fault/special scenarios: use any pipe and primary machine types
            all_pipes = list(ALL_CONNECTIONS.keys())
            pipes = all_pipes
            machines = [MachineType.TOP_DRIVE, MachineType.IRON_ROUGHNECK,
                       MachineType.POWER_TONG, MachineType.BUCKING_UNIT]

        # Apply filters
        if pipe_filter:
            pipes = [p for p in pipe_filter if p in ALL_CONNECTIONS]
        if machine_filter:
            machines = [m for m in machine_filter if m in machines] or machine_filter

        if not pipes:
            pipes = list(PIPE_CATALOG.keys())
        if not machines:
            machines = [MachineType.TOP_DRIVE]

        pipe_name = self.rng.choice(pipes)
        machine = self.rng.choice(machines)
        return pipe_name, machine

    def _build_scenario(self, stype: ScenarioType, pipe: PipeSpec,
                        cfg: SimConfig, seed: int,
                        machine: MachineType) -> ConnectionScenario:
        rng = np.random.default_rng(seed)

        rpm = rng.uniform(*cfg.rand_rpm_setpoint)
        breakout_rpm = rng.uniform(8.0, 15.0)

        compound_name = rng.choice(list(COMPOUND_CATALOG.keys()))
        cfg.compound = COMPOUND_CATALOG[compound_name]

        string_length = rng.uniform(*cfg.rand_string_length)
        string_joints = int(rng.integers(1, 4))
        cfg.pipe_string.length_ft = string_length
        cfg.pipe_string.num_joints = string_joints

        scenario = ConnectionScenario(
            scenario_type=stype, pipe=pipe, config=cfg,
            machine_type=machine, rpm_setpoint=rpm,
            breakout_rpm=breakout_rpm, seed=seed,
            compound_name=compound_name,
            string_length_ft=string_length,
            string_num_joints=string_joints,
        )

        # ─── Normal Scenarios ────────────────────────────────

        if stype in (ScenarioType.NORMAL_CASING_LTC, ScenarioType.NORMAL_CASING_BTC,
                     ScenarioType.NORMAL_CASING_PREMIUM, ScenarioType.NORMAL_DRILL_PIPE,
                     ScenarioType.NORMAL_TUBING):
            scenario.target_torque = pipe.optimum_torque_ftlbs * rng.uniform(0.95, 1.05)
            scenario.label = f"Normal {stype.value}: {pipe.name} [{machine.value}] @ {rpm:.0f} RPM"

        elif stype == ScenarioType.NORMAL_BREAKOUT:
            scenario.label = f"Normal breakout: {pipe.name} [{machine.value}]"

        elif stype == ScenarioType.FULL_CYCLE:
            scenario.target_torque = pipe.optimum_torque_ftlbs * rng.uniform(0.95, 1.05)
            scenario.label = f"Full cycle: {pipe.name} [{machine.value}] @ {rpm:.0f} RPM"

        # ─── Fault Scenarios (Section 4.1.3) ──────────────────

        elif stype == ScenarioType.CROSS_THREAD:
            scenario.cross_thread_turns = rng.uniform(0.5, 2.0)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Cross-thread @ {scenario.cross_thread_turns:.1f} turns: {pipe.name}"

        elif stype == ScenarioType.GALLING:
            scenario.galling_onset_turns = pipe.turns_to_shoulder * rng.uniform(0.5, 0.9)
            scenario.galling_rate = rng.uniform(1.5, 3.0)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Galling @ {scenario.galling_onset_turns:.1f} turns: {pipe.name}"

        elif stype == ScenarioType.OVER_TORQUE:
            scenario.over_torque_factor = rng.uniform(1.15, 1.40)
            scenario.target_torque = pipe.optimum_torque_ftlbs * scenario.over_torque_factor
            scenario.label = f"Over-torque ({scenario.over_torque_factor:.0%}): {pipe.name}"

        elif stype == ScenarioType.UNDER_TORQUE:
            scenario.stall_pressure_limit = cfg.operating_pressure_psi * rng.uniform(0.3, 0.6)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Under-torque: {pipe.name}"

        elif stype == ScenarioType.STALL:
            scenario.stall_pressure_limit = cfg.operating_pressure_psi * rng.uniform(0.1, 0.3)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.rpm_setpoint = rng.uniform(3.0, 8.0)
            scenario.label = f"Stall: {pipe.name}"

        elif stype == ScenarioType.WRONG_COMPOUND:
            # Shift friction factor significantly — shoulder appears at wrong position
            scenario.wrong_compound_kf_shift = rng.choice([-0.4, -0.3, 0.3, 0.5, 0.6])
            scenario.target_torque = pipe.optimum_torque_ftlbs
            direction = "low" if scenario.wrong_compound_kf_shift < 0 else "high"
            scenario.label = f"Wrong compound ({direction} friction): {pipe.name}"

        elif stype == ScenarioType.MISALIGNED_STABBING:
            scenario.misaligned_severity = rng.uniform(0.3, 1.0)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Misaligned stab (sev={scenario.misaligned_severity:.1f}): {pipe.name}"

        elif stype == ScenarioType.STRIPPED_THREAD:
            # Stripping occurs during power-tight zone
            scenario.stripped_thread_turn = pipe.turns_to_shoulder + pipe.delta_turns * rng.uniform(0.2, 0.7)
            scenario.stripped_thread_severity = rng.uniform(0.4, 0.9)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Stripped thread @ {scenario.stripped_thread_turn:.1f} turns: {pipe.name}"

        elif stype == ScenarioType.MULTI_CONNECTION:
            scenario.num_connections = int(rng.integers(3, 8))
            scenario.target_torque = pipe.optimum_torque_ftlbs * rng.uniform(0.95, 1.05)
            scenario.config.total_time = scenario.num_connections * 45.0
            scenario.label = f"Multi-connection ({scenario.num_connections}x): {pipe.name}"

        elif stype == ScenarioType.STICK_SLIP:
            scenario.stick_slip_enabled = True
            scenario.stick_slip_critical_rpm = rng.uniform(3.0, 8.0)
            scenario.rpm_setpoint = rng.uniform(3.0, 6.0)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            cfg.pipe_string.length_ft = rng.uniform(60.0, 120.0)
            cfg.pipe_string.num_joints = int(rng.integers(2, 5))
            scenario.label = f"Stick-slip @ {scenario.rpm_setpoint:.0f} RPM: {pipe.name}"

        elif stype == ScenarioType.CONNECTION_JUMP:
            scenario.connection_jump_turn = pipe.turns_to_shoulder * rng.uniform(0.3, 0.7)
            scenario.connection_jump_severity = rng.uniform(0.3, 1.0)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Conn jump @ {scenario.connection_jump_turn:.1f} turns: {pipe.name}"

        elif stype == ScenarioType.WASHOUT:
            scenario.washout_enabled = True
            scenario.washout_leak_rate = rng.uniform(0.1, 0.5)
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Washout: {pipe.name}"

        elif stype == ScenarioType.STAGED_FAULT:
            galling_start = pipe.turns_to_shoulder * rng.uniform(0.6, 0.8)
            severe_turn = pipe.turns_to_shoulder * rng.uniform(0.9, 1.0)
            seizure_turn = pipe.turns_to_shoulder + pipe.delta_turns * rng.uniform(0.3, 0.7)
            scenario.staged_faults = [
                (galling_start, 'mild_galling', {'rate': rng.uniform(0.5, 1.0)}),
                (severe_turn, 'severe_galling', {'rate': rng.uniform(2.0, 4.0)}),
                (seizure_turn, 'seizure', {'multiplier': rng.uniform(3.0, 5.0)}),
            ]
            scenario.target_torque = pipe.optimum_torque_ftlbs
            scenario.label = f"Staged fault: {pipe.name}"

        return scenario

    def get_distribution_summary(self) -> str:
        lines = ["Scenario Distribution (Section 6.2 targets):"]
        for stype, weight in sorted(self.weights.items(), key=lambda x: -x[1]):
            lines.append(f"  {stype.value:30s}  {weight:5.1%}")
        lines.append(f"\nConnection catalogs:")
        lines.append(f"  API 8-round (LTC/STC): {len(_get_ltc_stc_pipes())} types")
        lines.append(f"  API Buttress (BTC):    {len(_get_btc_pipes())} types")
        lines.append(f"  Premium shouldered:    {len(_get_premium_pipes())} types")
        lines.append(f"  Drill pipe (NC/IF):    {len(_get_drill_pipe_pipes())} types")
        lines.append(f"  Total connections:     {len(ALL_CONNECTIONS)} types")
        lines.append(f"\nCompound catalog: {len(COMPOUND_CATALOG)} types")
        lines.append(f"  " + ", ".join(COMPOUND_CATALOG.keys()))
        lines.append(f"\nMachine types: {', '.join(m.value for m in MachineType)}")
        return "\n".join(lines)


### `modbus_server.py`

In [ ]:
%%writefile modbus_server.py
"""
Modbus TCP Server: The Interface Layer
========================================
Zero-dependency implementation using only Python stdlib.

Register map matches Section 5.1.1 of the reference document exactly:
  %R6000-6001  FLOAT32  Torque (ft-lb)           [Confirmed]
  %R6002-6003  FLOAT32  RPM                      [Confirmed]
  %R6004-6005  FLOAT32  Pressure (PSI)           [Confirmed]
  %R6006-6007  FLOAT32  Oil Temperature (°F)     [Confirmed]
  %R6008-6009  FLOAT32  Encoder Counts           [Confirmed]
  %R6010-6011  FLOAT32  PID Setpoint             [Estimated]
  %R6012-6013  FLOAT32  PID Error                [Estimated]
  %R6014       INT16    PID Output (% × 100)     [Estimated]
  %R6015       INT16    Operating Mode            [Confirmed]
  %R6016-6017  FLOAT32  Target Torque (ft-lb)    [Estimated]
  %R6018-6019  FLOAT32  Accumulated Turns         [Estimated]
  %R6020       INT16    Fault Code (bitmask)      [Estimated]
  %R6021       INT16    Connection State           [Confirmed]
  %R6022-6023  FLOAT32  Peak Torque (ft-lb)       [Estimated]
  %R6024-6025  FLOAT32  Hookload (klbs)           [Estimated]
  %R6026-6027  FLOAT32  Shoulder Torque (ft-lb)   [Estimated]
  %R6028-6029  FLOAT32  Slope dT/dN (ft-lb/turn) [Estimated]
  %R6030       INT16    Connection Count           [Estimated]

GE CPE305 FLOAT32 encoding: word-swapped (low word at N, high word at N+1).
"""
import struct
import threading
import socketserver
import logging
from typing import Optional

from config import SimConfig

logger = logging.getLogger(__name__)


class RegisterBank:
    """Thread-safe register storage matching GE CPE305 %R space.

    GE FLOAT32 encoding (word-swapped):
      IEEE 754:  [High Word] [Low Word]
      GE CPE305: [Low Word]  [High Word]
    """

    def __init__(self, size: int = 10000):
        self._registers = [0] * size
        self._lock = threading.Lock()
        self.size = size

    def write_int16(self, address: int, value: int):
        with self._lock:
            if 0 <= address < self.size:
                self._registers[address] = value & 0xFFFF

    def write_int32(self, address: int, value: int):
        with self._lock:
            if 0 <= address < self.size - 1:
                self._registers[address] = (value >> 16) & 0xFFFF
                self._registers[address + 1] = value & 0xFFFF

    def write_float32(self, address: int, value: float):
        packed = struct.pack('>f', value)
        high_word = struct.unpack('>H', packed[0:2])[0]
        low_word = struct.unpack('>H', packed[2:4])[0]
        with self._lock:
            if 0 <= address < self.size - 1:
                self._registers[address] = low_word
                self._registers[address + 1] = high_word

    def read_registers(self, start: int, count: int) -> bytes:
        with self._lock:
            result = bytearray()
            for i in range(count):
                addr = start + i
                if 0 <= addr < self.size:
                    result.extend(struct.pack('>H', self._registers[addr]))
                else:
                    result.extend(b'\x00\x00')
            return bytes(result)


class ModbusRequestHandler(socketserver.BaseRequestHandler):

    def handle(self):
        client_addr = self.client_address
        logger.info(f"Modbus client connected: {client_addr}")
        try:
            while True:
                header = self._recv_exact(7)
                if not header:
                    break
                trans_id, protocol, length, unit_id = struct.unpack('>HHHB', header)
                if protocol != 0:
                    logger.warning(f"Invalid protocol ID: {protocol}")
                    continue
                pdu = self._recv_exact(length - 1)
                if not pdu:
                    break
                fc = pdu[0]
                if fc in (0x03, 0x04):
                    response = self._handle_read_registers(pdu)
                elif fc in (0x06,):
                    response = self._handle_write_single(pdu)
                elif fc in (0x10,):
                    response = self._handle_write_multiple(pdu)
                else:
                    response = bytes([fc | 0x80, 0x01])
                    logger.warning(f"Unsupported FC: 0x{fc:02X}")
                resp_length = len(response) + 1
                resp_header = struct.pack('>HHHB', trans_id, 0, resp_length, unit_id)
                self.request.sendall(resp_header + response)
        except (ConnectionResetError, BrokenPipeError, OSError):
            pass
        finally:
            logger.info(f"Modbus client disconnected: {client_addr}")

    def _handle_read_registers(self, pdu: bytes) -> bytes:
        fc = pdu[0]
        start_reg, count = struct.unpack('>HH', pdu[1:5])
        if count < 1 or count > 125:
            return bytes([fc | 0x80, 0x03])
        bank: RegisterBank = self.server.register_bank
        data = bank.read_registers(start_reg, count)
        byte_count = count * 2
        return bytes([fc, byte_count]) + data

    def _handle_write_single(self, pdu: bytes) -> bytes:
        """FC06: Write Single Register."""
        fc = pdu[0]
        reg_addr, value = struct.unpack('>HH', pdu[1:5])
        bank: RegisterBank = self.server.register_bank
        bank.write_int16(reg_addr, value)
        return pdu[:5]  # Echo back

    def _handle_write_multiple(self, pdu: bytes) -> bytes:
        """FC16: Write Multiple Registers."""
        fc = pdu[0]
        start_reg, count = struct.unpack('>HH', pdu[1:5])
        byte_count = pdu[5]
        bank: RegisterBank = self.server.register_bank
        for i in range(count):
            offset = 6 + i * 2
            if offset + 2 <= len(pdu):
                value = struct.unpack('>H', pdu[offset:offset+2])[0]
                bank.write_int16(start_reg + i, value)
        return bytes([fc]) + struct.pack('>HH', start_reg, count)

    def _recv_exact(self, n: int) -> Optional[bytes]:
        data = bytearray()
        while len(data) < n:
            try:
                chunk = self.request.recv(n - len(data))
                if not chunk:
                    return None
                data.extend(chunk)
            except (ConnectionResetError, OSError):
                return None
        return bytes(data)


class ThreadedModbusServer(socketserver.ThreadingMixIn, socketserver.TCPServer):
    allow_reuse_address = True
    daemon_threads = True


class ModbusTCPServer:
    """Public API for the Modbus TCP simulator server.

    Usage:
        server = ModbusTCPServer(cfg)
        server.start()
        server.update_from_reading(sensor_reading)
        server.stop()
    """

    def __init__(self, cfg: SimConfig, host: str = '0.0.0.0', port: int = 502):
        self.cfg = cfg
        self.host = host
        self.port = port
        self.register_bank = RegisterBank()
        self._server: Optional[ThreadedModbusServer] = None
        self._thread: Optional[threading.Thread] = None

    def start(self):
        self._server = ThreadedModbusServer((self.host, self.port), ModbusRequestHandler)
        self._server.register_bank = self.register_bank
        self._thread = threading.Thread(target=self._server.serve_forever, daemon=True)
        self._thread.start()
        logger.info(f"Modbus TCP server listening on {self.host}:{self.port}")

    def stop(self):
        if self._server:
            self._server.shutdown()
            self._server.server_close()
            logger.info("Modbus TCP server stopped")

    def update_from_reading(self, reading) -> None:
        """Write SensorReading into register bank matching Section 5.1.1."""
        cfg = self.cfg
        bank = self.register_bank

        # %R6000-6001: Torque (FLOAT32, GE word-swapped) [Confirmed]
        bank.write_float32(cfg.reg_torque, reading.torque_ftlbs)

        # %R6002-6003: RPM (FLOAT32) [Confirmed]
        bank.write_float32(cfg.reg_rpm, reading.rpm)

        # %R6004-6005: System Pressure (FLOAT32) [Confirmed]
        bank.write_float32(cfg.reg_pressure, reading.pressure_psi)

        # %R6006-6007: Oil Temperature (FLOAT32) [Confirmed]
        bank.write_float32(cfg.reg_temperature, reading.oil_temp_f)

        # %R6008-6009: Encoder Counts (FLOAT32) [Confirmed]
        bank.write_float32(cfg.reg_encoder_counts, float(reading.encoder_counts))

        # %R6010-6011: PID Setpoint (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_pid_setpoint, reading.pid_setpoint)

        # %R6012-6013: PID Error (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_pid_error, reading.pid_error)

        # %R6014: PID Output (INT16 — % × 100) [Estimated]
        bank.write_int16(cfg.reg_pid_output, int(reading.pid_output * 100))

        # %R6015: Operating Mode (INT16) [Confirmed]
        bank.write_int16(cfg.reg_mode, reading.operating_mode)

        # %R6016-6017: Target Torque (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_target_torque, reading.target_torque)

        # %R6018-6019: Accumulated Turns (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_turns_count, reading.turns)

        # %R6020: Fault Code (INT16 bitmask) [Estimated]
        bank.write_int16(cfg.reg_fault_code, reading.fault_code)

        # %R6021: Connection State (INT16) [Confirmed]
        bank.write_int16(cfg.reg_state, reading.connection_state)

        # %R6022-6023: Peak Torque (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_peak_torque, reading.peak_torque)

        # %R6024-6025: Hookload (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_hookload, reading.hookload_klbs)

        # %R6026-6027: Shoulder Torque (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_shoulder_torque, reading.shoulder_torque)

        # %R6028-6029: Slope dT/dN (FLOAT32) [Estimated]
        bank.write_float32(cfg.reg_slope, reading.slope_dT_dN)

        # %R6030: Connection Count (INT16) [Estimated]
        bank.write_int16(cfg.reg_connection_count, reading.connection_count)


### `runner.py`

In [ ]:
%%writefile runner.py
"""
Simulation Runner: The Orchestrator
=====================================
Ties together physics, sensors, Modbus, and scenario execution.

Enhanced with:
  - Real-time clock synchronization (drift-compensated)
  - PLC scan rate buffer (physics 100Hz -> Modbus at scan rate)
  - Dual-rate logging (physics rate vs CSV output rate)
  - Connection sequencing (field-realistic patterns)
  - FaultInjector delegation (clean fault injection)
  - Warm-up sequence (cold start oil circulation, hydraulic machines only)
  - Machine-type-aware initialization and output columns
  - Extended register map output (fault_code, peak_torque, hookload,
    shoulder_torque, slope, connection_count)
"""
import time
import csv
import logging
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any, Generator

from config import SimConfig, PipeSpec, MachineType
from physics_engine import PhysicsEngine, PhysicsState, ConnectionState, OperatingMode
from sensor_models import SensorModel, SensorReading
from modbus_server import ModbusTCPServer
from scenario import ConnectionScenario, ScenarioType, FaultInjector

logger = logging.getLogger(__name__)


# ═══════════════════════════════════════════════════════════════════
# Helper Classes
# ═══════════════════════════════════════════════════════════════════

class RealtimeClock:
    """Wall-clock synchronization for Modbus mode.

    Tracks cumulative drift and uses high-resolution timer.
    """

    def __init__(self):
        self._wall_start = time.perf_counter()
        self._sim_time = 0.0

    def sync(self, sim_time: float):
        """Sleep if simulation is ahead of wall clock."""
        self._sim_time = sim_time
        elapsed_wall = time.perf_counter() - self._wall_start
        ahead = sim_time - elapsed_wall
        if ahead > 0.001:
            time.sleep(ahead * 0.95)

    def reset(self):
        self._wall_start = time.perf_counter()
        self._sim_time = 0.0


class PLCScanBuffer:
    """Samples physics-rate readings at PLC scan rate for Modbus.

    Physics runs at 100Hz; real PLC reads sensors at 5-20Hz.
    This buffer ensures Modbus clients see scan-rate timing.
    """

    def __init__(self, scan_rate_hz: float = 10.0):
        self.scan_period = 1.0 / scan_rate_hz
        self._accumulator = 0.0
        self._last_reading = None

    def update(self, reading, dt: float):
        """Feed a physics-rate reading, returns reading if scan period elapsed."""
        self._last_reading = reading
        self._accumulator += dt
        if self._accumulator >= self.scan_period:
            self._accumulator -= self.scan_period
            return reading
        return None

    def reset(self):
        self._accumulator = 0.0
        self._last_reading = None


class DualRateLogger:
    """Decimation: physics at 100Hz internally, CSV at configurable rate."""

    def __init__(self, output_rate_hz: float = 100.0, physics_rate_hz: float = 100.0):
        self.decimation = max(1, int(physics_rate_hz / output_rate_hz))
        self._counter = 0

    def should_log(self) -> bool:
        self._counter += 1
        return self._counter % self.decimation == 0

    def reset(self):
        self._counter = 0


class ConnectionSequencer:
    """Field-realistic connection sequencing patterns."""

    def __init__(self, pattern: str = "single", rng: np.random.Generator = None):
        self.pattern = pattern
        self.rng = rng or np.random.default_rng()

    def next_delay(self) -> float:
        """Time between connections (seconds)."""
        if self.pattern == "single":
            return self.rng.uniform(15.0, 45.0)
        elif self.pattern == "doubles":
            return self.rng.uniform(5.0, 15.0)
        elif self.pattern == "casing":
            return self.rng.uniform(30.0, 90.0)
        return self.rng.uniform(10.0, 30.0)


# ═══════════════════════════════════════════════════════════════════
# Simulation Result
# ═══════════════════════════════════════════════════════════════════

@dataclass
class SimulationResult:
    """Complete output from one simulation run."""
    scenario: ConnectionScenario
    timestamps: list
    ground_truth: list
    sensor_data: list
    events: list
    metadata: dict


# ═══════════════════════════════════════════════════════════════════
# Ground Truth / Sensor Data Row Builders
# ═══════════════════════════════════════════════════════════════════

def _build_ground_truth_row(sim_time: float, state: PhysicsState) -> dict:
    """Build a single ground truth data row from PhysicsState."""
    return {
        'time': sim_time,
        'turns': state.turns,
        'rpm': state.rpm,
        'torque_ftlbs': state.torque_ftlbs,
        'pressure_psi': state.pressure_psi,
        'oil_temp_f': state.oil_temp_f,
        'encoder_counts': state.encoder_counts,
        'valve_command_pct': state.valve_command_pct,
        'connection_state': state.connection_state.value,
        'operating_mode': state.operating_mode.value,
        'pid_setpoint': state.pid_setpoint,
        'pid_error': state.pid_error,
        'pid_output': state.pid_output,
        'target_torque': state.target_torque_ftlbs,
        # Extended physics columns
        'valve_spool_pct': state.valve_spool_pct,
        'oil_viscosity_cst': state.oil_viscosity_cst,
        'motor_mech_efficiency': state.motor_mech_efficiency,
        'string_twist_deg': state.string_twist_deg,
        'connection_rpm': state.connection_rpm,
        'motor_torque_ftlbs': state.motor_torque_ftlbs,
        'thread_damage': state.thread_damage,
        'compound_friction_kf': state.compound_friction_kf,
        'leakage_flow_gpm': state.leakage_flow_gpm,
        'pump_ripple_psi': state.pump_ripple_psi,
        'manifold_temp_f': state.manifold_temp_f,
        'motor_case_temp_f': state.motor_case_temp_f,
        # New fields matching expanded register map (Section 5.1.1)
        'fault_code': state.fault_code,
        'peak_torque_ftlbs': state.peak_torque_ftlbs,
        'hookload_klbs': state.hookload_klbs,
        'shoulder_torque_ftlbs': state.shoulder_torque_ftlbs,
        'slope_dT_dN': state.slope_dT_dN,
        'connection_count': state.connection_count,
        # Machine-specific columns
        'machine_phase': state.machine_phase,
        'vfd_frequency_hz': state.vfd_frequency_hz,
        'vfd_current_pct': state.vfd_current_pct,
        'motor_speed_rpm': state.motor_speed_rpm,
    }


def _build_sensor_data_row(sim_time: float, reading: SensorReading) -> dict:
    """Build a single sensor data row from SensorReading."""
    return {
        'time': sim_time,
        'encoder_counts': reading.encoder_counts,
        'rpm': reading.rpm,
        'torque_ftlbs': reading.torque_ftlbs,
        'pressure_psi': reading.pressure_psi,
        'oil_temp_f': reading.oil_temp_f,
        'pid_setpoint': reading.pid_setpoint,
        'pid_error': reading.pid_error,
        'pid_output': reading.pid_output,
        'operating_mode': reading.operating_mode,
        'connection_state': reading.connection_state,
        'target_torque': reading.target_torque,
        'turns': reading.turns,
        # New fields matching expanded register map (Section 5.1.1)
        'fault_code': reading.fault_code,
        'peak_torque': reading.peak_torque,
        'hookload_klbs': reading.hookload_klbs,
        'shoulder_torque': reading.shoulder_torque,
        'slope_dT_dN': reading.slope_dT_dN,
        'connection_count': reading.connection_count,
    }


# ═══════════════════════════════════════════════════════════════════
# Simulation Runner
# ═══════════════════════════════════════════════════════════════════

class SimulationRunner:
    """Runs simulation scenarios and produces training data.

    Two modes:
      1. Batch mode: Run scenario, collect all data, return result
      2. Real-time mode: Run with Modbus server, synced to wall clock
    """

    def __init__(self, realtime: bool = False,
                 modbus_host: str = '0.0.0.0', modbus_port: int = 5020,
                 enable_modbus: bool = False,
                 csv_output_rate_hz: float = 100.0):
        self.realtime = realtime
        self.enable_modbus = enable_modbus or realtime
        self.modbus_host = modbus_host
        self.modbus_port = modbus_port
        self.csv_output_rate_hz = csv_output_rate_hz
        self._modbus_server: Optional[ModbusTCPServer] = None

    def run(self, scenario: ConnectionScenario) -> SimulationResult:
        """Execute a complete scenario simulation."""
        _run_t0 = time.perf_counter()
        cfg = scenario.config
        pipe = scenario.pipe
        rng = np.random.default_rng(scenario.seed)

        # Initialize subsystems
        engine = PhysicsEngine(cfg, pipe, rng)
        sensors = SensorModel(cfg, rng)

        # Configure connection
        target = scenario.target_torque or pipe.optimum_torque_ftlbs
        engine.configure_connection(
            rpm=scenario.rpm_setpoint,
            target_torque=target,
            breakout_rpm=scenario.breakout_rpm
        )

        # Start Modbus if enabled
        if self.enable_modbus:
            self._modbus_server = ModbusTCPServer(cfg, self.modbus_host, self.modbus_port)
            self._modbus_server.start()

        try:
            result = self._execute_scenario(engine, sensors, scenario, rng)
        finally:
            if self._modbus_server:
                self._modbus_server.stop()

        result.metadata['wall_time_s'] = round(time.perf_counter() - _run_t0, 3)
        return result

    def _execute_scenario(self, engine: PhysicsEngine, sensors: SensorModel,
                          scenario: ConnectionScenario,
                          rng: np.random.Generator) -> SimulationResult:
        """Core simulation loop with enhanced orchestration."""
        cfg = scenario.config
        dt = cfg.physics_dt

        # Data collection
        timestamps = []
        ground_truth = []
        sensor_data_list = []
        events = []

        # Helper objects
        fault_injector = FaultInjector(scenario, rng)
        scan_buffer = PLCScanBuffer(scan_rate_hz=1000.0 / cfg.pid_scan_rate_ms)
        rate_logger = DualRateLogger(
            output_rate_hz=self.csv_output_rate_hz if not self.realtime else 100.0
        )
        rt_clock = RealtimeClock() if self.realtime else None

        # State tracking
        prev_state = ConnectionState.IDLE
        connection_count = 0
        max_connections = scenario.num_connections
        sim_phase = 'idle'
        hold_timer = 0.0
        between_timer = 0.0

        # Optional warm-up for cold start (hydraulic machines only)
        if scenario.scenario_type == ScenarioType.COLD_START:
            if cfg.machine_type != MachineType.TOP_DRIVE:
                self._run_warmup(engine, duration_s=15.0)

        # Start first connection
        if scenario.scenario_type == ScenarioType.NORMAL_BREAKOUT:
            engine.start_breakout()
            sim_phase = 'breakout'
        else:
            engine.start_threading()
            sim_phase = 'threading'
            connection_count = 1

        total_steps = int(cfg.total_time / dt)

        for step_i in range(total_steps):
            sim_time = step_i * dt

            # ─── Fault Injection ──────────────────────────────
            fault_injector.apply(engine, dt)

            # ─── Physics Step ─────────────────────────────────
            state = engine.step(dt)

            # ─── Sensor Corruption ────────────────────────────
            reading = sensors.corrupt(state, dt)

            # ─── State Transition Events ──────────────────────
            if state.connection_state != prev_state:
                events.append({
                    'time': sim_time,
                    'event': 'state_change',
                    'from': prev_state.name,
                    'to': state.connection_state.name,
                    'turns': state.turns,
                    'torque': state.torque_ftlbs,
                    'connection': connection_count,
                })
                prev_state = state.connection_state

            # ─── Fault Events ─────────────────────────────────
            if state.is_over_torque and not any(e.get('event') == 'over_torque' for e in events):
                events.append({
                    'time': sim_time, 'event': 'over_torque',
                    'torque': state.torque_ftlbs, 'max': scenario.pipe.max_torque_ftlbs,
                })
            if state.is_cross_threaded and not any(e.get('event') == 'cross_thread' for e in events):
                events.append({
                    'time': sim_time, 'event': 'cross_thread',
                    'turns': state.turns, 'torque': state.torque_ftlbs,
                })
            if state.is_stick_slip and not any(e.get('event') == 'stick_slip' for e in events):
                events.append({
                    'time': sim_time, 'event': 'stick_slip',
                    'frequency_hz': state.stick_slip_frequency_hz,
                    'rpm': state.rpm,
                })
            if state.is_galled and not any(e.get('event') == 'galling' for e in events):
                events.append({
                    'time': sim_time, 'event': 'galling',
                    'turns': state.turns, 'torque': state.torque_ftlbs,
                })
            if state.is_stripped and not any(e.get('event') == 'stripped_thread' for e in events):
                events.append({
                    'time': sim_time, 'event': 'stripped_thread',
                    'turns': state.turns, 'torque': state.torque_ftlbs,
                })
            if state.is_misaligned and not any(e.get('event') == 'misaligned_stab' for e in events):
                events.append({
                    'time': sim_time, 'event': 'misaligned_stab',
                    'turns': state.turns, 'torque': state.torque_ftlbs,
                })
            if state.is_wrong_compound and not any(e.get('event') == 'wrong_compound' for e in events):
                events.append({
                    'time': sim_time, 'event': 'wrong_compound',
                    'turns': state.turns, 'torque': state.torque_ftlbs,
                    'kf': state.compound_friction_kf,
                })

            # ─── Shoulder Detection Event ─────────────────────
            if (state.shoulder_torque_ftlbs > 0 and
                    not any(e.get('event') == 'shoulder_detected' for e in events)):
                events.append({
                    'time': sim_time, 'event': 'shoulder_detected',
                    'turns': state.turns,
                    'shoulder_torque': state.shoulder_torque_ftlbs,
                })

            # ─── Modbus Update (via scan buffer) ──────────────
            if self._modbus_server:
                scan_reading = scan_buffer.update(reading, dt)
                if scan_reading is not None:
                    self._modbus_server.update_from_reading(scan_reading)

            # ─── Data Collection (at output rate) ─────────────
            if rate_logger.should_log():
                timestamps.append(sim_time)
                ground_truth.append(_build_ground_truth_row(sim_time, state))
                sensor_data_list.append(_build_sensor_data_row(sim_time, reading))

            # ─── Scenario Phase Management ────────────────────
            if state.connection_state == ConnectionState.COMPLETE:
                if sim_phase == 'threading':
                    if scenario.scenario_type == ScenarioType.FULL_CYCLE:
                        sim_phase = 'holding'
                        hold_timer = sim_time
                    elif scenario.scenario_type == ScenarioType.MULTI_CONNECTION:
                        if connection_count < max_connections:
                            sim_phase = 'between'
                            between_timer = sim_time
                        else:
                            break
                    else:
                        break
                elif sim_phase == 'breakout':
                    if scenario.scenario_type == ScenarioType.MULTI_CONNECTION:
                        if connection_count < max_connections:
                            sim_phase = 'between'
                            between_timer = sim_time
                        else:
                            break
                    else:
                        break

            # Hold -> breakout
            if sim_phase == 'holding' and sim_time - hold_timer > rng.uniform(2.0, 5.0):
                engine.reset()
                engine.configure_connection(
                    rpm=scenario.rpm_setpoint,
                    target_torque=scenario.target_torque or scenario.pipe.optimum_torque_ftlbs,
                    breakout_rpm=scenario.breakout_rpm
                )
                engine.state.turns = engine.torque_model.scaled_turns + engine.torque_model.scaled_delta
                engine.start_breakout()
                sim_phase = 'breakout'
                prev_state = ConnectionState.BREAKOUT

            # Between connections -> next makeup
            if sim_phase == 'between' and sim_time - between_timer > rng.uniform(3.0, 8.0):
                engine.reset()
                sensors.reset()
                fault_injector.reset()
                engine.configure_connection(
                    rpm=scenario.rpm_setpoint,
                    target_torque=scenario.target_torque or scenario.pipe.optimum_torque_ftlbs,
                    breakout_rpm=scenario.breakout_rpm
                )
                engine.start_threading()
                connection_count += 1
                sim_phase = 'threading'
                prev_state = ConnectionState.APPROACH

            # ─── Real-time Pacing ─────────────────────────────
            if rt_clock:
                rt_clock.sync(sim_time)

            # ─── Fault termination ────────────────────────────
            if state.is_over_torque or state.is_over_temp:
                events.append({
                    'time': sim_time, 'event': 'emergency_stop',
                    'reason': 'over_torque' if state.is_over_torque else 'over_temp',
                })
                # Wind down for 2 seconds
                for _ in range(int(2.0 / dt)):
                    engine.state.operating_mode = OperatingMode.IDLE
                    engine.state.valve_command_pct = 0
                    state = engine.step(dt)
                    reading = sensors.corrupt(state, dt)
                    if rate_logger.should_log():
                        timestamps.append(state.time)
                        ground_truth.append(_build_ground_truth_row(state.time, state))
                        sensor_data_list.append(_build_sensor_data_row(state.time, reading))
                break

        # ─── Metadata ────────────────────────────────────────
        metadata = {
            'scenario_type': scenario.scenario_type.value,
            'machine_type': scenario.machine_type.value,
            'pipe': scenario.pipe.name,
            'connection_type': scenario.pipe.connection_type,
            'seed': scenario.seed,
            'total_samples': len(timestamps),
            'duration_s': timestamps[-1] if timestamps else 0,
            'peak_torque_ftlbs': max((g['torque_ftlbs'] for g in ground_truth), default=0),
            'peak_pressure_psi': max((g['pressure_psi'] for g in ground_truth), default=0),
            'peak_rpm': max((g['rpm'] for g in ground_truth), default=0),
            'final_temp_f': ground_truth[-1]['oil_temp_f'] if ground_truth else 0,
            'shoulder_torque_ftlbs': max((g['shoulder_torque_ftlbs'] for g in ground_truth), default=0),
            'max_slope_dT_dN': max((g['slope_dT_dN'] for g in ground_truth), default=0),
            'max_hookload_klbs': max((g['hookload_klbs'] for g in ground_truth), default=0),
            'fault_code': ground_truth[-1]['fault_code'] if ground_truth else 0,
            'num_events': len(events),
            'connections_completed': connection_count,
            'has_fault': any(e['event'] in (
                'over_torque', 'cross_thread', 'galling', 'stick_slip',
                'stripped_thread', 'misaligned_stab', 'wrong_compound',
                'emergency_stop',
            ) for e in events),
            'label': scenario.label,
            'compound': scenario.compound_name,
            'string_length_ft': scenario.string_length_ft,
        }

        return SimulationResult(
            scenario=scenario,
            timestamps=timestamps,
            ground_truth=ground_truth,
            sensor_data=sensor_data_list,
            events=events,
            metadata=metadata,
        )

    def _run_warmup(self, engine: PhysicsEngine, duration_s: float = 15.0):
        """Simulate pump warm-up and oil circulation (cold start).

        Only applicable to hydraulic machines (not AC motor top drives).
        Runs hydraulics at low speed with no load to warm oil.
        """
        dt = engine.cfg.physics_dt
        steps = int(duration_s / dt)

        for _ in range(steps):
            # Low-speed circulation: 10% command, no load
            viscosity = engine.oil_model.viscosity_at_temp(engine.state.oil_temp_f)
            if hasattr(engine.drive, 'step'):
                try:
                    engine.drive.step(10.0, 0.0, viscosity, dt)
                except TypeError:
                    # AC motor drive has different signature; skip warmup
                    break
            Q_pump = 0.5  # Approximate pump loss HP
            engine.thermal.step(Q_pump, 0.0, 0.0, dt)
            engine.state.oil_temp_f = engine.thermal.get_oil_temp()

    def save_csv(self, result: SimulationResult, filepath: str,
                 data_type: str = 'sensor'):
        """Save simulation data to CSV."""
        data = result.sensor_data if data_type == 'sensor' else result.ground_truth
        if not data:
            logger.warning("No data to save")
            return

        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)

        fieldnames = list(data[0].keys())

        with open(filepath, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)

        logger.info(f"Saved {len(data)} rows to {filepath} ({data_type})")

    def save_events(self, result: SimulationResult, filepath: str):
        """Save event log to CSV."""
        if not result.events:
            return

        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)

        fieldnames = set()
        for e in result.events:
            fieldnames.update(e.keys())
        fieldnames = sorted(fieldnames)

        with open(filepath, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(result.events)


### `generate_dataset.py`

In [ ]:
%%writefile generate_dataset.py
#!/usr/bin/env python3
"""
TopDrive AI - Synthetic Dataset Generator
==========================================
Generates diverse training data using physics-based simulation.

Supports multiple machine types (top drive, iron roughneck, power tong,
bucking unit) and connection categories (API 8-Round LTC/STC, BTC,
premium shouldered, drill pipe).

Usage:
  # Generate 100 scenarios, save to output/
  python -m generate_dataset --count 100 --output ./data/synthetic

  # Generate with specific pipe type
  python -m generate_dataset --count 50 --pipe 7in_23lb_N80_LTC

  # Filter by machine type
  python -m generate_dataset --count 50 --machine-type top_drive

  # Filter by connection type
  python -m generate_dataset --count 50 --connection-type LTC

  # Real-time mode with Modbus server (for pipeline testing)
  python -m generate_dataset --realtime --modbus-port 5020

  # Single scenario for debugging
  python -m generate_dataset --single normal_casing_ltc --pipe 7in_23lb_N80_LTC

  # Phase 1 production dataset (Parquet, rebalanced 50/50 normal/fault)
  python generate_dataset.py --count 5000 --output ./data/synthetic_v2 \
    --output-format parquet --class-balance rebalanced --seed 42

Output structure:
  data/synthetic/
  ├── sensor/               # Noisy sensor data (for AI training)
  │   ├── scenario_0000_normal_casing_ltc.csv
  │   ├── scenario_0001_cross_thread.csv
  │   └── ...
  ├── truth/                # Ground truth (for validation)
  │   ├── scenario_0000_normal_casing_ltc.csv
  │   └── ...
  ├── events/               # State transitions and faults
  │   ├── scenario_0000_normal_casing_ltc.csv
  │   └── ...
  ├── manifest.csv          # Index of all scenarios with metadata
  └── stats.txt             # Dataset statistics
"""
import argparse
import json
import csv
import sys
import time
import logging
from pathlib import Path
from typing import Optional, Dict, List

import numpy as np

from config import SimConfig, MachineType, ALL_CONNECTIONS, PIPE_CATALOG, PREMIUM_CATALOG, DRILL_PIPE_CATALOG
from scenario import ScenarioGenerator, ScenarioType
from runner import SimulationRunner

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

import multiprocessing
import os as _os
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def _parse_machine_types(machine_type_str: Optional[str]) -> Optional[list]:
    """Parse machine type argument into list of MachineType enums."""
    if not machine_type_str:
        return None
    try:
        return [MachineType(machine_type_str)]
    except ValueError:
        valid = [m.value for m in MachineType]
        print(f"Unknown machine type: {machine_type_str}")
        print(f"Available: {valid}")
        sys.exit(1)


# Rebalanced class distribution for ML training (Section 2.1)
# 50/50 normal/fault split with inverse frequency weighting for rare faults.
REBALANCED_DISTRIBUTION: Dict[str, float] = {
    'normal_casing_ltc': 0.15,
    'normal_casing_btc': 0.10,
    'normal_casing_premium': 0.08,
    'normal_drill_pipe': 0.10,
    'normal_tubing': 0.04,
    'full_cycle': 0.03,
    'cross_thread': 0.065,
    'galling': 0.065,
    'stripped_thread': 0.055,
    'over_torque': 0.055,
    'under_torque': 0.055,
    'wrong_compound': 0.050,
    'misaligned_stabbing': 0.055,
    'stall': 0.050,
    'stick_slip': 0.015,
    'multi_connection': 0.010,
    'washout': 0.005,
    'connection_jump': 0.005,
}

# Fault class mapping (scenario_type -> numeric class for ML)
# 9 classes: 0=normal, 1=cross_thread, 2=galling, 3=stripped_thread,
#            4=over_torque, 5=under_torque, 6=wrong_compound,
#            7=misaligned_stab, 8=stall
# EVERY ScenarioType must appear here to avoid mislabeling faults as normal.
FAULT_CLASS_MAP: Dict[str, int] = {
    # Normal variants (class 0)
    'normal_casing_ltc': 0,
    'normal_casing_btc': 0,
    'normal_casing_premium': 0,
    'normal_drill_pipe': 0,
    'normal_tubing': 0,
    'normal_breakout': 0,
    'full_cycle': 0,
    'multi_connection': 0,
    'cold_start': 0,
    'hot_environment': 0,
    # Fault classes (1-8)
    'cross_thread': 1,
    'connection_jump': 1,
    'stick_slip': 1,
    'staged_fault': 1,
    'galling': 2,
    'stripped_thread': 3,
    'over_torque': 4,
    'washout': 4,
    'under_torque': 5,
    'wrong_compound': 6,
    'misaligned_stabbing': 7,
    'stall': 8,
}


def _save_parquet(data: List[dict], filepath: Path):
    """Save list of dicts as Parquet file using pyarrow."""
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError:
        # Fallback to CSV if pyarrow not installed
        logger.warning("pyarrow not installed, falling back to CSV")
        filepath = filepath.with_suffix('.csv')
        if data:
            with open(filepath, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=data[0].keys())
                writer.writeheader()
                writer.writerows(data)
        return

    import pandas as pd
    df = pd.DataFrame(data)
    df.to_parquet(filepath, index=False, engine='pyarrow')


def _filter_pipes_by_connection(connection_type: Optional[str]) -> Optional[list]:
    """Filter pipe catalog by connection type (LTC, BTC, STC, PREMIUM, DRILL_PIPE)."""
    if not connection_type:
        return None
    ct = connection_type.upper()
    matching = [
        name for name, spec in ALL_CONNECTIONS.items()
        if spec.connection_type.upper() == ct
    ]
    if not matching:
        all_types = sorted(set(s.connection_type for s in ALL_CONNECTIONS.values()))
        print(f"No pipes found for connection type: {connection_type}")
        print(f"Available types: {all_types}")
        sys.exit(1)
    return matching



def _run_single_scenario(args_tuple):
    """Worker function for multiprocessing. Runs one scenario, saves output, returns manifest entry."""
    idx, scenario, output_dir, use_parquet, output_rate, file_ext = args_tuple
    import time as _time
    from pathlib import Path
    from runner import SimulationRunner

    base_name = f"scenario_{idx:04d}"
    sensor_dir = Path(output_dir) / 'sensor'
    truth_dir = Path(output_dir) / 'truth'
    events_dir = Path(output_dir) / 'events'

    t0 = _time.perf_counter()
    try:
        runner = SimulationRunner(csv_output_rate_hz=output_rate)
        result = runner.run(scenario)
        elapsed = _time.perf_counter() - t0

        # Save files
        if use_parquet:
            _save_parquet(result.sensor_data, sensor_dir / f"{base_name}{file_ext}")
            _save_parquet(result.ground_truth, truth_dir / f"{base_name}{file_ext}")
            if result.events:
                _save_parquet(result.events, events_dir / f"{base_name}{file_ext}")
        else:
            runner.save_csv(result, sensor_dir / f"{base_name}{file_ext}", data_type='sensor')
            runner.save_csv(result, truth_dir / f"{base_name}{file_ext}", data_type='truth')
            if result.events:
                runner.save_events(result, events_dir / f"{base_name}{file_ext}")

        # Manifest entry
        fault_class = FAULT_CLASS_MAP.get(scenario.scenario_type.value, 0)
        entry = {
            'scenario_id': idx,
            'filename': f"{base_name}{file_ext}",
            'scenario_type': scenario.scenario_type.value,
            'fault_class': fault_class,
            'machine_type': scenario.machine_type.value,
            'pipe_name': scenario.pipe.name,
            'connection_type': scenario.pipe.connection_type,
            'pipe_od_in': scenario.pipe.od_inches,
            'target_torque_ftlbs': scenario.target_torque or scenario.pipe.optimum_torque_ftlbs,
            'expected_turns': scenario.pipe.turns_to_shoulder,
            'seed': scenario.seed,
            'num_samples': result.metadata['total_samples'],
            'duration_s': round(result.metadata['duration_s'], 2),
            'peak_torque_ftlbs': round(result.metadata['peak_torque_ftlbs'], 1),
            'peak_rpm': round(result.metadata['peak_rpm'], 1),
            'shoulder_torque_ftlbs': round(result.metadata.get('shoulder_torque_ftlbs', 0), 1),
            'has_fault': result.metadata['has_fault'],
            'fault_code': result.metadata.get('fault_code', 0),
            'wall_time_s': result.metadata.get('wall_time_s', round(elapsed, 3)),
            'label': scenario.label,
        }

        diag = {
            'idx': idx,
            'elapsed_s': round(elapsed, 2),
            'num_samples': result.metadata['total_samples'],
            'peak_torque': round(result.metadata['peak_torque_ftlbs'], 1),
            'peak_rpm': round(result.metadata['peak_rpm'], 1),
            'scenario_type': scenario.scenario_type.value,
            'error': None,
        }

        return (entry, diag, None)

    except Exception as e:
        elapsed = _time.perf_counter() - t0
        fault_class = FAULT_CLASS_MAP.get(scenario.scenario_type.value, 0)
        entry = {
            'scenario_id': idx,
            'filename': f"{base_name}{file_ext}",
            'scenario_type': scenario.scenario_type.value,
            'fault_class': fault_class,
            'machine_type': scenario.machine_type.value,
            'pipe_name': scenario.pipe.name,
            'connection_type': scenario.pipe.connection_type,
            'pipe_od_in': scenario.pipe.od_inches,
            'target_torque_ftlbs': 0,
            'expected_turns': 0,
            'seed': scenario.seed,
            'num_samples': 0,
            'duration_s': 0,
            'peak_torque_ftlbs': 0,
            'peak_rpm': 0,
            'shoulder_torque_ftlbs': 0,
            'has_fault': False,
            'fault_code': 0,
            'wall_time_s': round(elapsed, 3),
            'label': f"FAILED: {e}",
        }
        diag = {
            'idx': idx,
            'elapsed_s': round(elapsed, 2),
            'num_samples': 0,
            'peak_torque': 0,
            'peak_rpm': 0,
            'scenario_type': scenario.scenario_type.value,
            'error': str(e),
        }
        return (entry, diag, str(e))


def generate_batch_parallel(args, num_workers=0):
    """Generate training data using multiprocessing for parallel scenario execution.

    Args:
        args: Namespace with count, output, seed, output_format, class_balance,
              output_rate, pipe, machine_type, connection_type
        num_workers: Number of parallel workers. 0 = auto-detect (cpu_count - 1).
                     1 = sequential (no multiprocessing).
    """
    output_dir = Path(args.output)
    sensor_dir = output_dir / 'sensor'
    truth_dir = output_dir / 'truth'
    events_dir = output_dir / 'events'
    use_parquet = getattr(args, 'output_format', 'csv') == 'parquet'
    file_ext = '.parquet' if use_parquet else '.csv'
    output_rate = getattr(args, 'output_rate', 100.0)

    for d in [sensor_dir, truth_dir, events_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Initialize scenario generator
    gen = ScenarioGenerator(seed=args.seed)
    class_balance = getattr(args, 'class_balance', 'default')
    if class_balance == 'rebalanced':
        gen.weights = {}
        for stype_str, weight in REBALANCED_DISTRIBUTION.items():
            try:
                gen.weights[ScenarioType(stype_str)] = weight
            except ValueError:
                pass
        logger.info("Using REBALANCED class distribution (50/50 normal/fault)")

    # Build filters
    pipe_filter = None
    if getattr(args, 'pipe', None):
        pipe_filter = [args.pipe]
    elif getattr(args, 'connection_type', None):
        pipe_filter = _filter_pipes_by_connection(args.connection_type)
    machine_filter = _parse_machine_types(getattr(args, 'machine_type', None))

    # Generate all scenario configs (fast — no simulation yet)
    scenarios = gen.generate_batch(
        args.count,
        pipe_names=pipe_filter,
        machine_types=machine_filter,
    )
    logger.info(f"Generated {len(scenarios)} scenario configs")
    logger.info(gen.get_distribution_summary())

    # Determine worker count
    if num_workers <= 0:
        num_workers = max(1, (_os.cpu_count() or 2) - 1)
    num_workers = min(num_workers, len(scenarios))

    # Build worker args
    worker_args = [
        (i, sc, str(output_dir), use_parquet, output_rate, file_ext)
        for i, sc in enumerate(scenarios)
    ]

    start_time = time.monotonic()
    manifest = []
    diagnostics = []
    errors = []

    if num_workers <= 1:
        # Sequential fallback
        logger.info(f"Generating {len(scenarios)} scenarios sequentially...")
        for wa in tqdm(worker_args, desc="Generating", unit="scenario"):
            entry, diag, err = _run_single_scenario(wa)
            manifest.append(entry)
            diagnostics.append(diag)
            if err:
                errors.append(err)
                logger.error(f"  Scenario {diag['idx']} FAILED: {err}")
    else:
        logger.info(f"Generating {len(scenarios)} scenarios with {num_workers} workers...")
        with multiprocessing.Pool(num_workers) as pool:
            results_iter = pool.imap_unordered(_run_single_scenario, worker_args)
            pbar = tqdm(results_iter, total=len(worker_args),
                        desc="Generating", unit="scenario")
            for entry, diag, err in pbar:
                manifest.append(entry)
                diagnostics.append(diag)
                if err:
                    errors.append(err)
                    logger.error(f"  Scenario {diag['idx']} FAILED: {err}")
                # Update progress bar postfix with latest scenario info
                pbar.set_postfix({
                    'last': diag.get('scenario_type', '?')[:15],
                    'time': f"{diag.get('elapsed_s', 0):.0f}s",
                    'fail': len(errors),
                }, refresh=False)

    elapsed = time.monotonic() - start_time

    # Sort manifest by scenario_id (imap_unordered returns out of order)
    manifest.sort(key=lambda m: m['scenario_id'])
    diagnostics.sort(key=lambda d: d['idx'])

    # Compute derived stats
    total_samples = sum(int(m.get('num_samples', 0)) for m in manifest)
    fault_count = sum(1 for m in manifest if m.get('has_fault'))

    # Save manifest
    if manifest:
        if use_parquet:
            _save_parquet(manifest, output_dir / 'manifest.parquet')
        # Always save CSV manifest (human-readable)
        manifest_path = output_dir / 'manifest.csv'
        with open(manifest_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=manifest[0].keys())
            writer.writeheader()
            writer.writerows(manifest)

    # Save enhanced stats
    stats = _compute_stats(manifest, elapsed, total_samples, fault_count,
                           scenarios, num_workers=num_workers,
                           diagnostics=diagnostics, errors=errors)
    stats_path = output_dir / 'stats.txt'
    with open(stats_path, 'w') as f:
        f.write(stats)

    # Save generation config
    gen_config = {
        'count': args.count,
        'seed': args.seed,
        'output_format': 'parquet' if use_parquet else 'csv',
        'class_balance': class_balance,
        'output_rate_hz': output_rate,
        'num_workers': num_workers,
        'wall_time_s': round(elapsed, 1),
        'pipe_filter': getattr(args, 'pipe', None),
        'machine_type': getattr(args, 'machine_type', None),
        'connection_type': getattr(args, 'connection_type', None),
    }
    with open(output_dir / 'config.json', 'w') as f:
        json.dump(gen_config, f, indent=2)

    print(f"\n{'='*60}")
    print(stats)
    if errors:
        print(f"\nWARNING: {len(errors)} scenarios failed:")
        for e in errors[:10]:
            print(f"  - {e[:120]}")
    print(f"\nOutput: {output_dir.absolute()}")



def generate_batch(args):
    """Generate a batch of synthetic training data."""
    output_dir = Path(args.output)
    sensor_dir = output_dir / 'sensor'
    truth_dir = output_dir / 'truth'
    events_dir = output_dir / 'events'
    use_parquet = getattr(args, 'output_format', 'csv') == 'parquet'
    file_ext = '.parquet' if use_parquet else '.csv'

    for d in [sensor_dir, truth_dir, events_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Initialize
    gen = ScenarioGenerator(seed=args.seed)

    # Apply rebalanced distribution if requested
    class_balance = getattr(args, 'class_balance', 'default')
    if class_balance == 'rebalanced':
        gen.weights = {}
        for stype_str, weight in REBALANCED_DISTRIBUTION.items():
            try:
                gen.weights[ScenarioType(stype_str)] = weight
            except ValueError:
                pass
        logger.info("Using REBALANCED class distribution (50/50 normal/fault)")

    runner = SimulationRunner(csv_output_rate_hz=args.output_rate)

    # Build pipe filter
    pipe_filter = None
    if args.pipe:
        pipe_filter = [args.pipe]
    elif args.connection_type:
        pipe_filter = _filter_pipes_by_connection(args.connection_type)

    # Build machine filter
    machine_filter = _parse_machine_types(args.machine_type)

    scenarios = gen.generate_batch(
        args.count,
        pipe_names=pipe_filter,
        machine_types=machine_filter,
    )

    logger.info(f"Generating {len(scenarios)} scenarios...")
    logger.info(gen.get_distribution_summary())

    # Manifest tracking
    manifest = []
    total_samples = 0
    fault_count = 0
    start_time = time.monotonic()

    for i, scenario in enumerate(scenarios):
        base_name = f"scenario_{i:04d}"

        logger.info(f"[{i+1}/{len(scenarios)}] {scenario.label}")

        try:
            result = runner.run(scenario)

            # Save data (Parquet or CSV)
            if use_parquet:
                _save_parquet(result.sensor_data, sensor_dir / f"{base_name}{file_ext}")
                _save_parquet(result.ground_truth, truth_dir / f"{base_name}{file_ext}")
                if result.events:
                    _save_parquet(result.events, events_dir / f"{base_name}{file_ext}")
            else:
                runner.save_csv(result, sensor_dir / f"{base_name}{file_ext}", data_type='sensor')
                runner.save_csv(result, truth_dir / f"{base_name}{file_ext}", data_type='truth')
                if result.events:
                    runner.save_events(result, events_dir / f"{base_name}{file_ext}")

            # Manifest entry (Section 2.6 schema)
            fault_class = FAULT_CLASS_MAP.get(scenario.scenario_type.value, 0)
            manifest.append({
                'scenario_id': i,
                'filename': f"{base_name}{file_ext}",
                'scenario_type': scenario.scenario_type.value,
                'fault_class': fault_class,
                'machine_type': scenario.machine_type.value,
                'pipe_name': scenario.pipe.name,
                'connection_type': scenario.pipe.connection_type,
                'pipe_od_in': scenario.pipe.od_inches,
                'target_torque_ftlbs': scenario.target_torque or scenario.pipe.optimum_torque_ftlbs,
                'expected_turns': scenario.pipe.turns_to_shoulder,
                'seed': scenario.seed,
                'num_samples': result.metadata['total_samples'],
                'duration_s': round(result.metadata['duration_s'], 2),
                'peak_torque_ftlbs': round(result.metadata['peak_torque_ftlbs'], 1),
                'peak_rpm': round(result.metadata['peak_rpm'], 1),
                'shoulder_torque_ftlbs': round(result.metadata.get('shoulder_torque_ftlbs', 0), 1),
                'has_fault': result.metadata['has_fault'],
                'fault_code': result.metadata.get('fault_code', 0),
                'label': scenario.label,
            })

            total_samples += result.metadata['total_samples']
            if result.metadata['has_fault']:
                fault_count += 1

        except Exception as e:
            logger.error(f"  FAILED: {e}", exc_info=True)
            manifest.append({
                'scenario_id': i, 'filename': f"{base_name}{file_ext}",
                'scenario_type': scenario.scenario_type.value,
                'fault_class': FAULT_CLASS_MAP.get(scenario.scenario_type.value, 0),
                'machine_type': scenario.machine_type.value,
                'pipe_name': scenario.pipe.name,
                'connection_type': scenario.pipe.connection_type,
                'pipe_od_in': scenario.pipe.od_inches,
                'target_torque_ftlbs': 0, 'expected_turns': 0,
                'seed': scenario.seed,
                'num_samples': 0, 'duration_s': 0,
                'peak_torque_ftlbs': 0, 'peak_rpm': 0,
                'shoulder_torque_ftlbs': 0,
                'has_fault': False, 'fault_code': 0,
                'label': f"FAILED: {e}",
            })

    elapsed = time.monotonic() - start_time

    # Save manifest
    if manifest:
        if use_parquet:
            _save_parquet(manifest, output_dir / 'manifest.parquet')
        # Always save CSV manifest too (lightweight, human-readable)
        manifest_path = output_dir / 'manifest.csv'
        with open(manifest_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=manifest[0].keys())
            writer.writeheader()
            writer.writerows(manifest)

    # Save stats
    stats = _compute_stats(manifest, elapsed, total_samples, fault_count, scenarios)
    stats_path = output_dir / 'stats.txt'
    with open(stats_path, 'w') as f:
        f.write(stats)

    # Save generation config for reproducibility
    gen_config = {
        'count': args.count,
        'seed': args.seed,
        'output_format': 'parquet' if use_parquet else 'csv',
        'class_balance': class_balance,
        'output_rate_hz': args.output_rate,
        'pipe_filter': args.pipe,
        'machine_type': args.machine_type,
        'connection_type': args.connection_type,
    }
    with open(output_dir / 'config.json', 'w') as f:
        json.dump(gen_config, f, indent=2)

    print(f"\n{'='*60}")
    print(stats)
    print(f"Output: {output_dir.absolute()}")


def generate_single(args):
    """Generate a single scenario (for debugging)."""
    try:
        stype = ScenarioType(args.single)
    except ValueError:
        print(f"Unknown scenario type: {args.single}")
        print(f"Available: {[s.value for s in ScenarioType]}")
        sys.exit(1)

    gen = ScenarioGenerator(seed=args.seed)

    machine_type = None
    if args.machine_type:
        machine_type = _parse_machine_types(args.machine_type)
        machine_type = machine_type[0] if machine_type else None

    scenario = gen.generate_one(
        stype,
        pipe_name=args.pipe or "7in_23lb_N80_LTC",
        machine_type=machine_type,
    )

    runner = SimulationRunner(
        realtime=args.realtime,
        enable_modbus=args.realtime,
        modbus_port=args.modbus_port,
        csv_output_rate_hz=args.output_rate,
    )

    logger.info(f"Running: {scenario.label}")
    logger.info(f"Machine: {scenario.machine_type.value}")
    if args.realtime:
        logger.info(f"Modbus server on port {args.modbus_port}")

    result = runner.run(scenario)

    # Print summary
    print(f"\n{'='*60}")
    print(f"Scenario:       {scenario.label}")
    print(f"Machine:        {scenario.machine_type.value}")
    print(f"Connection:     {scenario.pipe.connection_type}")
    print(f"Pipe:           {scenario.pipe.name}")
    print(f"Samples:        {result.metadata['total_samples']}")
    print(f"Duration:       {result.metadata['duration_s']:.2f}s")
    print(f"Peak Torque:    {result.metadata['peak_torque_ftlbs']:.0f} ft-lbs")
    print(f"Peak Pressure:  {result.metadata['peak_pressure_psi']:.0f} PSI")
    print(f"Peak RPM:       {result.metadata['peak_rpm']:.1f}")
    print(f"Shoulder Torque:{result.metadata.get('shoulder_torque_ftlbs', 0):.0f} ft-lbs")
    print(f"Fault Code:     0x{result.metadata.get('fault_code', 0):04X}")
    print(f"Events:         {result.metadata['num_events']}")

    if result.events:
        print(f"\nEvent Log:")
        for e in result.events:
            print(f"  t={e['time']:.3f}s  {e['event']}: {e.get('from','')} -> {e.get('to','')}")

    # Save if output specified
    if args.output:
        output_dir = Path(args.output)
        output_dir.mkdir(parents=True, exist_ok=True)
        runner.save_csv(result, output_dir / 'sensor_data.csv', data_type='sensor')
        runner.save_csv(result, output_dir / 'ground_truth.csv', data_type='truth')
        runner.save_events(result, output_dir / 'events.csv')
        print(f"\nSaved to: {output_dir.absolute()}")


def realtime_mode(args):
    """Run continuous scenarios with live Modbus server."""
    gen = ScenarioGenerator(seed=args.seed)

    machine_filter = _parse_machine_types(args.machine_type)

    runner = SimulationRunner(
        realtime=True,
        enable_modbus=True,
        modbus_port=args.modbus_port,
        csv_output_rate_hz=args.output_rate,
    )

    logger.info(f"Real-time mode - Modbus server on port {args.modbus_port}")
    logger.info(f"Connect your data pipeline to localhost:{args.modbus_port}")
    logger.info("Press Ctrl+C to stop")

    try:
        cycle = 0
        while True:
            scenario = gen.generate_batch(1, machine_types=machine_filter)[0]
            cycle += 1
            logger.info(f"[Cycle {cycle}] {scenario.label} ({scenario.machine_type.value})")

            result = runner.run(scenario)
            logger.info(f"  Complete: {result.metadata['total_samples']} samples, "
                       f"peak {result.metadata['peak_torque_ftlbs']:.0f} ft-lbs, "
                       f"fault=0x{result.metadata.get('fault_code', 0):04X}")

            if args.output:
                output_dir = Path(args.output)
                runner.save_csv(result, output_dir / f'cycle_{cycle:04d}_sensor.csv')

            # Pause between connections (simulates field timing)
            time.sleep(2.0)

    except KeyboardInterrupt:
        logger.info(f"\nStopped after {cycle} cycles")


def _compute_stats(manifest, elapsed, total_samples, fault_count, scenarios,
                   num_workers=1, diagnostics=None, errors=None):
    """Compute and format dataset statistics with performance diagnostics."""
    from collections import Counter

    type_counts = Counter(m['scenario_type'] for m in manifest)
    pipe_counts = Counter(m['pipe_name'] for m in manifest)
    machine_counts = Counter(m.get('machine_type', 'unknown') for m in manifest)
    conn_type_counts = Counter(m.get('connection_type', 'unknown') for m in manifest)
    class_counts = Counter(int(m.get('fault_class', 0)) for m in manifest)
    successful = sum(1 for m in manifest if m.get('num_samples', 0) and int(m.get('num_samples', 0)) > 0)

    CLASS_NAMES = {
        0: 'normal', 1: 'cross_thread', 2: 'galling', 3: 'stripped_thread',
        4: 'over_torque', 5: 'under_torque', 6: 'wrong_compound',
        7: 'misaligned_stab', 8: 'stall',
    }

    lines = [
        f"TopDrive AI Synthetic Dataset Statistics",
        f"{'='*60}",
        f"Total scenarios:    {len(manifest)}",
        f"Successful:         {successful} ({successful/max(len(manifest),1):.0%})",
        f"Failed:             {len(manifest) - successful}",
        f"Total samples:      {total_samples:,}",
        f"Total sim duration: {sum(float(m.get('duration_s', 0)) for m in manifest):.0f}s simulated",
        f"Generation time:    {elapsed:.1f}s wall clock",
        f"Workers:            {num_workers}",
        f"Throughput:         {len(manifest)/max(elapsed,1):.1f} scenarios/sec",
        f"Throughput:         {total_samples/max(elapsed,1):.0f} samples/sec",
        f"Fault scenarios:    {fault_count} ({fault_count/max(len(manifest),1):.0%})",
    ]

    # Per-scenario timing
    if diagnostics:
        times = [d.get('elapsed_s', 0) for d in diagnostics if d.get('elapsed_s')]
        if times:
            lines.append(f"")
            lines.append(f"Per-Scenario Timing:")
            lines.append(f"  Average:  {sum(times)/len(times):.1f}s")
            lines.append(f"  Median:   {sorted(times)[len(times)//2]:.1f}s")
            lines.append(f"  Min:      {min(times):.1f}s")
            lines.append(f"  Max:      {max(times):.1f}s")
            lines.append(f"  Total:    {sum(times):.0f}s CPU time")
            if num_workers > 1:
                speedup = sum(times) / max(elapsed, 1)
                lines.append(f"  Speedup:  {speedup:.1f}x (vs sequential)")

    # Fault class distribution
    lines.append(f"")
    lines.append(f"Fault Class Distribution (9 ML classes):")
    for cls_id in sorted(class_counts.keys()):
        count = class_counts[cls_id]
        name = CLASS_NAMES.get(cls_id, f'class_{cls_id}')
        pct = count / max(len(manifest), 1)
        bar = '#' * int(pct * 40)
        lines.append(f"  {cls_id} {name:20s}  {count:5d}  ({pct:5.1%}) {bar}")

    lines.append(f"")
    lines.append(f"Scenario Type Distribution:")
    for stype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        lines.append(f"  {stype:30s}  {count:4d}  ({count/len(manifest):.1%})")

    lines.append(f"\nMachine Type Distribution:")
    for machine, count in sorted(machine_counts.items(), key=lambda x: -x[1]):
        lines.append(f"  {machine:30s}  {count:4d}  ({count/len(manifest):.1%})")

    lines.append(f"\nConnection Type Distribution:")
    for ct, count in sorted(conn_type_counts.items(), key=lambda x: -x[1]):
        lines.append(f"  {ct:30s}  {count:4d}  ({count/len(manifest):.1%})")

    lines.append(f"\nPipe Distribution (top 20):")
    for pipe, count in sorted(pipe_counts.items(), key=lambda x: -x[1])[:20]:
        lines.append(f"  {pipe:40s}  {count:4d}")

    if errors:
        lines.append(f"\nErrors ({len(errors)}):")
        for e in (errors or [])[:10]:
            lines.append(f"  - {str(e)[:120]}")

    return "\n".join(lines)


def main():
    parser = argparse.ArgumentParser(
        description='TopDrive AI Synthetic Dataset Generator',
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  %(prog)s --count 100 --output ./data/synthetic
  %(prog)s --single normal_casing_ltc --pipe 7in_23lb_N80_LTC
  %(prog)s --count 50 --machine-type top_drive
  %(prog)s --count 50 --connection-type LTC
  %(prog)s --realtime --modbus-port 5020
        """
    )

    parser.add_argument('--count', type=int, default=10,
                        help='Number of scenarios to generate (default: 10)')
    parser.add_argument('--output', '-o', type=str, default='./data/synthetic',
                        help='Output directory')
    parser.add_argument('--pipe', type=str, default=None,
                        help='Restrict to specific pipe type (by name)')
    parser.add_argument('--machine-type', type=str, default=None,
                        choices=[m.value for m in MachineType],
                        help='Filter by machine type')
    parser.add_argument('--connection-type', type=str, default=None,
                        help='Filter by connection type (LTC, BTC, STC, PREMIUM, DRILL_PIPE)')
    parser.add_argument('--output-rate', type=float, default=100.0,
                        help='CSV output rate in Hz (default: 100)')
    parser.add_argument('--seed', type=int, default=42,
                        help='Random seed for reproducibility')
    parser.add_argument('--single', type=str, default=None,
                        help='Run single scenario type')
    parser.add_argument('--realtime', action='store_true',
                        help='Run in real-time with Modbus server')
    parser.add_argument('--modbus-port', type=int, default=5020,
                        help='Modbus TCP port (default: 5020)')
    parser.add_argument('--output-format', type=str, default='csv',
                        choices=['csv', 'parquet'],
                        help='Output file format (default: csv)')
    parser.add_argument('--class-balance', type=str, default='default',
                        choices=['default', 'rebalanced'],
                        help='Class distribution: default (field 65/35) or rebalanced (50/50 normal/fault)')

    args = parser.parse_args()

    if args.realtime:
        realtime_mode(args)
    elif args.single:
        generate_single(args)
    else:
        generate_batch(args)


if __name__ == '__main__':
    main()


In [ ]:
import os, importlib

modules = ['config.py','physics_engine.py','sensor_models.py',
           'scenario.py','modbus_server.py','runner.py','generate_dataset.py']

print("Module file check:")
for m in modules:
    size = os.path.getsize(m)
    print(f"  {m:<30s}  {size:>8,} bytes  OK")

print("\nImport check:")
for m in modules:
    mod_name = m.replace('.py', '')
    try:
        importlib.import_module(mod_name)
        print(f"  import {mod_name:<25s}  OK")
    except Exception as e:
        print(f"  import {mod_name:<25s}  FAILED: {e}")

print("\nDependency check:")
for pkg in ['numpy', 'pandas', 'pyarrow', 'tqdm', 'multiprocessing']:
    try:
        importlib.import_module(pkg)
        print(f"  {pkg:<25s}  OK")
    except ImportError:
        print(f"  {pkg:<25s}  MISSING")

print("\nSmoke test (1 scenario):")
try:
    from scenario import ScenarioGenerator, ScenarioType
    from runner import SimulationRunner
    import time

    gen = ScenarioGenerator(seed=999)
    sc = gen.generate_one(ScenarioType.NORMAL_CASING_LTC, pipe_name="7in_23lb_N80_LTC")
    runner = SimulationRunner(csv_output_rate_hz=100.0)
    t0 = time.perf_counter()
    result = runner.run(sc)
    t1 = time.perf_counter()
    ns = result.metadata['total_samples']
    pt = result.metadata['peak_torque_ftlbs']
    wt = result.metadata.get('wall_time_s', t1 - t0)
    print(f"  Scenario: {sc.scenario_type.value}")
    print(f"  Samples:  {ns}")
    print(f"  Peak torque: {pt:.0f} ft-lbs")
    print(f"  Wall time:   {wt:.2f}s")
    print(f"  Smoke test PASSED")
except Exception as e:
    print(f"  Smoke test FAILED: {e}")
    import traceback
    traceback.print_exc()

print("\nAll modules ready.")


## Step 5 — Generate Dataset

This cell generates all scenarios and saves them to the local VM disk.
Progress is logged for each scenario.

**Tip:** Colab sessions time out after ~12 hours.
For large datasets, consider splitting into multiple runs with `--seed` offsets.


In [ ]:
import sys, types, logging, time, os

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

from generate_dataset import generate_batch_parallel, REBALANCED_DISTRIBUTION, FAULT_CLASS_MAP
from scenario import ScenarioGenerator, ScenarioType
from config import MachineType

# Build args namespace
class Args:
    count           = NUM_SCENARIOS
    output          = LOCAL_OUTPUT_DIR
    seed            = SEED
    output_format   = OUTPUT_FORMAT
    class_balance   = CLASS_BALANCE
    output_rate     = OUTPUT_RATE_HZ
    pipe            = None
    machine_type    = None
    connection_type = None

# Determine workers
num_workers = NUM_WORKERS if NUM_WORKERS > 0 else max(1, (os.cpu_count() or 2) - 1)

print(f"{'='*60}")
print(f"  TopDrive AI - Synthetic Dataset Generation")
print(f"{'='*60}")
print(f"  Scenarios:    {NUM_SCENARIOS}")
print(f"  Workers:      {num_workers}")
print(f"  Format:       {OUTPUT_FORMAT}")
print(f"  Balance:      {CLASS_BALANCE}")
print(f"  Output:       {LOCAL_OUTPUT_DIR}")
print(f"{'='*60}")
print()

t_start = time.perf_counter()
generate_batch_parallel(Args(), num_workers=num_workers)
t_total = time.perf_counter() - t_start

print(f"\nTotal wall time: {t_total:.1f}s ({t_total/60:.1f} min)")
print(f"Average: {t_total/NUM_SCENARIOS:.2f}s per scenario")
print("Generation complete!")


## Step 6: Data Quality Validation
Automated checks to verify the generated dataset meets quality thresholds before training.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

print("="*60)
print("  DATA QUALITY VALIDATION")
print("="*60)

output_dir = Path(LOCAL_OUTPUT_DIR)
manifest_path = output_dir / 'manifest.csv'

# Load manifest
assert manifest_path.exists(), f"Manifest not found: {manifest_path}"
df = pd.read_csv(manifest_path)
n = len(df)
print(f"\nManifest: {n} scenarios loaded\n")

results = []

def check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    mark = "[OK]" if passed else "[XX]"
    results.append((name, passed))
    msg = f"  {mark} {name}"
    if detail:
        msg += f" -- {detail}"
    print(msg)

# ── Check 1: All 9 fault classes present ──────────────────────────────────
class_counts = Counter(df['fault_class'].astype(int))
present = set(class_counts.keys())
expected = set(range(9))
missing = expected - present
check(
    "All 9 classes present",
    len(missing) == 0,
    f"Missing classes: {sorted(missing)}" if missing else f"All present: {sorted(present)}"
)

# ── Check 2: Class balance (each class > 1% of total) ────────────────────
min_class_pct = min(class_counts[c] / n for c in expected if c in class_counts) if present == expected else 0
check(
    "Class balance (each > 1%)",
    all(class_counts.get(c, 0) / n > 0.01 for c in expected),
    f"Smallest class: {min_class_pct:.1%}"
)

# ── Check 3: Normal/fault split roughly 50/50 ────────────────────────────
normal_count = class_counts.get(0, 0)
fault_count = n - normal_count
normal_pct = normal_count / max(n, 1)
check(
    "Normal/fault split (40-60%)",
    0.35 <= normal_pct <= 0.65,
    f"Normal: {normal_pct:.1%} ({normal_count}), Fault: {1-normal_pct:.1%} ({fault_count})"
)

# ── Check 4: Success rate > 95% ──────────────────────────────────────────
successful = (df['num_samples'] > 0).sum()
success_rate = successful / max(n, 1)
check(
    "Success rate > 95%",
    success_rate >= 0.95,
    f"{success_rate:.1%} ({successful}/{n})"
)

# ── Check 5: Physical bounds (torque, RPM, duration) ─────────────────────
valid_rows = df[df['num_samples'] > 0]
torque_ok = (valid_rows['peak_torque_ftlbs'] > 0).all() and (valid_rows['peak_torque_ftlbs'] < 300000).all()
rpm_ok = (valid_rows['peak_rpm'] > 0).all()
dur_ok = (valid_rows['duration_s'] > 0).all() and (valid_rows['duration_s'] <= 125).all()
check(
    "Physical bounds (torque/RPM/duration)",
    torque_ok and rpm_ok and dur_ok,
    f"Torque range: [{valid_rows['peak_torque_ftlbs'].min():.0f}, {valid_rows['peak_torque_ftlbs'].max():.0f}] ft-lbs, "
    f"RPM max: {valid_rows['peak_rpm'].max():.0f}, "
    f"Duration: [{valid_rows['duration_s'].min():.1f}, {valid_rows['duration_s'].max():.1f}]s"
)

# ── Check 6: NaN/Inf in sample of sensor files ───────────────────────────
sensor_dir = output_dir / 'sensor'
sensor_files = sorted(sensor_dir.glob('*.parquet'))
if not sensor_files:
    sensor_files = sorted(sensor_dir.glob('*.csv'))

n_check = min(50, len(sensor_files))
nan_count = 0
inf_count = 0
if n_check > 0:
    rng = np.random.default_rng(42)
    check_indices = rng.choice(len(sensor_files), size=n_check, replace=False)
    for ci in check_indices:
        fp = sensor_files[ci]
        try:
            if fp.suffix == '.parquet':
                import pyarrow.parquet as pq
                tbl = pq.read_table(fp)
                sdf = tbl.to_pandas()
            else:
                sdf = pd.read_csv(fp)
            num_cols = sdf.select_dtypes(include=[np.number])
            nan_count += num_cols.isna().sum().sum()
            inf_count += np.isinf(num_cols.values).sum()
        except Exception as e:
            nan_count += 1  # Count as error
check(
    f"NaN/Inf check ({n_check} files sampled)",
    nan_count == 0 and inf_count == 0,
    f"NaN: {nan_count}, Inf: {inf_count}" if (nan_count > 0 or inf_count > 0) else "Clean"
)

# ── Check 7: Minimum samples per scenario ─────────────────────────────────
min_samples = valid_rows['num_samples'].min() if len(valid_rows) > 0 else 0
check(
    "Min samples per scenario > 100",
    min_samples > 100,
    f"Min: {min_samples}, Median: {valid_rows['num_samples'].median():.0f}"
)

# ── Check 8: Fault code consistency ───────────────────────────────────────
from generate_dataset import FAULT_CLASS_MAP
mismatches = 0
for _, row in df.iterrows():
    expected_class = FAULT_CLASS_MAP.get(row['scenario_type'], 0)
    if int(row['fault_class']) != expected_class:
        mismatches += 1
check(
    "Fault code consistency",
    mismatches == 0,
    f"{mismatches} mismatches" if mismatches > 0 else "All consistent"
)

# ── Check 9: No duplicate scenario IDs ────────────────────────────────────
dups = df['scenario_id'].duplicated().sum()
check(
    "No duplicate scenario IDs",
    dups == 0,
    f"{dups} duplicates" if dups > 0 else "All unique"
)

# ── Check 10: File count matches manifest ─────────────────────────────────
file_count = len(list(sensor_dir.iterdir())) if sensor_dir.exists() else 0
check(
    "File count matches manifest",
    file_count == successful,
    f"Files: {file_count}, Expected: {successful}"
)

# ── Summary ───────────────────────────────────────────────────────────────
passed = sum(1 for _, p in results if p)
total = len(results)

print(f"\n{'='*60}")
if passed == total:
    print(f"  ALL {total} CHECKS PASSED - Dataset ready for training!")
else:
    print(f"  {passed}/{total} checks passed, {total-passed} FAILED")
    print(f"  Review failures above before proceeding to training.")
print(f"{'='*60}")

# Detailed class distribution table
print(f"\nDetailed Fault Class Distribution:")
print(f"  {'Class':5s} {'Name':20s} {'Count':>6s} {'Pct':>7s}")
print(f"  {'-'*5} {'-'*20} {'-'*6} {'-'*7}")
CLASS_NAMES = {
    0: 'normal', 1: 'cross_thread', 2: 'galling', 3: 'stripped_thread',
    4: 'over_torque', 5: 'under_torque', 6: 'wrong_compound',
    7: 'misaligned_stab', 8: 'stall',
}
for cls_id in range(9):
    cnt = class_counts.get(cls_id, 0)
    name = CLASS_NAMES.get(cls_id, '?')
    print(f"  {cls_id:5d} {name:20s} {cnt:6d} {cnt/max(n,1):7.1%}")


## Step 7 — Copy to Google Drive

Copies the generated dataset to Drive for persistent storage.
This step is safe to re-run if interrupted.


In [ ]:
import shutil, os

print(f"Syncing {LOCAL_OUTPUT_DIR} -> {DRIVE_OUTPUT_DIR}")
shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
print("Sync complete.\n")

# Summary with sizes
total_size = 0
for subdir in ['sensor', 'truth', 'events']:
    path = os.path.join(DRIVE_OUTPUT_DIR, subdir)
    if os.path.exists(path):
        files = os.listdir(path)
        n = len(files)
        size = sum(os.path.getsize(os.path.join(path, f)) for f in files)
        total_size += size
        print(f"  {subdir}/  {n:5d} files  ({size/1e6:.1f} MB)")

print(f"\n  Total size: {total_size/1e6:.1f} MB ({total_size/1e9:.2f} GB)")

stats_path = os.path.join(DRIVE_OUTPUT_DIR, 'stats.txt')
if os.path.exists(stats_path):
    print()
    with open(stats_path) as f:
        print(f.read())
